In [1]:
from google.colab import files

uploaded = files.upload()

Saving merged_features_final.xlsx to merged_features_final.xlsx


In [2]:
import pandas as pd

df = pd.read_excel("merged_features_final.xlsx")

print("Shape:", df.shape)
df.head()

Shape: (60000, 20)


,url,entropy,url_length,num_www,ratio_digits_url,num_subdomains,length_hostname,random_domain,num_dots,num_hyphens,ratio_digits_host,prefix_suffix,num_question_mark,num_eq,domain_age,dns_resolvable,num_ips,ttl,num_name_servers,label
0,https://manua.ls,3.500000,16,0,0.000000,0,8,0,1,0,0.0,0,0,0,-1.0,1.0,1.0,3600.0,4.0,legitimate
1,https://t.co/r77VHQx9lP,4.023137,23,0,0.130435,0,4,1,1,0,0.0,0,0,0,5745.0,1.0,1.0,300.0,8.0,phishing
2,https://sites.google.com/view/attyahoo-plus-la...,4.145908,58,0,0.000000,1,16,0,2,3,0.0,0,0,0,10351.0,1.0,6.0,300.0,4.0,phishing
3,https://bit.ly/48c5385,3.845351,22,0,0.272727,0,6,0,1,0,0.0,0,0,0,6453.0,1.0,2.0,300.0,8.0,phishing
4,https://strato.de,3.381580,17,0,0.000000,0,9,0,1,0,0.0,0,0,0,-1.0,1.0,1.0,3600.0,4.0,legitimate


In [3]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

Shape: (60000, 20)

Columns:
['url', 'entropy', 'url_length', 'num_www', 'ratio_digits_url', 'num_subdomains', 'length_hostname', 'random_domain', 'num_dots', 'num_hyphens', 'ratio_digits_host', 'prefix_suffix', 'num_question_mark', 'num_eq', 'domain_age', 'dns_resolvable', 'num_ips', 'ttl', 'num_name_servers', 'label']

Data types:
url                   object
entropy              float64
url_length             int64
num_www                int64
ratio_digits_url     float64
num_subdomains         int64
length_hostname        int64
random_domain          int64
num_dots               int64
num_hyphens            int64
ratio_digits_host    float64
prefix_suffix          int64
num_question_mark      int64
num_eq                 int64
domain_age           float64
dns_resolvable       float64
num_ips              float64
ttl                  float64
num_name_servers     float64
label                 object
dtype: object

Missing values:
url                  0
entropy              0
url_leng

In [4]:
print(df["label"].value_counts())
print()
print(df["label"].value_counts(normalize=True) * 100)

label
phishing      31641
legitimate    28359
Name: count, dtype: int64

label
phishing      52.735
legitimate    47.265
Name: proportion, dtype: float64


In [5]:
print("Duplicate URLs:", df["url"].duplicated().sum())

Duplicate URLs: 0


In [6]:
duplicate_urls = df[df["url"].duplicated(keep=False)].sort_values("url")

duplicate_urls.head(20)

,url,entropy,url_length,num_www,ratio_digits_url,num_subdomains,length_hostname,random_domain,num_dots,num_hyphens,ratio_digits_host,prefix_suffix,num_question_mark,num_eq,domain_age,dns_resolvable,num_ips,ttl,num_name_servers,label


In [7]:
from urllib.parse import urlparse

def extract_domain(url):
    try:
        return urlparse(str(url)).netloc.lower().replace("www.", "")
    except:
        return ""

df["domain"] = df["url"].apply(extract_domain)

df[["url", "domain", "label"]].head(10)

,url,domain,label
0,https://manua.ls,manua.ls,legitimate
1,https://t.co/r77VHQx9lP,t.co,phishing
2,https://sites.google.com/view/attyahoo-plus-la...,sites.google.com,phishing
3,https://bit.ly/48c5385,bit.ly,phishing
4,https://strato.de,strato.de,legitimate
5,https://techmeme.com,techmeme.com,legitimate
6,https://mahabhumi.gov.in,mahabhumi.gov.in,legitimate
7,https://goldenfrog.com,goldenfrog.com,legitimate
8,https://myproductglobalsource.weebly.com/,myproductglobalsource.weebly.com,phishing
9,https://www.dropbox.com/scl/fi/av83rq75mbe8otu...,dropbox.com,phishing


In [8]:
# 1. Number of unique domains
print("Total URLs:", len(df))
print("Unique URLs:", df["url"].nunique())
print("Unique domains:", df["domain"].nunique())

# 2. Repeated domains
domain_counts = df["domain"].value_counts()

print("\nDomains appearing more than once:",
      (domain_counts > 1).sum())

# 3. Domains having both labels
domain_label_counts = df.groupby("domain")["label"].nunique()

conflicting_domains = domain_label_counts[
    domain_label_counts > 1
]

print("\nDomains with BOTH phishing and legitimate URLs:",
      len(conflicting_domains))

Total URLs: 60000
Unique URLs: 60000
Unique domains: 40906

Domains appearing more than once: 3975

Domains with BOTH phishing and legitimate URLs: 52


In [9]:
conflicting_domains = (
    df.groupby("domain")["label"]
    .nunique()
)

conflicting_domains = conflicting_domains[
    conflicting_domains > 1
].index

print("Number of conflicting domains:", len(conflicting_domains))

display(
    df[df["domain"].isin(conflicting_domains)]
    [["url", "domain", "label"]]
    .sort_values("domain")
    .head(100)
)

Number of conflicting domains: 52


,url,domain,label
27791,https://www.art.com,art.com,legitimate
49714,https://www.art.com/,art.com,phishing
37937,https://artisteer.com,artisteer.com,legitimate
38375,http://www.artisteer.com/?p=affr&amp;redirect_...,artisteer.com,phishing
19891,http://awin1.com/cread.php?awinmid=10178&awina...,awin1.com,phishing
...,...,...,...
33397,https://bit.ly/3C1bkYn,bit.ly,phishing
33332,https://bit.ly/482Yepp,bit.ly,phishing
33327,https://bit.ly/4fCcyYB,bit.ly,phishing
33255,https://bit.ly/46xGP7A,bit.ly,phishing


In [10]:
from urllib.parse import urlparse, urlunparse

def normalize_url(url):
    try:
        parsed = urlparse(str(url).strip().lower())

        # Remove trailing slash from path
        path = parsed.path.rstrip("/")

        # Rebuild URL
        normalized = urlunparse((
            parsed.scheme,
            parsed.netloc,
            path,
            "",
            parsed.query,
            ""
        ))

        return normalized

    except:
        return str(url).strip().lower()


df["normalized_url"] = df["url"].apply(normalize_url)

print("Original unique URLs:",
      df["url"].nunique())

print("Normalized unique URLs:",
      df["normalized_url"].nunique())

print("New duplicate URLs after normalization:",
      df["normalized_url"].duplicated().sum())

Original unique URLs: 60000
Normalized unique URLs: 59816
New duplicate URLs after normalization: 184


In [11]:
# Check whether normalized duplicate URLs have conflicting labels

normalized_label_counts = (
    df.groupby("normalized_url")["label"]
    .nunique()
)

conflicting_normalized_urls = normalized_label_counts[
    normalized_label_counts > 1
]

print(
    "Normalized URL groups with conflicting labels:",
    len(conflicting_normalized_urls)
)

Normalized URL groups with conflicting labels: 2


In [12]:
# Show the conflicting cases

conflicting_data = df[
    df["normalized_url"].isin(conflicting_normalized_urls.index)
][["url", "normalized_url", "domain", "label"]].sort_values(
    "normalized_url"
)

display(conflicting_data)

,url,normalized_url,domain,label
5595,https://kakaku.com,https://kakaku.com,kakaku.com,legitimate
20437,https://kakaku.com/,https://kakaku.com,kakaku.com,phishing
27791,https://www.art.com,https://www.art.com,art.com,legitimate
49714,https://www.art.com/,https://www.art.com,art.com,phishing


In [13]:
print("Conflicting normalized URL groups:",
      len(conflicting_normalized_urls))

print("Rows involved in conflicts:",
      df["normalized_url"].isin(
          conflicting_normalized_urls.index
      ).sum())

Conflicting normalized URL groups: 2
Rows involved in conflicts: 4


In [14]:
df_clean = df[
    ~df["normalized_url"].isin(conflicting_normalized_urls.index)
].copy()

print("Original rows:", len(df))
print("After removing conflicting URLs:", len(df_clean))

Original rows: 60000
After removing conflicting URLs: 59996


In [15]:
before = len(df_clean)

df_clean = df_clean.drop_duplicates(
    subset="normalized_url",
    keep="first"
).copy()

print("Rows before:", before)
print("Rows after:", len(df_clean))
print("Rows removed:", before - len(df_clean))

Rows before: 59996
Rows after: 59814
Rows removed: 182


In [16]:
print("Final rows:", len(df_clean))

print(
    "Duplicate normalized URLs:",
    df_clean["normalized_url"].duplicated().sum()
)

print(
    "Conflicting normalized URLs:",
    df_clean.groupby("normalized_url")["label"].nunique()
        .gt(1).sum()
)

Final rows: 59814
Duplicate normalized URLs: 0
Conflicting normalized URLs: 0


In [17]:
print("Shape:", df_clean.shape)

print("\nColumns:")
print(df_clean.columns.tolist())

print("\nData types:")
print(df_clean.dtypes)

Shape: (59814, 22)

Columns:
['url', 'entropy', 'url_length', 'num_www', 'ratio_digits_url', 'num_subdomains', 'length_hostname', 'random_domain', 'num_dots', 'num_hyphens', 'ratio_digits_host', 'prefix_suffix', 'num_question_mark', 'num_eq', 'domain_age', 'dns_resolvable', 'num_ips', 'ttl', 'num_name_servers', 'label', 'domain', 'normalized_url']

Data types:
url                   object
entropy              float64
url_length             int64
num_www                int64
ratio_digits_url     float64
num_subdomains         int64
length_hostname        int64
random_domain          int64
num_dots               int64
num_hyphens            int64
ratio_digits_host    float64
prefix_suffix          int64
num_question_mark      int64
num_eq                 int64
domain_age           float64
dns_resolvable       float64
num_ips              float64
ttl                  float64
num_name_servers     float64
label                 object
domain                object
normalized_url        object

In [18]:
missing = df_clean.isnull().sum()

print(missing)
print("\nTotal missing values:", missing.sum())

url                  0
entropy              0
url_length           0
num_www              0
ratio_digits_url     0
num_subdomains       0
length_hostname      0
random_domain        0
num_dots             0
num_hyphens          0
ratio_digits_host    0
prefix_suffix        0
num_question_mark    0
num_eq               0
domain_age           2
dns_resolvable       2
num_ips              2
ttl                  2
num_name_servers     2
label                0
domain               0
normalized_url       0
dtype: int64

Total missing values: 10


In [19]:
df_clean.describe().T

,count,mean,std,min,25%,50%,75%,max
entropy,59814.0,4.071604,0.563374,2.353338,3.719295,3.93626,4.221085,6.032729
url_length,59814.0,44.096666,117.899608,12.000000,20.000000,24.00000,41.000000,25523.000000
num_www,59814.0,0.097736,0.298868,0.000000,0.000000,0.00000,0.000000,3.000000
ratio_digits_url,59814.0,0.045274,0.084474,0.000000,0.000000,0.00000,0.067211,0.826087
num_subdomains,59814.0,0.481242,0.575883,0.000000,0.000000,0.00000,1.000000,7.000000
length_hostname,59814.0,16.649965,10.366546,4.000000,10.000000,15.00000,20.000000,100.000000
random_domain,59814.0,0.148360,0.355459,0.000000,0.000000,0.00000,0.000000,1.000000
num_dots,59814.0,1.668339,0.974183,1.000000,1.000000,2.00000,2.000000,23.000000
num_hyphens,59814.0,0.472699,1.100419,0.000000,0.000000,0.00000,1.000000,50.000000
ratio_digits_host,59814.0,0.033040,0.103424,0.000000,0.000000,0.00000,0.000000,0.800000


In [20]:
# Check sentinel values and missing values in DNS/domain features

dns_cols = [
    "domain_age",
    "dns_resolvable",
    "num_ips",
    "ttl",
    "num_name_servers"
]

for col in dns_cols:
    print(f"\n===== {col} =====")
    print("Missing:", df_clean[col].isna().sum())
    print("Value counts:")
    print(df_clean[col].value_counts(dropna=False).head(10))


===== domain_age =====
Missing: 2
Value counts:
domain_age
-1.0        18611
 10351.0     3119
 10352.0     1198
 7234.0      1116
 6453.0       993
 4841.0       788
 5099.0       567
 14306.0      507
 7235.0       457
 4842.0       447
Name: count, dtype: int64

===== dns_resolvable =====
Missing: 2
Value counts:
dns_resolvable
1.0    55675
0.0     4137
NaN        2
Name: count, dtype: int64

===== num_ips =====
Missing: 2
Value counts:
num_ips
1.0     21624
2.0     18442
4.0      6884
6.0      4996
0.0      4137
3.0      3242
5.0       247
8.0       146
7.0        27
10.0       26
Name: count, dtype: int64

===== ttl =====
Missing: 2
Value counts:
ttl
 300.0      25520
 60.0        9794
-1.0         4137
 3600.0      4117
 21600.0     3235
 900.0       3208
 600.0       2597
 20.0        1840
 14400.0     1064
 30.0         589
Name: count, dtype: int64

===== num_name_servers =====
Missing: 2
Value counts:
num_name_servers
4.0     25414
2.0     20445
0.0      4294
3.0      3299
6

In [21]:
print("URLs > 200:", (df_clean["url_length"] > 200).sum())
print("URLs > 500:", (df_clean["url_length"] > 500).sum())
print("URLs > 1000:", (df_clean["url_length"] > 1000).sum())

display(
    df_clean.nlargest(10, "url_length")[
        ["url", "url_length", "label"]
    ]
)

URLs > 200: 583
URLs > 500: 74
URLs > 1000: 21


,url,url_length,label
28140,https://authjgxszcloxpnassocl.firebaseapp.com/...,25523,phishing
2773,https://sobakemozhno.ru/wp-content/plugins/wor...,1520,phishing
7647,https://u24148341.ct.sendgrid.net/ls/click?upn...,1436,phishing
35311,https://sub.followerswave.com/track-followers/...,1261,phishing
49799,http://copia.vspabogados.com/wp-admin/app.php?...,1223,phishing
6134,https://mytimemakeup.com/l/6nmDqdoAkutDBlOyhZb...,1201,phishing
54267,https://toystodolls.com/l/TFXxpwdh0V0rcHBvw8Tk...,1201,phishing
29761,http://fieldconnect.jbktechnologies.net/core/v...,1186,phishing
49826,https://recaptcha-metacloud.com/two_step_verif...,1185,phishing
46647,https://forcarorbike.com/l/Z0MuClXHjtoBCnxPQiP...,1154,phishing


In [22]:
url_features = [
    "entropy",
    "url_length",
    "num_www",
    "ratio_digits_url",
    "num_subdomains",
    "length_hostname",
    "random_domain",
    "num_dots",
    "num_hyphens",
    "ratio_digits_host",
    "prefix_suffix",
    "num_question_mark",
    "num_eq"
]

feature_means = (
    df_clean.groupby("label")[url_features]
    .mean()
    .T
)

display(feature_means)

label,legitimate,phishing
entropy,3.699512,4.407027
url_length,20.826639,65.073497
num_www,0.160772,0.040913
ratio_digits_url,0.005438,0.081184
num_subdomains,0.160348,0.770512
length_hostname,12.826639,20.096513
random_domain,0.099482,0.192421
num_dots,1.237261,2.056935
num_hyphens,0.055683,0.848619
ratio_digits_host,0.009543,0.054223


In [23]:
domain_label = (
    df_clean.groupby("domain")["label"]
    .agg(["count", "nunique"])
)

print("Total domains:", len(domain_label))

print("\nDomains containing only one label:")
print(
    (domain_label["nunique"] == 1).sum()
)

print("\nDomains containing both labels:")
print(
    (domain_label["nunique"] > 1).sum()
)

Total domains: 40904

Domains containing only one label:
40854

Domains containing both labels:
50


In [24]:
url_features = [
    "entropy",
    "url_length",
    "num_www",
    "ratio_digits_url",
    "num_subdomains",
    "length_hostname",
    "random_domain",
    "num_dots",
    "num_hyphens",
    "ratio_digits_host",
    "prefix_suffix",
    "num_question_mark",
    "num_eq"
]

X = df_clean[url_features]
y = df_clean["label"]

groups = df_clean["domain"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of groups:", groups.nunique())

X shape: (59814, 13)
y shape: (59814,)
Number of groups: 40904


In [25]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Classes:", label_encoder.classes_)

Classes: ['legitimate' 'phishing']


In [26]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y_encoded, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y_encoded[train_idx]
y_test = y_encoded[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 47323
Testing rows: 12491


In [27]:
train_domains = set(groups_train)
test_domains = set(groups_test)

overlap = train_domains.intersection(test_domains)

print("Training domains:", len(train_domains))
print("Testing domains:", len(test_domains))
print("Overlapping domains:", len(overlap))

Training domains: 32723
Testing domains: 8181
Overlapping domains: 0


In [28]:
import pandas as pd

print("Training distribution:")
print(
    pd.Series(y_train)
    .map({0: "legitimate", 1: "phishing"})
    .value_counts(normalize=True) * 100
)

print("\nTesting distribution:")
print(
    pd.Series(y_test)
    .map({0: "legitimate", 1: "phishing"})
    .value_counts(normalize=True) * 100
)

Training distribution:
phishing      52.088836
legitimate    47.911164
Name: proportion, dtype: float64

Testing distribution:
phishing      54.495237
legitimate    45.504763
Name: proportion, dtype: float64


In [29]:
print(X_train.shape)
print(X_test.shape)

print("\nTraining missing values:")
print(X_train.isnull().sum())

print("\nTesting missing values:")
print(X_test.isnull().sum())

(47323, 13)
(12491, 13)

Training missing values:
entropy              0
url_length           0
num_www              0
ratio_digits_url     0
num_subdomains       0
length_hostname      0
random_domain        0
num_dots             0
num_hyphens          0
ratio_digits_host    0
prefix_suffix        0
num_question_mark    0
num_eq               0
dtype: int64

Testing missing values:
entropy              0
url_length           0
num_www              0
ratio_digits_url     0
num_subdomains       0
length_hostname      0
random_domain        0
num_dots             0
num_hyphens          0
ratio_digits_host    0
prefix_suffix        0
num_question_mark    0
num_eq               0
dtype: int64


In [30]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train, y_train)

print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


In [31]:
# Predict classes
y_pred_lr = logistic_model.predict(X_test)

# Probability that each URL is phishing
y_prob_lr = logistic_model.predict_proba(X_test)[:, 1]

print("Predictions completed!")

Predictions completed!


In [32]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

print("Accuracy :", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall   :", recall_score(y_test, y_pred_lr))
print("F1 Score :", f1_score(y_test, y_pred_lr))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_lr))

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred_lr,
        target_names=label_encoder.classes_
    )
)

Accuracy : 0.9916740052838043
Precision: 0.9985125687936933
Recall   : 0.9861906860584693
F1 Score : 0.9923133776792313
ROC-AUC  : 0.9948789108202665

Classification Report:

              precision    recall  f1-score   support

  legitimate       0.98      1.00      0.99      5684
    phishing       1.00      0.99      0.99      6807

    accuracy                           0.99     12491
   macro avg       0.99      0.99      0.99     12491
weighted avg       0.99      0.99      0.99     12491



In [33]:
import pandas as pd

coefficients = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": logistic_model.coef_[0]
})

coefficients["abs_coefficient"] = coefficients["coefficient"].abs()

coefficients = coefficients.sort_values(
    "abs_coefficient",
    ascending=False
)

display(coefficients)

,feature,coefficient,abs_coefficient
2,num_www,-5.944282,5.944282
4,num_subdomains,5.568151,5.568151
1,url_length,4.063343,4.063343
5,length_hostname,-3.805339,3.805339
9,ratio_digits_host,2.718853,2.718853
3,ratio_digits_url,2.219695,2.219695
8,num_hyphens,1.192515,1.192515
0,entropy,-0.799002,0.799002
6,random_domain,-0.603861,0.603861
7,num_dots,-0.177126,0.177126


In [34]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [35]:
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

In [36]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

print("Accuracy :", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall   :", recall_score(y_test, y_pred_rf))
print("F1 Score :", f1_score(y_test, y_pred_rf))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_rf))

print("\nClassification Report:\n")

print(classification_report(
    y_test,
    y_pred_rf,
    target_names=label_encoder.classes_
))

Accuracy : 0.991273717076295
Precision: 0.9970317601662214
Recall   : 0.9869252240340826
F1 Score : 0.991952750092285
ROC-AUC  : 0.9979784956641582

Classification Report:

              precision    recall  f1-score   support

  legitimate       0.98      1.00      0.99      5684
    phishing       1.00      0.99      0.99      6807

    accuracy                           0.99     12491
   macro avg       0.99      0.99      0.99     12491
weighted avg       0.99      0.99      0.99     12491



In [37]:
rf_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(rf_importance)

,feature,importance
0,entropy,0.256516
1,url_length,0.248009
5,length_hostname,0.138630
3,ratio_digits_url,0.096186
4,num_subdomains,0.075616
2,num_www,0.069469
7,num_dots,0.051071
8,num_hyphens,0.027339
9,ratio_digits_host,0.015851
6,random_domain,0.006612


In [38]:
df_corr = df_clean[url_features].copy()

df_corr["label_encoded"] = (
    df_clean["label"] == "phishing"
).astype(int)

correlations = (
    df_corr.corr()["label_encoded"]
    .drop("label_encoded")
    .sort_values(key=abs, ascending=False)
)

display(correlations)

,label_encoded
entropy,0.627088
num_subdomains,0.529056
ratio_digits_url,0.447735
num_dots,0.420136
num_hyphens,0.359807
length_hostname,0.350173
num_question_mark,0.314538
num_eq,0.244216
ratio_digits_host,0.215717
num_www,-0.200253


In [39]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaled_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

scaled_lr.fit(X_train, y_train)

print("Scaled Logistic Regression trained successfully!")

Scaled Logistic Regression trained successfully!


In [40]:
y_pred_scaled = scaled_lr.predict(X_test)

y_prob_scaled = scaled_lr.predict_proba(X_test)[:, 1]

In [41]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Accuracy :", accuracy_score(y_test, y_pred_scaled))
print("Precision:", precision_score(y_test, y_pred_scaled))
print("Recall   :", recall_score(y_test, y_pred_scaled))
print("F1 Score :", f1_score(y_test, y_pred_scaled))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_scaled))

Accuracy : 0.9782243215114883
Precision: 0.9960528313344467
Recall   : 0.9638607315998237
F1 Score : 0.9796923995819024
ROC-AUC  : 0.990240724791003


In [42]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred_scaled,
        target_names=label_encoder.classes_
    )
)

              precision    recall  f1-score   support

  legitimate       0.96      1.00      0.98      5684
    phishing       1.00      0.96      0.98      6807

    accuracy                           0.98     12491
   macro avg       0.98      0.98      0.98     12491
weighted avg       0.98      0.98      0.98     12491



In [43]:
scaled_lr_model = scaled_lr.named_steps["model"]

scaled_coefficients = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": scaled_lr_model.coef_[0]
})

scaled_coefficients["abs_coefficient"] = (
    scaled_coefficients["coefficient"].abs()
)

scaled_coefficients = scaled_coefficients.sort_values(
    "abs_coefficient",
    ascending=False
)

display(scaled_coefficients)

,feature,coefficient,abs_coefficient
1,url_length,39.204277,39.204277
3,ratio_digits_url,4.376677,4.376677
5,length_hostname,-4.134753,4.134753
9,ratio_digits_host,-3.485750,3.485750
4,num_subdomains,3.317998,3.317998
8,num_hyphens,3.257121,3.257121
0,entropy,2.902089,2.902089
2,num_www,-1.883480,1.883480
11,num_question_mark,1.125723,1.125723
12,num_eq,0.725048,0.725048


In [44]:
features_no_length = [
    "entropy",
    "num_www",
    "ratio_digits_url",
    "num_subdomains",
    "length_hostname",
    "random_domain",
    "num_dots",
    "num_hyphens",
    "ratio_digits_host",
    "prefix_suffix",
    "num_question_mark",
    "num_eq"
]

X_train_no_length = X_train[features_no_length]
X_test_no_length = X_test[features_no_length]

In [45]:
lr_no_length = LogisticRegression(
    max_iter=2000,
    random_state=42
)

lr_no_length.fit(
    X_train_no_length,
    y_train
)

LogisticRegression(max_iter=2000, random_state=42)

In [46]:
y_pred_no_length = lr_no_length.predict(X_test_no_length)
y_prob_no_length = lr_no_length.predict_proba(
    X_test_no_length
)[:, 1]

print("Accuracy :",
      accuracy_score(y_test, y_pred_no_length))

print("Precision:",
      precision_score(y_test, y_pred_no_length))

print("Recall   :",
      recall_score(y_test, y_pred_no_length))

print("F1 Score :",
      f1_score(y_test, y_pred_no_length))

print("ROC-AUC  :",
      roc_auc_score(y_test, y_prob_no_length))

Accuracy : 0.962933311984629
Precision: 0.9844227244960293
Recall   : 0.946966358160717
F1 Score : 0.9653313365780607
ROC-AUC  : 0.9820091438347348


In [47]:
import numpy as np

df_clean["log_url_length"] = np.log1p(
    df_clean["url_length"]
)

print(
    df_clean[
        ["url_length", "log_url_length"]
    ].describe()
)

         url_length  log_url_length
count  59814.000000    59814.000000
mean      44.096666        3.495553
std      117.899608        0.661092
min       12.000000        2.564949
25%       20.000000        3.044522
50%       24.000000        3.218876
75%       41.000000        3.737670
max    25523.000000       10.147374


In [48]:
log_features = [
    "entropy",
    "log_url_length",
    "num_www",
    "ratio_digits_url",
    "num_subdomains",
    "length_hostname",
    "random_domain",
    "num_dots",
    "num_hyphens",
    "ratio_digits_host",
    "prefix_suffix",
    "num_question_mark",
    "num_eq"
]

In [49]:
X_log = df_clean[log_features]

X_train_log = X_log.iloc[train_idx]
X_test_log = X_log.iloc[test_idx]

In [50]:
lr_log = LogisticRegression(
    max_iter=2000,
    random_state=42
)

lr_log.fit(
    X_train_log,
    y_train
)

LogisticRegression(max_iter=2000, random_state=42)

In [51]:
y_pred_log = lr_log.predict(X_test_log)

y_prob_log = lr_log.predict_proba(
    X_test_log
)[:, 1]

print("Accuracy :",
      accuracy_score(y_test, y_pred_log))

print("Precision:",
      precision_score(y_test, y_pred_log))

print("Recall   :",
      recall_score(y_test, y_pred_log))

print("F1 Score :",
      f1_score(y_test, y_pred_log))

print("ROC-AUC  :",
      roc_auc_score(y_test, y_prob_log))

Accuracy : 0.9839084140581218
Precision: 0.9969906710803491
Recall   : 0.9734097252827971
F1 Score : 0.985059094625734
ROC-AUC  : 0.9948094114319336


In [52]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


In [53]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

print("XGBoost trained successfully!")

XGBoost trained successfully!


In [54]:
y_pred_xgb = xgb_model.predict(X_test)

y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

In [55]:
print("Accuracy :",
      accuracy_score(y_test, y_pred_xgb))

print("Precision:",
      precision_score(y_test, y_pred_xgb))

print("Recall   :",
      recall_score(y_test, y_pred_xgb))

print("F1 Score :",
      f1_score(y_test, y_pred_xgb))

print("ROC-AUC  :",
      roc_auc_score(y_test, y_prob_xgb))

Accuracy : 0.9927948122648307
Precision: 0.9988118223674439
Recall   : 0.9879535771999413
F1 Score : 0.9933530280649926
ROC-AUC  : 0.9979972984923517


In [56]:
print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred_xgb,
        target_names=label_encoder.classes_
    )
)


Classification Report:

              precision    recall  f1-score   support

  legitimate       0.99      1.00      0.99      5684
    phishing       1.00      0.99      0.99      6807

    accuracy                           0.99     12491
   macro avg       0.99      0.99      0.99     12491
weighted avg       0.99      0.99      0.99     12491



In [57]:
xgb_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": xgb_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(xgb_importance)

,feature,importance
0,entropy,0.198639
1,url_length,0.171328
2,num_www,0.147780
4,num_subdomains,0.117610
5,length_hostname,0.089346
3,ratio_digits_url,0.073683
9,ratio_digits_host,0.046478
12,num_eq,0.043932
7,num_dots,0.039194
6,random_domain,0.030090


In [58]:
test_results = df_clean.iloc[test_idx].copy()

test_results["actual"] = y_test
test_results["predicted"] = y_pred_xgb
test_results["phishing_probability"] = y_prob_xgb

In [59]:
false_negatives = test_results[
    (test_results["actual"] == 1) &
    (test_results["predicted"] == 0)
].copy()

print("False negatives:", len(false_negatives))

display(
    false_negatives[
        [
            "url",
            "entropy",
            "url_length",
            "num_www",
            "num_subdomains",
            "length_hostname",
            "ratio_digits_url",
            "phishing_probability"
        ]
    ].head(20)
)

False negatives: 82


,url,entropy,url_length,num_www,num_subdomains,length_hostname,ratio_digits_url,phishing_probability
104,https://particulierchequeenergie.com/,3.985556,37,0,0,28,0.0,0.263155
234,https://www-nomura.njatx.cn,4.004344,27,1,1,19,0.0,0.156472
1455,http://cw-net.de,3.452820,16,0,0,9,0.0,0.146194
3882,https://fireflyskinz.com/,4.133661,25,0,0,16,0.0,0.440122
4193,https://www.workjunction.in/,3.797903,28,1,1,19,0.0,0.166868
5850,http://synergyminerals.in,3.893661,25,0,0,18,0.0,0.020057
7890,https://reipojtpoerjp.com,3.623465,25,0,0,17,0.0,0.023288
9290,https://internetbankingonline.digital,3.817995,37,0,0,29,0.0,0.144280
9526,https://www.peage-renouvellement.com/,4.080415,37,1,1,28,0.0,0.479991
9834,http://limonizza.de/,3.784184,20,0,0,12,0.0,0.009640


In [60]:
false_positives = test_results[
    (test_results["actual"] == 0) &
    (test_results["predicted"] == 1)
].copy()

print("False positives:", len(false_positives))

display(
    false_positives[
        [
            "url",
            "entropy",
            "url_length",
            "num_www",
            "num_subdomains",
            "length_hostname",
            "ratio_digits_url",
            "phishing_probability"
        ]
    ].head(20)
)

False positives: 8


,url,entropy,url_length,num_www,num_subdomains,length_hostname,ratio_digits_url,phishing_probability
7106,https://moushikomi-uketsukekun.com,3.822182,34,0,0,26,0.00000,0.515733
24997,https://0123456789nonexistent.com,4.415458,33,0,0,25,0.30303,0.547047
36293,https://marketingautomation.services,4.041336,36,0,0,28,0.00000,0.505879
37923,https://uspreventiveservicestaskforce.org,3.945984,41,0,0,33,0.00000,0.894661
40874,https://www.uspreventiveservicestaskforce.org,4.055649,45,1,1,37,0.00000,0.817843
41147,https://www.kingsoft-office-service.com,4.156195,39,1,1,31,0.00000,0.917707
53906,https://washclothexcludingincluding.com,4.162981,39,0,0,31,0.00000,0.665978
55790,https://use-application-dns.net,3.913401,31,0,0,23,0.00000,0.579338


In [61]:
test_results["uncertainty"] = abs(
    test_results["phishing_probability"] - 0.5
)

uncertain_urls = test_results.sort_values(
    "uncertainty"
).head(20)

display(
    uncertain_urls[
        [
            "url",
            "label",
            "phishing_probability",
            "entropy",
            "url_length",
            "num_subdomains",
            "ratio_digits_url"
        ]
    ]
)

,url,label,phishing_probability,entropy,url_length,num_subdomains,ratio_digits_url
59676,https://askgriff.com/,phishing,0.499261,3.880180,21,0,0.0
36293,https://marketingautomation.services,legitimate,0.505879,4.041336,36,0,0.0
18659,https://eximiamedical.ro/,phishing,0.491847,3.943465,25,0,0.0
22365,https://ksatnam.com/,phishing,0.510813,3.546439,20,0,0.0
58305,https://jorimts.com/,phishing,0.510813,3.546439,20,0,0.0
17338,https://jal-co.bond/,phishing,0.486252,3.884184,20,0,0.0
7106,https://moushikomi-uketsukekun.com,legitimate,0.515733,3.822182,34,0,0.0
22038,https://axsroute.com/,phishing,0.516220,3.748995,21,0,0.0
5238,https://www.fortunebusinessinsights.com,legitimate,0.483649,4.009135,39,1,0.0
9526,https://www.peage-renouvellement.com/,phishing,0.479991,4.080415,37,1,0.0


In [62]:
import joblib

joblib.dump(xgb_model, "phishing_xgb_model.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")

print("Model saved successfully!")

Model saved successfully!


In [63]:
from urllib.parse import urlparse
import math
from collections import Counter
import re


def calculate_entropy(text):
    if not text:
        return 0

    counts = Counter(text)
    length = len(text)

    entropy = 0

    for count in counts.values():
        probability = count / length
        entropy -= probability * math.log2(probability)

    return entropy


def extract_url_features(url):

    # Make sure URL has a scheme
    if not re.match(r"^https?://", url, re.IGNORECASE):
        url = "http://" + url

    parsed = urlparse(url)

    hostname = parsed.hostname or ""
    full_url = url

    # -----------------------------
    # Basic URL features
    # -----------------------------

    url_length = len(full_url)

    num_www = full_url.lower().count("www")

    digits_url = sum(
        char.isdigit() for char in full_url
    )

    ratio_digits_url = (
        digits_url / url_length
        if url_length > 0 else 0
    )

    # -----------------------------
    # Hostname features
    # -----------------------------

    hostname_length = len(hostname)

    digits_host = sum(
        char.isdigit() for char in hostname
    )

    ratio_digits_host = (
        digits_host / hostname_length
        if hostname_length > 0 else 0
    )

    # -----------------------------
    # Subdomains
    # -----------------------------

    hostname_parts = hostname.split(".")

    num_subdomains = max(
        len(hostname_parts) - 2,
        0
    )

    # -----------------------------
    # URL characters
    # -----------------------------

    num_dots = full_url.count(".")

    num_hyphens = full_url.count("-")

    num_question_mark = full_url.count("?")

    num_eq = full_url.count("=")

    # -----------------------------
    # Domain features
    # -----------------------------

    domain = hostname

    prefix_suffix = 1 if "-" in domain else 0

    # -----------------------------
    # Entropy
    # -----------------------------

    entropy = calculate_entropy(full_url)

    # -----------------------------
    # Return features
    # -----------------------------

    features = {
        "entropy": entropy,
        "url_length": url_length,
        "num_www": num_www,
        "ratio_digits_url": ratio_digits_url,
        "num_subdomains": num_subdomains,
        "length_hostname": hostname_length,
        "random_domain": 0,  # we'll handle this separately
        "num_dots": num_dots,
        "num_hyphens": num_hyphens,
        "ratio_digits_host": ratio_digits_host,
        "prefix_suffix": prefix_suffix,
        "num_question_mark": num_question_mark,
        "num_eq": num_eq
    }

    return features

In [64]:
test_url = "https://www.google.com/search?q=machine+learning"

features = extract_url_features(test_url)

features

{'entropy': 4.334147917027245,
 'url_length': 48,
 'num_www': 1,
 'ratio_digits_url': 0.0,
 'num_subdomains': 1,
 'length_hostname': 14,
 'random_domain': 0,
 'num_dots': 2,
 'num_hyphens': 0,
 'ratio_digits_host': 0.0,
 'prefix_suffix': 0,
 'num_question_mark': 1,
 'num_eq': 1}

In [65]:
sample_url = df_clean.iloc[0]["url"]

print("URL:")
print(sample_url)

print("\nDataset features:")

display(
    df_clean[
        df_clean["url"] == sample_url
    ][[
        "entropy",
        "url_length",
        "num_www",
        "ratio_digits_url",
        "num_subdomains",
        "length_hostname",
        "random_domain",
        "num_dots",
        "num_hyphens",
        "ratio_digits_host",
        "prefix_suffix",
        "num_question_mark",
        "num_eq"
    ]]
)

URL:
https://manua.ls

Dataset features:


,entropy,url_length,num_www,ratio_digits_url,num_subdomains,length_hostname,random_domain,num_dots,num_hyphens,ratio_digits_host,prefix_suffix,num_question_mark,num_eq
0,3.5,16,0,0.0,0,8,0,1,0,0.0,0,0,0


In [66]:
print("\nOur extracted features:")

display(
    pd.DataFrame([extract_url_features(sample_url)])
)


Our extracted features:


,entropy,url_length,num_www,ratio_digits_url,num_subdomains,length_hostname,random_domain,num_dots,num_hyphens,ratio_digits_host,prefix_suffix,num_question_mark,num_eq
0,3.5,16,0,0.0,0,8,0,1,0,0.0,0,0,0


In [67]:
import pandas as pd
import numpy as np

feature_columns = [
    "entropy",
    "url_length",
    "num_www",
    "ratio_digits_url",
    "num_subdomains",
    "length_hostname",
    "random_domain",
    "num_dots",
    "num_hyphens",
    "ratio_digits_host",
    "prefix_suffix",
    "num_question_mark",
    "num_eq"
]

sample = df_clean.sample(
    n=100,
    random_state=42
)

mismatches = []

for idx, row in sample.iterrows():

    extracted = extract_url_features(row["url"])

    for feature in feature_columns:

        dataset_value = row[feature]
        extracted_value = extracted[feature]

        if not np.isclose(
            dataset_value,
            extracted_value,
            atol=1e-6
        ):
            mismatches.append({
                "index": idx,
                "url": row["url"],
                "feature": feature,
                "dataset": dataset_value,
                "extracted": extracted_value
            })

mismatch_df = pd.DataFrame(mismatches)

print("Total comparisons:",
      len(sample) * len(feature_columns))

print("Total mismatches:",
      len(mismatch_df))

display(mismatch_df.head(20))

Total comparisons: 1300
Total mismatches: 32


,index,url,feature,dataset,extracted
0,28418,https://dc.gov,random_domain,1,0
1,18878,https://mytdatasecureicu.app/v/s/a/session_ind...,random_domain,1,0
2,47723,https://cv76108-wordpress-zmpmr.tw1.ru/wp-cont...,random_domain,1,0
3,47723,https://cv76108-wordpress-zmpmr.tw1.ru/wp-cont...,prefix_suffix,0,1
4,30572,https://deliveroo.co.uk,num_subdomains,0,1
5,14692,https://us17.campaign-archive.com/?e=5a66c928e...,random_domain,1,0
6,27632,https://q-r.to/bfTF0j?IQ=74003j3d,random_domain,1,0
7,12232,https://helpdesk-umwedu.weebly.com/,prefix_suffix,0,1
8,19099,https://identifiez-vous.godaddysites.com/,prefix_suffix,0,1
9,30037,https://t.co/fhLWCgBW4J,random_domain,1,0


In [71]:
import tldextract

def get_num_subdomains(url):

    ext = tldextract.extract(url)

    if not ext.subdomain:
        return 0

    return len(ext.subdomain.split("."))

In [69]:
!pip install -q tldextract

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 3.3 MB/s eta 0:00:00


In [72]:
test_urls = [
    "https://www.chevrolet.com.ar",
    "https://www.google.com",
    "https://mail.google.com",
    "https://a.b.example.com"
]

for url in test_urls:
    print(url, "→", get_num_subdomains(url))

https://www.chevrolet.com.ar → 1
https://www.google.com → 1
https://mail.google.com → 1
https://a.b.example.com → 2


In [73]:
import tldextract

def get_num_subdomains(url):
    ext = tldextract.extract(url)

    if not ext.subdomain:
        return 0

    return len(ext.subdomain.split("."))

In [74]:
test_urls = [
    "https://www.google.com",
    "https://mail.google.com",
    "https://a.b.example.com",
    "https://www.chevrolet.com.ar",
    "https://example.co.uk"
]

for url in test_urls:
    print(url, "→", get_num_subdomains(url))

https://www.google.com → 1
https://mail.google.com → 1
https://a.b.example.com → 2
https://www.chevrolet.com.ar → 1
https://example.co.uk → 0


In [75]:
prefix_mismatches = mismatch_df[
    mismatch_df["feature"] == "prefix_suffix"
]

display(prefix_mismatches)

,index,url,feature,dataset,extracted
3,47723,https://cv76108-wordpress-zmpmr.tw1.ru/wp-cont...,prefix_suffix,0,1
7,12232,https://helpdesk-umwedu.weebly.com/,prefix_suffix,0,1
8,19099,https://identifiez-vous.godaddysites.com/,prefix_suffix,0,1
11,13284,https://home-io-trzo-login.pages.dev/,prefix_suffix,0,1
13,57969,https://validate-bell0-mailbox.weebly.com/,prefix_suffix,0,1
15,47404,https://pub-2850cd5d83b9406498615dc0f50afc0d.r...,prefix_suffix,0,1
17,34770,https://help-trzorstart.gitbook.io/us/,prefix_suffix,0,1
22,18378,https://pub-713a6cc1335d4ba1b83057a9bd89b15a.r...,prefix_suffix,0,1
27,52627,https://exod-usweb.pages.dev,prefix_suffix,0,1
29,37585,https://ionos-webmail-de-ionos-id-login-ionos....,prefix_suffix,0,1


In [76]:
prefix_indices = prefix_mismatches["index"].tolist()

display(
    df_clean.loc[
        prefix_indices,
        ["url", "domain", "prefix_suffix"]
    ]
)

,url,domain,prefix_suffix
47723,https://cv76108-wordpress-zmpmr.tw1.ru/wp-cont...,cv76108-wordpress-zmpmr.tw1.ru,0
12232,https://helpdesk-umwedu.weebly.com/,helpdesk-umwedu.weebly.com,0
19099,https://identifiez-vous.godaddysites.com/,identifiez-vous.godaddysites.com,0
13284,https://home-io-trzo-login.pages.dev/,home-io-trzo-login.pages.dev,0
57969,https://validate-bell0-mailbox.weebly.com/,validate-bell0-mailbox.weebly.com,0
47404,https://pub-2850cd5d83b9406498615dc0f50afc0d.r...,pub-2850cd5d83b9406498615dc0f50afc0d.r2.dev,0
34770,https://help-trzorstart.gitbook.io/us/,help-trzorstart.gitbook.io,0
18378,https://pub-713a6cc1335d4ba1b83057a9bd89b15a.r...,pub-713a6cc1335d4ba1b83057a9bd89b15a.r2.dev,0
52627,https://exod-usweb.pages.dev,exod-usweb.pages.dev,0
37585,https://ionos-webmail-de-ionos-id-login-ionos....,ionos-webmail-de-ionos-id-login-ionos.blogspot...,0


In [77]:
random_mismatches = mismatch_df[
    mismatch_df["feature"] == "random_domain"
]

display(random_mismatches)

,index,url,feature,dataset,extracted
0,28418,https://dc.gov,random_domain,1,0
1,18878,https://mytdatasecureicu.app/v/s/a/session_ind...,random_domain,1,0
2,47723,https://cv76108-wordpress-zmpmr.tw1.ru/wp-cont...,random_domain,1,0
5,14692,https://us17.campaign-archive.com/?e=5a66c928e...,random_domain,1,0
6,27632,https://q-r.to/bfTF0j?IQ=74003j3d,random_domain,1,0
9,30037,https://t.co/fhLWCgBW4J,random_domain,1,0
10,2359,https://q-r.to/bfMR70,random_domain,1,0
14,47404,https://pub-2850cd5d83b9406498615dc0f50afc0d.r...,random_domain,1,0
16,25466,https://cdn-fc.com,random_domain,1,0
18,30238,https://expedition-relais.com/as.php,random_domain,1,0


In [78]:
random_indices = random_mismatches["index"].tolist()

display(
    df_clean.loc[
        random_indices,
        ["url", "domain", "random_domain"]
    ]
)

,url,domain,random_domain
28418,https://dc.gov,dc.gov,1
18878,https://mytdatasecureicu.app/v/s/a/session_ind...,mytdatasecureicu.app,1
47723,https://cv76108-wordpress-zmpmr.tw1.ru/wp-cont...,cv76108-wordpress-zmpmr.tw1.ru,1
14692,https://us17.campaign-archive.com/?e=5a66c928e...,us17.campaign-archive.com,1
27632,https://q-r.to/bfTF0j?IQ=74003j3d,q-r.to,1
30037,https://t.co/fhLWCgBW4J,t.co,1
2359,https://q-r.to/bfMR70,q-r.to,1
47404,https://pub-2850cd5d83b9406498615dc0f50afc0d.r...,pub-2850cd5d83b9406498615dc0f50afc0d.r2.dev,1
25466,https://cdn-fc.com,cdn-fc.com,1
30238,https://expedition-relais.com/as.php,expedition-relais.com,1


In [79]:
# Check XGBoost importance of the two difficult features

importance_df = pd.DataFrame({
    "feature": url_features,
    "importance": xgb_model.feature_importances_
}).sort_values("importance", ascending=False)

display(importance_df)

,feature,importance
0,entropy,0.198639
1,url_length,0.171328
2,num_www,0.147780
4,num_subdomains,0.117610
5,length_hostname,0.089346
3,ratio_digits_url,0.073683
9,ratio_digits_host,0.046478
12,num_eq,0.043932
7,num_dots,0.039194
6,random_domain,0.030090


In [80]:
clean_features = [
    "entropy",
    "url_length",
    "num_www",
    "ratio_digits_url",
    "num_subdomains",
    "length_hostname",
    # "random_domain",       # removed
    "num_dots",
    "num_hyphens",
    "ratio_digits_host",
    # "prefix_suffix",       # removed
    "num_question_mark",
    "num_eq"
]

X_train_clean = X_train[clean_features]
X_test_clean = X_test[clean_features]

xgb_clean = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_clean.fit(X_train_clean, y_train)

y_pred_clean = xgb_clean.predict(X_test_clean)
y_prob_clean = xgb_clean.predict_proba(X_test_clean)[:, 1]

In [81]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

clean_results = {
    "Accuracy": accuracy_score(y_test, y_pred_clean),
    "Precision": precision_score(y_test, y_pred_clean),
    "Recall": recall_score(y_test, y_pred_clean),
    "F1": f1_score(y_test, y_pred_clean),
    "ROC-AUC": roc_auc_score(y_test, y_prob_clean)
}

clean_results

{'Accuracy': 0.9931150428308382,
 'Precision': 0.998812527831379,
 'Recall': 0.988541207580432,
 'F1': 0.9936503248670998,
 'ROC-AUC': np.float64(0.9979547304400704)}

In [82]:
url_features_final = [
    "entropy",
    "url_length",
    "num_www",
    "ratio_digits_url",
    "num_subdomains",
    "length_hostname",
    "num_dots",
    "num_hyphens",
    "ratio_digits_host",
    "num_question_mark",
    "num_eq"
]

In [83]:
import math
import re
from collections import Counter
from urllib.parse import urlparse
import tldextract


def calculate_entropy(text):
    if not text:
        return 0

    counter = Counter(text)
    length = len(text)

    entropy = 0

    for count in counter.values():
        probability = count / length
        entropy -= probability * math.log2(probability)

    return entropy


def extract_url_features(url):

    # -----------------------------
    # 1. Parse URL
    # -----------------------------
    parsed = urlparse(url)

    hostname = parsed.hostname or ""


    # -----------------------------
    # 2. Extract registered domain
    #    and subdomains
    # -----------------------------
    ext = tldextract.extract(url)

    if ext.subdomain:
        num_subdomains = len(ext.subdomain.split("."))
    else:
        num_subdomains = 0


    # -----------------------------
    # 3. URL-level features
    # -----------------------------
    url_length = len(url)

    entropy = calculate_entropy(url)

    num_www = url.lower().count("www")

    digit_count_url = sum(char.isdigit() for char in url)

    ratio_digits_url = (
        digit_count_url / url_length
        if url_length > 0
        else 0
    )

    num_dots = url.count(".")

    num_hyphens = url.count("-")

    num_question_mark = url.count("?")

    num_eq = url.count("=")


    # -----------------------------
    # 4. Hostname-level features
    # -----------------------------
    length_hostname = len(hostname)

    digit_count_host = sum(
        char.isdigit() for char in hostname
    )

    ratio_digits_host = (
        digit_count_host / length_hostname
        if length_hostname > 0
        else 0
    )


    # -----------------------------
    # 5. Return exact feature order
    # -----------------------------
    return {
        "entropy": entropy,
        "url_length": url_length,
        "num_www": num_www,
        "ratio_digits_url": ratio_digits_url,
        "num_subdomains": num_subdomains,
        "length_hostname": length_hostname,
        "num_dots": num_dots,
        "num_hyphens": num_hyphens,
        "ratio_digits_host": ratio_digits_host,
        "num_question_mark": num_question_mark,
        "num_eq": num_eq
    }

In [84]:
test_url = "https://manua.ls"

features = extract_url_features(test_url)

features

{'entropy': 3.5,
 'url_length': 16,
 'num_www': 0,
 'ratio_digits_url': 0.0,
 'num_subdomains': 0,
 'length_hostname': 8,
 'num_dots': 1,
 'num_hyphens': 0,
 'ratio_digits_host': 0.0,
 'num_question_mark': 0,
 'num_eq': 0}

In [85]:
import random
import pandas as pd

random.seed(42)

sample_indices = random.sample(
    list(df_clean.index),
    100
)

mismatches = []

for idx in sample_indices:

    url = df_clean.loc[idx, "url"]

    extracted = extract_url_features(url)

    for feature in url_features_final:

        dataset_value = df_clean.loc[idx, feature]
        extracted_value = extracted[feature]

        # Allow tiny floating-point differences
        if isinstance(dataset_value, float):

            if not math.isclose(
                dataset_value,
                extracted_value,
                rel_tol=1e-9,
                abs_tol=1e-9
            ):
                mismatches.append({
                    "index": idx,
                    "url": url,
                    "feature": feature,
                    "dataset": dataset_value,
                    "extracted": extracted_value
                })

        else:

            if dataset_value != extracted_value:
                mismatches.append({
                    "index": idx,
                    "url": url,
                    "feature": feature,
                    "dataset": dataset_value,
                    "extracted": extracted_value
                })


mismatch_df_final = pd.DataFrame(mismatches)

print("Total comparisons:", len(sample_indices) * len(url_features_final))
print("Total mismatches:", len(mismatch_df_final))

display(mismatch_df_final)

Total comparisons: 1100
Total mismatches: 0


""


In [86]:
from xgboost import XGBClassifier

final_xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

final_xgb_model.fit(
    X_train[ url_features_final ],
    y_train
)

print("Final XGBoost model trained successfully.")

Final XGBoost model trained successfully.


In [87]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

y_pred_final = final_xgb_model.predict(
    X_test[url_features_final]
)

y_prob_final = final_xgb_model.predict_proba(
    X_test[url_features_final]
)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred_final))
print("Precision:", precision_score(y_test, y_pred_final))
print("Recall   :", recall_score(y_test, y_pred_final))
print("F1 Score :", f1_score(y_test, y_pred_final))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_final))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_final))

Accuracy : 0.9931150428308382
Precision: 0.998812527831379
Recall   : 0.988541207580432
F1 Score : 0.9936503248670998
ROC-AUC  : 0.9979547304400704

Classification Report:

              precision    recall  f1-score   support

           0       0.99      1.00      0.99      5684
           1       1.00      0.99      0.99      6807

    accuracy                           0.99     12491
   macro avg       0.99      0.99      0.99     12491
weighted avg       0.99      0.99      0.99     12491



In [88]:
import joblib

joblib.dump(
    final_xgb_model,
    "phishing_xgb_model.pkl"
)

joblib.dump(
    url_features_final,
    "feature_names.pkl"
)

print("Model saved successfully.")
print("Feature list saved successfully.")

Model saved successfully.
Feature list saved successfully.


In [89]:
def predict_url(url):

    # Extract features
    features = extract_url_features(url)

    # Convert dictionary to DataFrame
    X_input = pd.DataFrame(
        [features],
        columns=url_features_final
    )

    # Get probability
    phishing_probability = final_xgb_model.predict_proba(
        X_input
    )[0][1]

    # Prediction
    if phishing_probability >= 0.5:
        prediction = "Phishing"
    else:
        prediction = "Legitimate"

    return {
        "url": url,
        "prediction": prediction,
        "phishing_probability": phishing_probability,
        "legitimate_probability": 1 - phishing_probability
    }

In [90]:
result = predict_url("https://www.google.com")

result

{'url': 'https://www.google.com',
 'prediction': 'Legitimate',
 'phishing_probability': np.float32(0.005525957),
 'legitimate_probability': np.float32(0.99447405)}

In [91]:
result = predict_url("https://www.github.com")

result

{'url': 'https://www.github.com',
 'prediction': 'Legitimate',
 'phishing_probability': np.float32(0.0046431352),
 'legitimate_probability': np.float32(0.99535686)}

In [92]:
result = predict_url(
    "http://secure-login-account-verification-12345.com/login"
)

result

{'url': 'http://secure-login-account-verification-12345.com/login',
 'prediction': 'Phishing',
 'phishing_probability': np.float32(0.93720645),
 'legitimate_probability': np.float32(0.06279355)}

In [93]:
unseen_test_urls = [
    "https://www.microsoft.com",
    "https://www.apple.com",
    "https://www.amazon.com",
    "https://www.wikipedia.org",
    "https://www.python.org",
    "https://stackoverflow.com",
    "https://www.linkedin.com",
    "https://www.netflix.com",

    "http://secure-account-login-verification-92837.com/login",
    "http://paypal-account-security-check-48291.com/verify",
    "http://free-prize-winner-92731.com/claim",
    "http://bank-account-verification-18472.com/login"
]

In [94]:
unseen_test_urls = [
    "https://www.microsoft.com",
    "https://www.apple.com",
    "https://www.amazon.com",
    "https://www.wikipedia.org",
    "https://www.python.org",
    "https://stackoverflow.com",
    "https://www.linkedin.com",
    "https://www.netflix.com",

    "http://secure-account-login-verification-92837.com/login",
    "http://paypal-account-security-check-48291.com/verify",
    "http://free-prize-winner-92731.com/claim",
    "http://bank-account-verification-18472.com/login"
]

In [95]:
import tldextract

for url in unseen_test_urls:

    ext = tldextract.extract(url)

    if ext.registered_domain:
        domain = ext.registered_domain
    else:
        domain = ext.domain

    domain_exists = (
        df_clean["domain"]
        .astype(str)
        .str.lower()
        .eq(domain.lower())
        .any()
    )

    url_exists = (
        df_clean["normalized_url"]
        .astype(str)
        .str.lower()
        .eq(url.lower().rstrip("/"))
        .any()
    )

    print(
        url,
        "| domain in dataset:",
        domain_exists,
        "| URL in dataset:",
        url_exists
    )

/tmp/ipykernel_1303/313667722.py:7: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  if ext.registered_domain:
/tmp/ipykernel_1303/313667722.py:8: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  domain = ext.registered_domain
/tmp/ipykernel_1303/313667722.py:7: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  if ext.registered_domain:
/tmp/ipykernel_1303/313667722.py:8: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_do

https://www.microsoft.com | domain in dataset: True | URL in dataset: False
https://www.apple.com | domain in dataset: False | URL in dataset: False
https://www.amazon.com | domain in dataset: True | URL in dataset: False
https://www.wikipedia.org | domain in dataset: True | URL in dataset: False
https://www.python.org | domain in dataset: True | URL in dataset: False


/tmp/ipykernel_1303/313667722.py:7: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  if ext.registered_domain:
/tmp/ipykernel_1303/313667722.py:8: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  domain = ext.registered_domain
/tmp/ipykernel_1303/313667722.py:7: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  if ext.registered_domain:
/tmp/ipykernel_1303/313667722.py:8: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_do

https://stackoverflow.com | domain in dataset: False | URL in dataset: False
https://www.linkedin.com | domain in dataset: False | URL in dataset: False
https://www.netflix.com | domain in dataset: True | URL in dataset: False
http://secure-account-login-verification-92837.com/login | domain in dataset: False | URL in dataset: False
http://paypal-account-security-check-48291.com/verify | domain in dataset: False | URL in dataset: False
http://free-prize-winner-92731.com/claim | domain in dataset: False | URL in dataset: False
http://bank-account-verification-18472.com/login | domain in dataset: False | URL in dataset: False


/tmp/ipykernel_1303/313667722.py:7: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  if ext.registered_domain:
/tmp/ipykernel_1303/313667722.py:8: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  domain = ext.registered_domain


In [96]:
results = []

for url in unseen_test_urls:

    result = predict_url(url)

    results.append({
        "URL": url,
        "Prediction": result["prediction"],
        "Phishing Probability": float(
            result["phishing_probability"]
        )
    })

unseen_results = pd.DataFrame(results)

display(unseen_results)

,URL,Prediction,Phishing Probability
0,https://www.microsoft.com,Legitimate,0.009345
1,https://www.apple.com,Legitimate,0.009769
2,https://www.amazon.com,Legitimate,0.003954
3,https://www.wikipedia.org,Legitimate,0.018236
4,https://www.python.org,Legitimate,0.004724
5,https://stackoverflow.com,Legitimate,0.021552
6,https://www.linkedin.com,Legitimate,0.004804
7,https://www.netflix.com,Legitimate,0.009535
8,http://secure-account-login-verification-92837...,Phishing,0.937206
9,http://paypal-account-security-check-48291.com...,Phishing,0.882909


In [97]:
dns_features = [
    "domain_age",
    "dns_resolvable",
    "num_ips",
    "ttl",
    "num_name_servers"
]

display(
    df_clean[dns_features].describe(include="all").T
)

,count,mean,std,min,25%,50%,75%,max
domain_age,59812.0,4616.968083,4279.347374,-1.0,-1.0,4558.0,8611.0,15148.0
dns_resolvable,59812.0,0.930833,0.253740,0.0,1.0,1.0,1.0,1.0
num_ips,59812.0,2.159099,1.652231,0.0,1.0,2.0,3.0,28.0
ttl,59812.0,2017.700478,5192.450180,-1.0,60.0,300.0,600.0,21600.0
num_name_servers,59812.0,3.267705,1.743243,0.0,2.0,4.0,4.0,22.0


In [98]:
for feature in dns_features:
    print("\n==============================")
    print(feature)
    print("==============================")
    print(df_clean[feature].value_counts(dropna=False).head(20))


domain_age
domain_age
-1.0        18611
 10351.0     3119
 10352.0     1198
 7234.0      1116
 6453.0       993
 4841.0       788
 5099.0       567
 14306.0      507
 7235.0       457
 4842.0       447
 6454.0       375
 5745.0       327
 5100.0       321
 3249.0       274
 6953.0       236
 11161.0      229
 5001.0       226
 9089.0       203
 7373.0       195
 14307.0      172
Name: count, dtype: int64

dns_resolvable
dns_resolvable
1.0    55675
0.0     4137
NaN        2
Name: count, dtype: int64

num_ips
num_ips
1.0     21624
2.0     18442
4.0      6884
6.0      4996
0.0      4137
3.0      3242
5.0       247
8.0       146
7.0        27
10.0       26
9.0        13
11.0        7
12.0        6
16.0        4
18.0        3
20.0        3
NaN         2
14.0        2
15.0        1
27.0        1
Name: count, dtype: int64

ttl
ttl
 300.0      25520
 60.0        9794
-1.0         4137
 3600.0      4117
 21600.0     3235
 900.0       3208
 600.0       2597
 20.0        1840
 14400.0     1064
 

In [99]:
display(
    df_clean.groupby("label")[dns_features].mean()
)

,domain_age,dns_resolvable,num_ips,ttl,num_name_servers
label,,,,,
legitimate,5294.548965,0.993723,1.959128,2011.495786,3.197306
phishing,4006.122079,0.874138,2.339374,2023.294071,3.331172


In [100]:
for feature in dns_features:

    print("\n==============================")
    print(feature)
    print("==============================")

    display(
        df_clean.groupby("label")[feature]
        .describe()
    )


domain_age


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
legitimate,28357.0,5294.548965,4175.070058,-1.0,370.0,5397.0,9298.0,15148.0
phishing,31455.0,4006.122079,4280.316907,-1.0,-1.0,3068.0,7234.0,14307.0



dns_resolvable


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
legitimate,28357.0,0.993723,0.078981,0.0,1.0,1.0,1.0,1.0
phishing,31455.0,0.874138,0.331699,0.0,1.0,1.0,1.0,1.0



num_ips


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
legitimate,28357.0,1.959128,1.265970,0.0,1.0,2.0,2.0,28.0
phishing,31455.0,2.339374,1.917698,0.0,1.0,2.0,4.0,12.0



ttl


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
legitimate,28357.0,2011.495786,4908.352303,-1.0,120.0,300.0,600.0,21600.0
phishing,31455.0,2023.294071,5435.927643,-1.0,60.0,300.0,300.0,21600.0



num_name_servers


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
legitimate,28357.0,3.197306,1.507155,0.0,2.0,3.0,4.0,22.0
phishing,31455.0,3.331172,1.929326,0.0,2.0,4.0,4.0,10.0


In [101]:
dns_failed = df_clean[
    df_clean["dns_resolvable"] == 0
]

display(
    dns_failed[
        [
            "url",
            "domain",
            "label",
            "domain_age",
            "dns_resolvable",
            "num_ips",
            "ttl",
            "num_name_servers"
        ]
    ].head(30)
)

,url,domain,label,domain_age,dns_resolvable,num_ips,ttl,num_name_servers
38,https://share.hsforms.com/14mI_206kTRGfwzo7boR...,share.hsforms.com,phishing,-1.0,0.0,0.0,-1.0,0.0
40,https://l.wl.co/l?u=https://qr-codes.io/p8BGXc,l.wl.co,phishing,-1.0,0.0,0.0,-1.0,0.0
50,https://allegrolokalnie.0b3918v.shop/,allegrolokalnie.0b3918v.shop,phishing,-1.0,0.0,0.0,-1.0,0.0
54,https://poczta.ncelodir.cfd/payment/cae44c860d...,poczta.ncelodir.cfd,phishing,-1.0,0.0,0.0,-1.0,0.0
55,http://43.156.237.181/ServiceLogin?passive=120...,43.156.237.181,phishing,-1.0,0.0,0.0,-1.0,0.0
74,http://tekdatagt.com/mail.verify/Sites/index.html,tekdatagt.com,phishing,-1.0,0.0,0.0,-1.0,0.0
81,https://mashreghnews.ir,mashreghnews.ir,legitimate,-1.0,0.0,0.0,-1.0,0.0
84,https://www.mobiletoken-bancasella.com/indexip...,mobiletoken-bancasella.com,phishing,107.0,0.0,0.0,-1.0,0.0
88,https://avend.cfd/jp/,avend.cfd,phishing,-1.0,0.0,0.0,-1.0,0.0
106,http://43.156.0.130/InteractiveLogin?continue=...,43.156.0.130,phishing,-1.0,0.0,0.0,-1.0,0.0


In [102]:
dns_relationship = pd.crosstab(
    df_clean["dns_resolvable"],
    df_clean["num_ips"]
)

display(dns_relationship)

num_ips,0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,14.0,15.0,16.0,18.0,20.0,27.0,28.0
dns_resolvable,,,,,,,,,,,,,,,,,,,,
0.0,4137,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1.0,0,21624,18442,3242,6884,247,4996,27,146,13,26,7,6,2,1,4,3,3,1,1


In [103]:
display(
    df_clean[
        ["dns_resolvable", "num_ips", "ttl", "num_name_servers"]
    ]
    .groupby("dns_resolvable")
    .agg(["mean", "min", "max"])
)

num_ips                     ttl                \
                    mean  min   max         mean  min      max   
dns_resolvable                                                   
0.0             0.000000  0.0   0.0    -1.000000 -1.0     -1.0   
1.0             2.319533  1.0  28.0  2167.702524  0.0  21600.0   

               num_name_servers             
                           mean  min   max  
dns_resolvable                              
0.0                    0.000000  0.0   0.0  
1.0                    3.510516  0.0  22.0

In [104]:
domain_age_unknown = df_clean[
    df_clean["domain_age"] == -1
]

display(
    domain_age_unknown[
        [
            "url",
            "domain",
            "label",
            "domain_age",
            "dns_resolvable",
            "num_ips",
            "ttl",
            "num_name_servers"
        ]
    ].head(30)
)

,url,domain,label,domain_age,dns_resolvable,num_ips,ttl,num_name_servers
0,https://manua.ls,manua.ls,legitimate,-1.0,1.0,1.0,3600.0,4.0
4,https://strato.de,strato.de,legitimate,-1.0,1.0,1.0,3600.0,4.0
11,https://interactivedns.com,interactivedns.com,legitimate,-1.0,1.0,1.0,300.0,2.0
15,https://flow.page/vbybtvvrtby65gf4,flow.page,phishing,-1.0,1.0,2.0,300.0,2.0
21,https://locker-prbs-fr.com/,locker-prbs-fr.com,phishing,-1.0,1.0,1.0,900.0,3.0
23,https://urlz.fr/uRb2,urlz.fr,phishing,-1.0,1.0,2.0,300.0,2.0
24,https://pitch.com,pitch.com,legitimate,-1.0,1.0,4.0,60.0,4.0
25,https://culturedcode.com,culturedcode.com,legitimate,-1.0,1.0,4.0,60.0,4.0
28,https://applesupport-2oa.pages.dev/,applesupport-2oa.pages.dev,phishing,-1.0,1.0,2.0,300.0,2.0
29,https://q-r.to/bfeYdf,q-r.to,phishing,-1.0,1.0,4.0,60.0,4.0


In [105]:
print("Total domain_age = -1:",
      len(domain_age_unknown))

print("\nLabel distribution:")
display(
    domain_age_unknown["label"].value_counts()
)

Total domain_age = -1: 18611

Label distribution:


,count
label,
phishing,12052
legitimate,6559


In [106]:
dns_summary = (
    df_clean
    .groupby("label")["dns_resolvable"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("percentage")
    .reset_index()
)

display(dns_summary)

,label,dns_resolvable,percentage
0,legitimate,1.0,99.372289
1,legitimate,0.0,0.627711
2,phishing,1.0,87.413766
3,phishing,0.0,12.586234


In [107]:
dns_features = [
    "domain_age",
    "dns_resolvable",
    "num_ips",
    "ttl",
    "num_name_servers"
]

In [108]:
print("X_train rows:", len(X_train))
print("X_test rows:", len(X_test))

print("First train indices:", X_train.index[:5].tolist())
print("First test indices:", X_test.index[:5].tolist())

X_train rows: 47323
X_test rows: 12491
First train indices: [0, 2, 3, 4, 5]
First test indices: [1, 9, 14, 19, 25]


In [109]:
X_train_dns_raw = df_clean.loc[
    X_train.index,
    dns_features
]

X_test_dns_raw = df_clean.loc[
    X_test.index,
    dns_features
]

print("DNS train shape:", X_train_dns_raw.shape)
print("DNS test shape :", X_test_dns_raw.shape)

DNS train shape: (47323, 5)
DNS test shape : (12491, 5)


In [110]:
from sklearn.impute import SimpleImputer

dns_imputer = SimpleImputer(strategy="median")

X_train_dns = dns_imputer.fit_transform(
    X_train_dns_raw
)

X_test_dns = dns_imputer.transform(
    X_test_dns_raw
)

print("DNS train shape:", X_train_dns.shape)
print("DNS test shape :", X_test_dns.shape)

DNS train shape: (47323, 5)
DNS test shape : (12491, 5)


In [111]:
from xgboost import XGBClassifier

dns_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

dns_model.fit(
    X_train_dns,
    y_train
)

print("DNS model trained successfully.")

DNS model trained successfully.


In [112]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

dns_pred = dns_model.predict(X_test_dns)

dns_prob = dns_model.predict_proba(
    X_test_dns
)[:, 1]

dns_results = {
    "Accuracy": accuracy_score(y_test, dns_pred),
    "Precision": precision_score(y_test, dns_pred),
    "Recall": recall_score(y_test, dns_pred),
    "F1": f1_score(y_test, dns_pred),
    "ROC-AUC": roc_auc_score(y_test, dns_prob)
}

dns_results

{'Accuracy': 0.7058682251220879,
 'Precision': 0.8388492321003678,
 'Recall': 0.5697076538857059,
 'F1': 0.678565179352581,
 'ROC-AUC': np.float64(0.7755068570489851)}

In [113]:
dns_importance = pd.DataFrame({
    "feature": dns_features,
    "importance": dns_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(dns_importance)

,feature,importance
1,dns_resolvable,0.419229
2,num_ips,0.202377
0,domain_age,0.177349
4,num_name_servers,0.110885
3,ttl,0.090160


In [114]:
combined_features = url_features_final + dns_features

print("Total features:", len(combined_features))
print(combined_features)

Total features: 16
['entropy', 'url_length', 'num_www', 'ratio_digits_url', 'num_subdomains', 'length_hostname', 'num_dots', 'num_hyphens', 'ratio_digits_host', 'num_question_mark', 'num_eq', 'domain_age', 'dns_resolvable', 'num_ips', 'ttl', 'num_name_servers']


In [115]:
X_train_combined_url = df_clean.loc[
    X_train.index,
    url_features_final
]

X_test_combined_url = df_clean.loc[
    X_test.index,
    url_features_final
]

In [116]:
X_train_combined_dns = df_clean.loc[
    X_train.index,
    dns_features
]

X_test_combined_dns = df_clean.loc[
    X_test.index,
    dns_features
]

In [117]:
dns_imputer_combined = SimpleImputer(strategy="median")

X_train_dns_imputed = dns_imputer_combined.fit_transform(
    X_train_combined_dns
)

X_test_dns_imputed = dns_imputer_combined.transform(
    X_test_combined_dns
)

In [118]:
import numpy as np

X_train_combined = np.hstack([
    X_train_combined_url.values,
    X_train_dns_imputed
])

X_test_combined = np.hstack([
    X_test_combined_url.values,
    X_test_dns_imputed
])

print("Combined train shape:", X_train_combined.shape)
print("Combined test shape :", X_test_combined.shape)

Combined train shape: (47323, 16)
Combined test shape : (12491, 16)


In [119]:
combined_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

combined_xgb.fit(
    X_train_combined,
    y_train
)

print("Combined model trained successfully.")

Combined model trained successfully.


In [120]:
combined_pred = combined_xgb.predict(
    X_test_combined
)

combined_prob = combined_xgb.predict_proba(
    X_test_combined
)[:, 1]

combined_results = {
    "Accuracy": accuracy_score(
        y_test,
        combined_pred
    ),
    "Precision": precision_score(
        y_test,
        combined_pred
    ),
    "Recall": recall_score(
        y_test,
        combined_pred
    ),
    "F1": f1_score(
        y_test,
        combined_pred
    ),
    "ROC-AUC": roc_auc_score(
        y_test,
        combined_prob
    )
}

combined_results

{'Accuracy': 0.9931150428308382,
 'Precision': 0.9982209043736101,
 'Recall': 0.9891288379609225,
 'F1': 0.9936540731995277,
 'ROC-AUC': np.float64(0.9992782557013018)}

In [121]:
comparison = pd.DataFrame({
    "URL-only": {
        "Accuracy": accuracy_score(y_test, y_pred_final),
        "Precision": precision_score(y_test, y_pred_final),
        "Recall": recall_score(y_test, y_pred_final),
        "F1": f1_score(y_test, y_pred_final),
        "ROC-AUC": roc_auc_score(y_test, y_prob_final)
    },

    "DNS-only": dns_results,

    "URL + DNS": combined_results
})

display(comparison)

,URL-only,DNS-only,URL + DNS
Accuracy,0.993115,0.705868,0.993115
Precision,0.998813,0.838849,0.998221
Recall,0.988541,0.569708,0.989129
F1,0.993650,0.678565,0.993654
ROC-AUC,0.997955,0.775507,0.999278


In [122]:
combined_importance = pd.DataFrame({
    "feature": combined_features,
    "importance": combined_xgb.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(combined_importance)

,feature,importance
0,entropy,0.244756
1,url_length,0.170260
2,num_www,0.115897
5,length_hostname,0.083579
12,dns_resolvable,0.078929
3,ratio_digits_url,0.074248
4,num_subdomains,0.065923
8,ratio_digits_host,0.055738
6,num_dots,0.028688
11,domain_age,0.023088


In [123]:
combined_errors = df_clean.loc[X_test.index].copy()

combined_errors["actual"] = y_test
combined_errors["predicted"] = combined_pred
combined_errors["probability"] = combined_prob

combined_errors["error"] = (
    combined_errors["actual"] !=
    combined_errors["predicted"]
)

display(
    combined_errors[combined_errors["error"]][
        [
            "url",
            "label",
            "domain_age",
            "dns_resolvable",
            "num_ips",
            "ttl",
            "num_name_servers",
            "probability",
            "predicted"
        ]
    ].head(30)
)

,url,label,domain_age,dns_resolvable,num_ips,ttl,num_name_servers,probability,predicted
234,https://www-nomura.njatx.cn,phishing,1078.0,1.0,1.0,600.0,2.0,0.099862,0
1455,http://cw-net.de,phishing,-1.0,1.0,1.0,21600.0,2.0,0.010224,0
3882,https://fireflyskinz.com/,phishing,198.0,1.0,2.0,300.0,2.0,0.194481,0
4193,https://www.workjunction.in/,phishing,2512.0,1.0,2.0,300.0,2.0,0.080414,0
4616,https://hardened-php.net,legitimate,2388.0,1.0,1.0,14400.0,2.0,0.574713,1
5650,https://satisfactory-calculator.com,legitimate,-1.0,1.0,1.0,300.0,2.0,0.585035,1
5850,http://synergyminerals.in,phishing,-1.0,1.0,1.0,14400.0,2.0,0.331251,0
7890,https://reipojtpoerjp.com,phishing,280.0,1.0,2.0,300.0,2.0,0.054150,0
9290,https://internetbankingonline.digital,phishing,-1.0,1.0,2.0,300.0,2.0,0.356833,0
9834,http://limonizza.de/,phishing,-1.0,1.0,1.0,600.0,3.0,0.029887,0


In [124]:
combined_importance = pd.DataFrame({
    "feature": combined_features,
    "importance": combined_xgb.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(combined_importance)

,feature,importance
0,entropy,0.244756
1,url_length,0.170260
2,num_www,0.115897
5,length_hostname,0.083579
12,dns_resolvable,0.078929
3,ratio_digits_url,0.074248
4,num_subdomains,0.065923
8,ratio_digits_host,0.055738
6,num_dots,0.028688
11,domain_age,0.023088


In [125]:
# Predictions from URL-only model
url_errors = (
    y_pred_final != y_test
)

# Predictions from combined model
combined_errors_mask = (
    combined_pred != y_test
)

print("URL-only errors:", url_errors.sum())
print("Combined errors:", combined_errors_mask.sum())

print("\nURL-only correct:", (~url_errors).sum())
print("Combined correct:", (~combined_errors_mask).sum())

URL-only errors: 86
Combined errors: 86

URL-only correct: 12405
Combined correct: 12405


In [126]:
# Cases where URL-only was wrong but combined model became correct
dns_rescued = (
    url_errors &
    ~combined_errors_mask
)

# Cases where URL-only was correct but combined model became wrong
dns_hurt = (
    ~url_errors &
    combined_errors_mask
)

print("Cases DNS/combined model rescued:", dns_rescued.sum())
print("Cases combined model changed correct → wrong:", dns_hurt.sum())

Cases DNS/combined model rescued: 29
Cases combined model changed correct → wrong: 29


In [127]:
!pip install -q dnspython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 6.9 MB/s eta 0:00:00


In [128]:
import dns.resolver
import tldextract

In [129]:
import dns.resolver
import socket

import dns.resolver
import tldextract
from urllib.parse import urlparse


def get_live_dns_features(url):

    # -----------------------------------
    # Extract hostname
    # -----------------------------------
    parsed = urlparse(url)
    hostname = parsed.hostname

    if not hostname:
        return {
            "dns_resolvable": 0,
            "num_ips": 0,
            "ttl": -1,
            "num_name_servers": 0
        }

    # -----------------------------------
    # Extract registered/root domain
    # -----------------------------------
    ext = tldextract.extract(url)

    registered_domain = ext.registered_domain

    if not registered_domain:
        registered_domain = hostname

    resolver = dns.resolver.Resolver()

    dns_resolvable = 0
    num_ips = 0
    ttl = -1
    num_name_servers = 0

    # -----------------------------------
    # A records
    # -----------------------------------
    try:

        answer = resolver.resolve(
            hostname,
            "A"
        )

        ips = set(
            record.address
            for record in answer
        )

        num_ips = len(ips)

        if answer.rrset is not None:
            ttl = answer.rrset.ttl

        if num_ips > 0:
            dns_resolvable = 1

    except Exception:
        pass

    # -----------------------------------
    # NS records
    # Query registered domain, NOT hostname
    # -----------------------------------
    try:

        ns_answer = resolver.resolve(
            registered_domain,
            "NS"
        )

        num_name_servers = len(
            list(ns_answer)
        )

    except Exception:
        pass

    return {
        "dns_resolvable": dns_resolvable,
        "num_ips": num_ips,
        "ttl": ttl,
        "num_name_servers": num_name_servers
    }

In [130]:
test_dns_urls = [
    "https://www.google.com",
    "https://www.github.com",
    "https://www.microsoft.com",
    "https://www.apple.com",
    "https://www.amazon.com",
    "https://www.wikipedia.org",
    "https://www.python.org"
]

for url in test_dns_urls:

    print("\nURL:", url)

    dns_info = get_live_dns_features(url)

    print(dns_info)


URL: https://www.google.com
{'dns_resolvable': 1, 'num_ips': 8, 'ttl': 300, 'num_name_servers': 4}

URL: https://www.github.com
{'dns_resolvable': 1, 'num_ips': 1, 'ttl': 60, 'num_name_servers': 8}

URL: https://www.microsoft.com
{'dns_resolvable': 1, 'num_ips': 1, 'ttl': 20, 'num_name_servers': 4}

URL: https://www.apple.com
{'dns_resolvable': 1, 'num_ips': 1, 'ttl': 20, 'num_name_servers': 4}

URL: https://www.amazon.com


/tmp/ipykernel_1303/4118925904.py:30: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  registered_domain = ext.registered_domain


{'dns_resolvable': 1, 'num_ips': 1, 'ttl': 60, 'num_name_servers': 4}

URL: https://www.wikipedia.org
{'dns_resolvable': 1, 'num_ips': 1, 'ttl': 159, 'num_name_servers': 3}

URL: https://www.python.org
{'dns_resolvable': 1, 'num_ips': 4, 'ttl': 37, 'num_name_servers': 4}


In [131]:

live_dns_results = []

for url in unseen_test_urls:

    dns_info = get_live_dns_features(url)

    live_dns_results.append({
        "URL": url,
        **dns_info
    })

live_dns_df = pd.DataFrame(
    live_dns_results
)

display(live_dns_df)

/tmp/ipykernel_1303/4118925904.py:30: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  registered_domain = ext.registered_domain


,URL,dns_resolvable,num_ips,ttl,num_name_servers
0,https://www.microsoft.com,1,1,18,4
1,https://www.apple.com,1,1,18,4
2,https://www.amazon.com,1,1,58,4
3,https://www.wikipedia.org,1,1,1,3
4,https://www.python.org,1,4,14,4
5,https://stackoverflow.com,1,1,300,2
6,https://www.linkedin.com,1,2,143,8
7,https://www.netflix.com,1,2,60,4
8,http://secure-account-login-verification-92837...,0,0,-1,0
9,http://paypal-account-security-check-48291.com...,0,0,-1,0


In [132]:
test_subdomain_urls = [
    "https://www.google.com",
    "https://mail.google.com",
    "https://accounts.google.com",
    "https://a.b.example.com",
    "https://www.chevrolet.com.ar"
]

for url in test_subdomain_urls:

    ext = tldextract.extract(url)

    print("\nURL:", url)
    print("Hostname:", urlparse(url).hostname)
    print("Registered domain:", ext.registered_domain)
    print("Subdomain:", ext.subdomain)

    print(
        "DNS:",
        get_live_dns_features(url)
    )


URL: https://www.google.com
Hostname: www.google.com
Registered domain: google.com
Subdomain: www
DNS: {'dns_resolvable': 1, 'num_ips': 8, 'ttl': 296, 'num_name_servers': 4}

URL: https://mail.google.com
Hostname: mail.google.com
Registered domain: google.com
Subdomain: mail
DNS: {'dns_resolvable': 1, 'num_ips': 4, 'ttl': 300, 'num_name_servers': 4}

URL: https://accounts.google.com
Hostname: accounts.google.com
Registered domain: google.com
Subdomain: accounts
DNS: {'dns_resolvable': 1, 'num_ips': 1, 'ttl': 300, 'num_name_servers': 4}

URL: https://a.b.example.com
Hostname: a.b.example.com
Registered domain: example.com
Subdomain: a.b
DNS: {'dns_resolvable': 0, 'num_ips': 0, 'ttl': -1, 'num_name_servers': 2}

URL: https://www.chevrolet.com.ar
Hostname: www.chevrolet.com.ar
Registered domain: chevrolet.com.ar
Subdomain: www
DNS: {'dns_resolvable': 1, 'num_ips': 2, 'ttl': 20, 'num_name_servers': 4}


/tmp/ipykernel_1303/3914519048.py:15: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  print("Registered domain:", ext.registered_domain)
/tmp/ipykernel_1303/4118925904.py:30: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  registered_domain = ext.registered_domain


In [133]:
!pip install -q requests

In [134]:
import requests
from datetime import datetime, timezone

In [135]:
import requests

RDAP_BOOTSTRAP_URL = "https://data.iana.org/rdap/dns.json"

rdap_bootstrap = requests.get(
    RDAP_BOOTSTRAP_URL,
    timeout=10
).json()

print("RDAP bootstrap loaded.")

RDAP bootstrap loaded.


In [136]:
def get_rdap_server(domain):

    tld = domain.split(".")[-1].lower()

    services = rdap_bootstrap.get("services", [])

    for service in services:

        tlds = service[0]
        urls = service[1]

        if tld in tlds:
            return urls[0]

    return None

In [137]:
test_domains = [
    "google.com",
    "github.com",
    "python.org",
    "google.co.uk",
    "example.in"
]

for domain in test_domains:

    print(
        domain,
        "→",
        get_rdap_server(domain)
    )

google.com → https://rdap.verisign.com/com/v1/
github.com → https://rdap.verisign.com/com/v1/
python.org → https://rdap.publicinterestregistry.org/rdap/
google.co.uk → https://rdap.nominet.uk/uk/
example.in → https://rdap.nixiregistry.in/rdap/


In [138]:
def get_domain_registration_date(domain):

    rdap_server = get_rdap_server(domain)

    if not rdap_server:
        return None

    url = (
        rdap_server.rstrip("/")
        + "/domain/"
        + domain
    )

    try:

        response = requests.get(
            url,
            timeout=10,
            headers={
                "Accept": "application/rdap+json"
            }
        )

        if response.status_code != 200:
            return None

        data = response.json()

    except Exception:
        return None


    # Look for registration event
    events = data.get("events", [])

    for event in events:

        if event.get("eventAction") == "registration":

            registration_date = event.get(
                "eventDate"
            )

            if registration_date:
                return registration_date

    return None

In [139]:
def get_domain_age(domain):

    registration_date = (
        get_domain_registration_date(domain)
    )

    if not registration_date:
        return -1

    try:

        registration_dt = datetime.fromisoformat(
            registration_date.replace("Z", "+00:00")
        )

        now = datetime.now(timezone.utc)

        age_days = (
            now - registration_dt
        ).days

        return max(age_days, 0)

    except Exception:
        return -1

In [140]:
test_domains = [
    "google.com",
    "github.com",
    "microsoft.com",
    "apple.com",
    "python.org"
]

for domain in test_domains:

    age = get_domain_age(domain)

    print(
        domain,
        "→",
        age,
        "days"
    )

google.com → 10606 days
github.com → 6929 days
microsoft.com → 12934 days
apple.com → 14467 days
python.org → 11509 days


In [141]:
def get_live_domain_intelligence(url):

    parsed = urlparse(url)

    hostname = parsed.hostname

    if not hostname:
        return {
            "domain_age": -1,
            "dns_resolvable": 0,
            "num_ips": 0,
            "ttl": -1,
            "num_name_servers": 0
        }

    ext = tldextract.extract(url)

    registered_domain = (
        ext.top_domain_under_public_suffix
    )

    dns_info = get_live_dns_features(url)

    domain_age = get_domain_age(
        registered_domain
    )

    return {
        "domain_age": domain_age,
        **dns_info
    }

In [142]:
test_urls = [
    "https://www.google.com",
    "https://www.github.com",
    "https://www.microsoft.com",
    "https://www.apple.com",
    "https://www.python.org",
    "https://www.chevrolet.com.ar"
]

for url in test_urls:

    print("\nURL:", url)

    info = get_live_domain_intelligence(url)

    print(info)


URL: https://www.google.com
{'domain_age': 10606, 'dns_resolvable': 1, 'num_ips': 8, 'ttl': 300, 'num_name_servers': 4}

URL: https://www.github.com


/tmp/ipykernel_1303/4118925904.py:30: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  registered_domain = ext.registered_domain


{'domain_age': 6929, 'dns_resolvable': 1, 'num_ips': 1, 'ttl': 9, 'num_name_servers': 8}

URL: https://www.microsoft.com
{'domain_age': 12934, 'dns_resolvable': 1, 'num_ips': 1, 'ttl': 20, 'num_name_servers': 4}

URL: https://www.apple.com
{'domain_age': 14467, 'dns_resolvable': 1, 'num_ips': 1, 'ttl': 20, 'num_name_servers': 4}

URL: https://www.python.org
{'domain_age': 11509, 'dns_resolvable': 1, 'num_ips': 4, 'ttl': 29, 'num_name_servers': 4}

URL: https://www.chevrolet.com.ar
{'domain_age': -1, 'dns_resolvable': 1, 'num_ips': 2, 'ttl': 20, 'num_name_servers': 4}


In [143]:
# ============================================
# LIVE DOMAIN INTELLIGENCE - VALIDATION TEST
# ============================================

test_urls = [
    # 1. Normal well-known domains
    "https://www.google.com",
    "https://www.github.com",
    "https://www.microsoft.com",
    "https://www.apple.com",
    "https://www.python.org",
    "https://www.wikipedia.org",

    # 2. Subdomains
    "https://mail.google.com",
    "https://accounts.google.com",
    "https://docs.google.com",

    # 3. URL with path
    "https://www.github.com/login",

    # 4. URL with query parameters
    "https://www.google.com/search?q=python",

    # 5. Domain without www
    "https://github.com",

    # 6. Safe synthetic non-resolving domain
    "https://this-domain-should-not-exist-928374928374.com"
]


for url in test_urls:

    print("=" * 80)
    print("URL:", url)

    try:
        info = get_live_domain_intelligence(url)

        print("Domain Age       :", info["domain_age"], "days")
        print("DNS Resolvable   :", info["dns_resolvable"])
        print("Number of IPs    :", info["num_ips"])
        print("TTL              :", info["ttl"])
        print("Name Servers     :", info["num_name_servers"])

    except Exception as e:
        print("ERROR:", e)

URL: https://www.google.com
Domain Age       : 10606 days
DNS Resolvable   : 1
Number of IPs    : 8
TTL              : 275
Name Servers     : 4
URL: https://www.github.com


/tmp/ipykernel_1303/4118925904.py:30: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  registered_domain = ext.registered_domain


Domain Age       : 6929 days
DNS Resolvable   : 1
Number of IPs    : 1
TTL              : 60
Name Servers     : 8
URL: https://www.microsoft.com
Domain Age       : 12934 days
DNS Resolvable   : 1
Number of IPs    : 1
TTL              : 20
Name Servers     : 4
URL: https://www.apple.com
Domain Age       : 14467 days
DNS Resolvable   : 1
Number of IPs    : 1
TTL              : 12
Name Servers     : 4
URL: https://www.python.org
Domain Age       : 11509 days
DNS Resolvable   : 1
Number of IPs    : 4
TTL              : 4
Name Servers     : 4
URL: https://www.wikipedia.org
Domain Age       : 9390 days
DNS Resolvable   : 1
Number of IPs    : 1
TTL              : 91
Name Servers     : 3
URL: https://mail.google.com
Domain Age       : 10606 days
DNS Resolvable   : 1
Number of IPs    : 4
TTL              : 300
Name Servers     : 4
URL: https://accounts.google.com
Domain Age       : 10606 days
DNS Resolvable   : 1
Number of IPs    : 1
TTL              : 300
Name Servers     : 4
URL: https://docs

In [144]:
# ============================================
# LIVE DOMAIN INTELLIGENCE
# DNS + RDAP DOMAIN AGE
# ============================================

import requests
import dns.resolver
import tldextract

from datetime import datetime, timezone
from urllib.parse import urlparse


# --------------------------------------------
# 1. Load IANA RDAP bootstrap
# --------------------------------------------

RDAP_BOOTSTRAP_URL = "https://data.iana.org/rdap/dns.json"

rdap_bootstrap = requests.get(
    RDAP_BOOTSTRAP_URL,
    timeout=10
).json()


# --------------------------------------------
# 2. Find RDAP server for a TLD
# --------------------------------------------

def get_rdap_server(domain):

    tld = domain.split(".")[-1].lower()

    services = rdap_bootstrap.get("services", [])

    for service in services:

        tlds = service[0]
        urls = service[1]

        if tld in tlds:
            return urls[0]

    return None


# --------------------------------------------
# 3. Get domain registration date
# --------------------------------------------

def get_domain_registration_date(domain):

    rdap_server = get_rdap_server(domain)

    if not rdap_server:
        return None

    url = (
        rdap_server.rstrip("/")
        + "/domain/"
        + domain
    )

    try:

        response = requests.get(
            url,
            timeout=10,
            headers={
                "Accept": "application/rdap+json"
            }
        )

        if response.status_code != 200:
            return None

        data = response.json()

    except Exception:
        return None


    for event in data.get("events", []):

        if event.get("eventAction") == "registration":

            registration_date = event.get("eventDate")

            if registration_date:
                return registration_date

    return None


# --------------------------------------------
# 4. Calculate domain age
# --------------------------------------------

def get_domain_age(domain):

    registration_date = get_domain_registration_date(domain)

    if not registration_date:
        return -1

    try:

        registration_dt = datetime.fromisoformat(
            registration_date.replace(
                "Z",
                "+00:00"
            )
        )

        now = datetime.now(timezone.utc)

        age_days = (
            now - registration_dt
        ).days

        return max(age_days, 0)

    except Exception:
        return -1


# --------------------------------------------
# 5. Get live DNS information
# --------------------------------------------

def get_live_dns_features(url):

    parsed = urlparse(url)

    hostname = parsed.hostname

    # Invalid URL
    if not hostname:

        return {
            "dns_resolvable": 0,
            "num_ips": 0,
            "ttl": -1,
            "num_name_servers": 0
        }


    # Extract registered/apex domain
    ext = tldextract.extract(url)

    # IMPORTANT:
    # Use the new property instead of
    # deprecated registered_domain
    registered_domain = (
        ext.top_domain_under_public_suffix
    )

    if not registered_domain:
        registered_domain = hostname


    resolver = dns.resolver.Resolver()

    dns_resolvable = 0
    num_ips = 0
    ttl = -1
    num_name_servers = 0


    # ----------------------------------------
    # A record
    # ----------------------------------------

    try:

        answer = resolver.resolve(
            hostname,
            "A"
        )

        ips = set(
            record.address
            for record in answer
        )

        num_ips = len(ips)

        if answer.rrset is not None:
            ttl = answer.rrset.ttl

        if num_ips > 0:
            dns_resolvable = 1

    except Exception:
        pass


    # ----------------------------------------
    # NS records
    # Query registered domain, NOT hostname
    # ----------------------------------------

    try:

        ns_answer = resolver.resolve(
            registered_domain,
            "NS"
        )

        num_name_servers = len(
            list(ns_answer)
        )

    except Exception:
        pass


    return {
        "dns_resolvable": dns_resolvable,
        "num_ips": num_ips,
        "ttl": ttl,
        "num_name_servers": num_name_servers
    }


# --------------------------------------------
# 6. Complete live domain intelligence
# --------------------------------------------

def get_live_domain_intelligence(url):

    parsed = urlparse(url)

    hostname = parsed.hostname

    # Invalid URL
    if not hostname:

        return {
            "domain_age": -1,
            "dns_resolvable": 0,
            "num_ips": 0,
            "ttl": -1,
            "num_name_servers": 0
        }


    # Extract registered domain
    ext = tldextract.extract(url)

    registered_domain = (
        ext.top_domain_under_public_suffix
    )

    if not registered_domain:
        registered_domain = hostname


    # Get live DNS
    dns_info = get_live_dns_features(url)


    # Get live domain age
    domain_age = get_domain_age(
        registered_domain
    )


    return {
        "domain_age": domain_age,
        **dns_info
    }


print("Live DNS + RDAP intelligence functions loaded successfully.")

Live DNS + RDAP intelligence functions loaded successfully.


In [145]:
# Quick verification

test_url = "https://www.github.com"

print("URL:", test_url)

info = get_live_domain_intelligence(test_url)

print(info)

URL: https://www.github.com
{'domain_age': 6929, 'dns_resolvable': 1, 'num_ips': 1, 'ttl': 5, 'num_name_servers': 8}


In [146]:
# ============================================
# COMPLETE URL ANALYSIS
# ML + LIVE DNS/RDAP INTELLIGENCE
# ============================================

def analyze_url(url):

    # ----------------------------------------
    # 1. ML prediction
    # ----------------------------------------

    ml_result = predict_url(url)


    # ----------------------------------------
    # 2. Live DNS + RDAP intelligence
    # ----------------------------------------

    live_result = get_live_domain_intelligence(url)


    # ----------------------------------------
    # 3. Combine everything
    # ----------------------------------------

    result = {
        "url": url,

        # ML results
        "prediction": ml_result["prediction"],
        "phishing_probability": ml_result["phishing_probability"],
        "legitimate_probability": ml_result["legitimate_probability"],

        # Live intelligence
        "domain_age": live_result["domain_age"],
        "dns_resolvable": live_result["dns_resolvable"],
        "num_ips": live_result["num_ips"],
        "ttl": live_result["ttl"],
        "num_name_servers": live_result["num_name_servers"]
    }

    return result

In [147]:
# ============================================
# TEST COMPLETE ANALYSIS
# ============================================

test_url = "https://www.google.com"

result = analyze_url(test_url)

result

{'url': 'https://www.google.com',
 'prediction': 'Legitimate',
 'phishing_probability': np.float32(0.005525957),
 'legitimate_probability': np.float32(0.99447405),
 'domain_age': 10606,
 'dns_resolvable': 1,
 'num_ips': 8,
 'ttl': 300,
 'num_name_servers': 4}

In [148]:
test_url = "http://secure-login-account-verification-12345.com/login"

result = analyze_url(test_url)

result

{'url': 'http://secure-login-account-verification-12345.com/login',
 'prediction': 'Phishing',
 'phishing_probability': np.float32(0.93720645),
 'legitimate_probability': np.float32(0.06279355),
 'domain_age': -1,
 'dns_resolvable': 0,
 'num_ips': 0,
 'ttl': -1,
 'num_name_servers': 0}

In [150]:
# ============================================
# RISK & EXPLANATION LAYER
# ============================================

def generate_explanation(result):

    explanation = {}

    # ----------------------------------------
    # 1. ML prediction
    # ----------------------------------------

    prediction = result["prediction"]

    phishing_probability = float(
        result["phishing_probability"]
    )

    legitimate_probability = float(
        result["legitimate_probability"]
    )

    explanation["prediction"] = prediction

    explanation["phishing_probability"] = round(
        phishing_probability * 100,
        2
    )

    explanation["legitimate_probability"] = round(
        legitimate_probability * 100,
        2
    )


    # ----------------------------------------
    # 2. Risk level
    # ----------------------------------------

    if phishing_probability >= 0.80:

        risk_level = "High"

    elif phishing_probability >= 0.50:

        risk_level = "Medium"

    else:

        risk_level = "Low"


    explanation["risk_level"] = risk_level


    # ----------------------------------------
    # 3. Live DNS/RDAP information
    # ----------------------------------------

    explanation["domain_age_days"] = result[
        "domain_age"
    ]

    explanation["dns_resolvable"] = (
        "Yes"
        if result["dns_resolvable"] == 1
        else "No"
    )

    explanation["num_ips"] = result[
        "num_ips"
    ]

    explanation["ttl"] = result[
        "ttl"
    ]

    explanation["num_name_servers"] = result[
        "num_name_servers"
    ]


    # ----------------------------------------
    # 4. Human-readable evidence
    # ----------------------------------------

    evidence = []


    # DNS evidence
    if result["dns_resolvable"] == 0:

        evidence.append(
            "The hostname could not be resolved through DNS."
        )

    else:

        evidence.append(
            "The hostname successfully resolves through DNS."
        )


    # Domain age evidence
    if result["domain_age"] == -1:

        evidence.append(
            "Domain registration age was unavailable."
        )

    else:

        evidence.append(
            f"The domain has existed for approximately "
            f"{result['domain_age']} days."
        )


    # IP evidence
    if result["num_ips"] == 0:

        evidence.append(
            "No IP addresses were returned."
        )

    else:

        evidence.append(
            f"{result['num_ips']} IP address(es) were returned."
        )


    explanation["evidence"] = evidence


    # ----------------------------------------
    # 5. Final interpretation
    # ----------------------------------------

    if prediction == "Phishing":

        explanation["summary"] = (
            "The machine-learning model classified this "
            "URL as phishing based on the URL patterns "
            "learned from the training data."
        )

    else:

        explanation["summary"] = (
            "The machine-learning model classified this "
            "URL as legitimate based on the URL patterns "
            "learned from the training data."
        )


    # ----------------------------------------
    # 6. Important limitation
    # ----------------------------------------

    explanation["note"] = (
        "Live DNS and domain information is supporting "
        "evidence. It does not by itself prove that a "
        "URL is malicious or legitimate."
    )


    return explanation

In [151]:
explanation = generate_explanation(result)

explanation

{'prediction': 'Phishing',
 'phishing_probability': 93.72,
 'legitimate_probability': 6.28,
 'risk_level': 'High',
 'domain_age_days': -1,
 'dns_resolvable': 'No',
 'num_ips': 0,
 'ttl': -1,
 'num_name_servers': 0,
 'evidence': ['The hostname could not be resolved through DNS.',
  'Domain registration age was unavailable.',
  'No IP addresses were returned.'],
 'summary': 'The machine-learning model classified this URL as phishing based on the URL patterns learned from the training data.',
 'note': 'Live DNS and domain information is supporting evidence. It does not by itself prove that a URL is malicious or legitimate.'}

In [152]:
# ============================================
# SHAP SETUP
# ============================================

!pip install -q shap

import shap

print("SHAP version:", shap.__version__)
print("SHAP loaded successfully.")

SHAP version: 0.52.0
SHAP loaded successfully.


In [153]:
# ============================================
# SHAP EXPLANATION FOR ONE URL
# ============================================

# URL we want to explain
test_url = "http://secure-login-account-verification-12345.com/login"


# --------------------------------------------
# 1. Extract URL features
# --------------------------------------------

features = extract_url_features(test_url)

X_explain = pd.DataFrame(
    [features],
    columns=url_features_final
)

print("Extracted Features:")
display(X_explain)


# --------------------------------------------
# 2. Create SHAP TreeExplainer
# --------------------------------------------

explainer = shap.TreeExplainer(final_xgb_model)

shap_values = explainer.shap_values(X_explain)


# --------------------------------------------
# 3. Display SHAP values
# --------------------------------------------

shap_df = pd.DataFrame({
    "feature": url_features_final,
    "value": X_explain.iloc[0].values,
    "shap_value": shap_values[0]
})

# Sort by absolute contribution
shap_df["absolute_shap"] = shap_df["shap_value"].abs()

shap_df = shap_df.sort_values(
    "absolute_shap",
    ascending=False
)

print("\nFeature Contributions:")
display(shap_df)

Extracted Features:


,entropy,url_length,num_www,ratio_digits_url,num_subdomains,length_hostname,num_dots,num_hyphens,ratio_digits_host,num_question_mark,num_eq
0,4.40845,56,0,0.089286,0,43,1,4,0.116279,0,0



Feature Contributions:


,feature,value,shap_value,absolute_shap
1,url_length,56.000000,5.488614,5.488614
5,length_hostname,43.000000,-2.531798,2.531798
0,entropy,4.408450,1.003412,1.003412
6,num_dots,1.000000,-0.931066,0.931066
4,num_subdomains,0.000000,-0.787012,0.787012
3,ratio_digits_url,0.089286,0.622482,0.622482
7,num_hyphens,4.000000,-0.303168,0.303168
2,num_www,0.000000,0.272731,0.272731
8,ratio_digits_host,0.116279,-0.250589,0.250589
10,num_eq,0.000000,-0.002824,0.002824


In [154]:
# ============================================
# AUTOMATIC SHAP FEATURE EXPLANATION
# ============================================

# Create the SHAP explainer once
shap_explainer = shap.TreeExplainer(final_xgb_model)


def generate_feature_explanation(url):

    # ----------------------------------------
    # 1. Extract URL features
    # ----------------------------------------

    features = extract_url_features(url)

    X_input = pd.DataFrame(
        [features],
        columns=url_features_final
    )


    # ----------------------------------------
    # 2. Get SHAP values
    # ----------------------------------------

    shap_values = shap_explainer.shap_values(X_input)

    shap_values = shap_values[0]


    # ----------------------------------------
    # 3. Create explanation table
    # ----------------------------------------

    explanation_df = pd.DataFrame({
        "feature": url_features_final,
        "value": X_input.iloc[0].values,
        "shap_value": shap_values
    })


    # ----------------------------------------
    # 4. Separate positive and negative
    # ----------------------------------------

    positive_features = explanation_df[
        explanation_df["shap_value"] > 0
    ].copy()

    negative_features = explanation_df[
        explanation_df["shap_value"] < 0
    ].copy()


    # Sort strongest first
    positive_features = positive_features.sort_values(
        "shap_value",
        ascending=False
    )

    negative_features = negative_features.sort_values(
        "shap_value",
        ascending=True
    )


    # ----------------------------------------
    # 5. Take strongest contributors
    # ----------------------------------------

    top_positive = positive_features.head(3)

    top_negative = negative_features.head(3)


    # ----------------------------------------
    # 6. Create readable explanations
    # ----------------------------------------

    positive_explanations = []

    for _, row in top_positive.iterrows():

        feature = row["feature"]
        value = row["value"]

        sentence = (
            f"{feature} = {value:.4f} "
            f"pushed the prediction toward phishing."
        )

        positive_explanations.append(sentence)


    negative_explanations = []

    for _, row in top_negative.iterrows():

        feature = row["feature"]
        value = row["value"]

        sentence = (
            f"{feature} = {value:.4f} "
            f"pushed the prediction toward legitimate."
        )

        negative_explanations.append(sentence)


    # ----------------------------------------
    # 7. Return everything
    # ----------------------------------------

    return {
        "positive_contributors": positive_explanations,
        "negative_contributors": negative_explanations,
        "shap_table": explanation_df.sort_values(
            "shap_value",
            key=lambda x: x.abs(),
            ascending=False
        )
    }

In [155]:
test_url = "http://secure-login-account-verification-12345.com/login"

feature_explanation = generate_feature_explanation(test_url)

print("========================================")
print("FEATURES PUSHING TOWARD PHISHING")
print("========================================")

for item in feature_explanation["positive_contributors"]:
    print("•", item)


print("\n========================================")
print("FEATURES PUSHING TOWARD LEGITIMATE")
print("========================================")

for item in feature_explanation["negative_contributors"]:
    print("•", item)


print("\n========================================")
print("FULL SHAP TABLE")
print("========================================")

display(feature_explanation["shap_table"])

FEATURES PUSHING TOWARD PHISHING
• url_length = 56.0000 pushed the prediction toward phishing.
• entropy = 4.4084 pushed the prediction toward phishing.
• ratio_digits_url = 0.0893 pushed the prediction toward phishing.

FEATURES PUSHING TOWARD LEGITIMATE
• length_hostname = 43.0000 pushed the prediction toward legitimate.
• num_dots = 1.0000 pushed the prediction toward legitimate.
• num_subdomains = 0.0000 pushed the prediction toward legitimate.

FULL SHAP TABLE


,feature,value,shap_value
1,url_length,56.000000,5.488614
5,length_hostname,43.000000,-2.531798
0,entropy,4.408450,1.003412
6,num_dots,1.000000,-0.931066
4,num_subdomains,0.000000,-0.787012
3,ratio_digits_url,0.089286,0.622482
7,num_hyphens,4.000000,-0.303168
2,num_www,0.000000,0.272731
8,ratio_digits_host,0.116279,-0.250589
10,num_eq,0.000000,-0.002824


In [156]:
# ============================================
# HUMAN-READABLE FEATURE DESCRIPTIONS
# ============================================

feature_descriptions = {

    "entropy":
        "URL character entropy",

    "url_length":
        "URL length",

    "num_www":
        "Number of 'www' occurrences",

    "ratio_digits_url":
        "Digit ratio in the URL",

    "num_subdomains":
        "Number of subdomains",

    "length_hostname":
        "Hostname length",

    "num_dots":
        "Number of dots in the URL",

    "num_hyphens":
        "Number of hyphens in the URL",

    "ratio_digits_host":
        "Digit ratio in the hostname",

    "num_question_mark":
        "Number of question marks",

    "num_eq":
        "Number of '=' characters"
}

In [157]:
# ============================================
# HUMAN-READABLE SHAP EXPLANATION
# ============================================

def format_feature_value(feature, value):

    if feature == "url_length":
        return f"{int(value)} characters"

    elif feature == "entropy":
        return f"{value:.2f}"

    elif feature == "ratio_digits_url":
        return f"{value * 100:.2f}%"

    elif feature == "ratio_digits_host":
        return f"{value * 100:.2f}%"

    elif feature in [
        "num_www",
        "num_subdomains",
        "length_hostname",
        "num_dots",
        "num_hyphens",
        "num_question_mark",
        "num_eq"
    ]:
        return f"{int(value)}"

    else:
        return f"{value:.2f}"


def generate_readable_feature_explanation(url):

    # Get SHAP explanation
    result = generate_feature_explanation(url)

    shap_table = result["shap_table"]


    # ----------------------------------------
    # Strongest phishing contributors
    # ----------------------------------------

    positive = shap_table[
        shap_table["shap_value"] > 0
    ].sort_values(
        "shap_value",
        ascending=False
    ).head(3)


    # ----------------------------------------
    # Strongest legitimate contributors
    # ----------------------------------------

    negative = shap_table[
        shap_table["shap_value"] < 0
    ].sort_values(
        "shap_value",
        ascending=True
    ).head(3)


    # ----------------------------------------
    # Convert to readable sentences
    # ----------------------------------------

    phishing_reasons = []

    for _, row in positive.iterrows():

        feature = row["feature"]
        value = row["value"]

        readable_name = feature_descriptions[feature]

        readable_value = format_feature_value(
            feature,
            value
        )

        phishing_reasons.append(
            f"{readable_name}: {readable_value} "
            f"(contributed toward the phishing prediction)"
        )


    legitimate_reasons = []

    for _, row in negative.iterrows():

        feature = row["feature"]
        value = row["value"]

        readable_name = feature_descriptions[feature]

        readable_value = format_feature_value(
            feature,
            value
        )

        legitimate_reasons.append(
            f"{readable_name}: {readable_value} "
            f"(contributed toward the legitimate prediction)"
        )


    return {
        "phishing_reasons": phishing_reasons,
        "legitimate_reasons": legitimate_reasons,
        "shap_table": shap_table
    }

In [158]:
test_url = "http://secure-login-account-verification-12345.com/login"

readable_explanation = generate_readable_feature_explanation(
    test_url
)

print("========================================")
print("WHY THE MODEL LEANED TOWARD PHISHING")
print("========================================")

for reason in readable_explanation["phishing_reasons"]:
    print("•", reason)


print("\n========================================")
print("FEATURES PUSHING THE OTHER WAY")
print("========================================")

for reason in readable_explanation["legitimate_reasons"]:
    print("•", reason)

WHY THE MODEL LEANED TOWARD PHISHING
• URL length: 56 characters (contributed toward the phishing prediction)
• URL character entropy: 4.41 (contributed toward the phishing prediction)
• Digit ratio in the URL: 8.93% (contributed toward the phishing prediction)

FEATURES PUSHING THE OTHER WAY
• Hostname length: 43 (contributed toward the legitimate prediction)
• Number of dots in the URL: 1 (contributed toward the legitimate prediction)
• Number of subdomains: 0 (contributed toward the legitimate prediction)


In [159]:
# ============================================
# FINAL URL ANALYSIS PIPELINE
# ML + SHAP + DNS + RDAP
# ============================================

def analyze_url_complete(url):

    # ----------------------------------------
    # 1. ML prediction + live intelligence
    # ----------------------------------------

    result = analyze_url(url)


    # ----------------------------------------
    # 2. Risk level
    # ----------------------------------------

    phishing_probability = float(
        result["phishing_probability"]
    )

    if phishing_probability >= 0.80:
        risk_level = "High"

    elif phishing_probability >= 0.50:
        risk_level = "Medium"

    else:
        risk_level = "Low"


    # ----------------------------------------
    # 3. SHAP explanation
    # ----------------------------------------

    feature_explanation = (
        generate_readable_feature_explanation(url)
    )


    # ----------------------------------------
    # 4. Combine everything
    # ----------------------------------------

    complete_result = {

        # Basic information
        "url": url,

        # ML
        "prediction": result["prediction"],

        "phishing_probability": round(
            float(result["phishing_probability"]) * 100,
            2
        ),

        "legitimate_probability": round(
            float(result["legitimate_probability"]) * 100,
            2
        ),

        "risk_level": risk_level,


        # SHAP explanations
        "phishing_reasons": (
            feature_explanation[
                "phishing_reasons"
            ]
        ),

        "legitimate_reasons": (
            feature_explanation[
                "legitimate_reasons"
            ]
        ),


        # Live DNS/RDAP
        "domain_age_days": result[
            "domain_age"
        ],

        "dns_resolvable": (
            "Yes"
            if result["dns_resolvable"] == 1
            else "No"
        ),

        "num_ips": result[
            "num_ips"
        ],

        "ttl": result[
            "ttl"
        ],

        "num_name_servers": result[
            "num_name_servers"
        ]
    }


    return complete_result

In [161]:
test_url = "http://secure-login-account-verification-12345.com/login"

final_result = analyze_url_complete(test_url)

final_result

{'url': 'http://secure-login-account-verification-12345.com/login',
 'prediction': 'Phishing',
 'phishing_probability': 93.72,
 'legitimate_probability': 6.28,
 'risk_level': 'High',
 'phishing_reasons': ['URL length: 56 characters (contributed toward the phishing prediction)',
  'URL character entropy: 4.41 (contributed toward the phishing prediction)',
  'Digit ratio in the URL: 8.93% (contributed toward the phishing prediction)'],
 'legitimate_reasons': ['Hostname length: 43 (contributed toward the legitimate prediction)',
  'Number of dots in the URL: 1 (contributed toward the legitimate prediction)',
  'Number of subdomains: 0 (contributed toward the legitimate prediction)'],
 'domain_age_days': -1,
 'dns_resolvable': 'No',
 'num_ips': 0,
 'ttl': -1,
 'num_name_servers': 0}

In [162]:
# ============================================
# COMPLETE PIPELINE - MULTIPLE URL TEST
# ============================================

test_urls = [
    "https://www.google.com",
    "https://github.com",
    "http://paypal-account-security-check-48291.com/verify",
    "http://free-prize-winner-92731.com/claim",
    "https://www.google.com/search?q=python"
]

for url in test_urls:

    print("\n" + "=" * 80)
    print("URL:", url)
    print("=" * 80)

    result = analyze_url_complete(url)

    print("Prediction          :", result["prediction"])
    print("Phishing Probability:", result["phishing_probability"], "%")
    print("Risk Level          :", result["risk_level"])

    print("\nTop phishing contributors:")
    for reason in result["phishing_reasons"]:
        print("  •", reason)

    print("\nTop opposing contributors:")
    for reason in result["legitimate_reasons"]:
        print("  •", reason)

    print("\nLive Intelligence:")
    print("  Domain age        :", result["domain_age_days"], "days")
    print("  DNS resolvable    :", result["dns_resolvable"])
    print("  IPs               :", result["num_ips"])
    print("  TTL               :", result["ttl"])
    print("  Name servers      :", result["num_name_servers"])


URL: https://www.google.com
Prediction          : Legitimate
Phishing Probability: 0.55 %
Risk Level          : Low

Top phishing contributors:
  • Number of subdomains: 1 (contributed toward the phishing prediction)
  • Hostname length: 14 (contributed toward the phishing prediction)
  • Digit ratio in the hostname: 0.00% (contributed toward the phishing prediction)

Top opposing contributors:
  • URL length: 22 characters (contributed toward the legitimate prediction)
  • Number of 'www' occurrences: 1 (contributed toward the legitimate prediction)
  • URL character entropy: 3.66 (contributed toward the legitimate prediction)

Live Intelligence:
  Domain age        : 10606 days
  DNS resolvable    : Yes
  IPs               : 8
  TTL               : 300
  Name servers      : 4

URL: https://github.com
Prediction          : Legitimate
Phishing Probability: 0.48 %
Risk Level          : Low

Top phishing contributors:
  • Hostname length: 10 (contributed toward the phishing prediction)


In [163]:
# ============================================
# COMPARE GOOGLE URL FEATURES
# ============================================

urls_to_compare = [
    "https://www.google.com",
    "https://www.google.com/search",
    "https://www.google.com/search?q=python",
    "https://www.google.com/?q=python",
    "https://www.google.com/search?q=test"
]

for url in urls_to_compare:

    features = extract_url_features(url)

    result = predict_url(url)

    print("\n" + "=" * 80)
    print("URL:", url)
    print("Prediction:", result["prediction"])
    print(
        "Phishing probability:",
        float(result["phishing_probability"])
    )

    print("\nFeatures:")

    for feature in url_features_final:
        print(
            f"{feature:20s}: {features[feature]}"
        )


URL: https://www.google.com
Prediction: Legitimate
Phishing probability: 0.005525956861674786

Features:
entropy             : 3.663532754804254
url_length          : 22
num_www             : 1
ratio_digits_url    : 0.0
num_subdomains      : 1
length_hostname     : 14
num_dots            : 2
num_hyphens         : 0
ratio_digits_host   : 0.0
num_question_mark   : 0
num_eq              : 0

URL: https://www.google.com/search
Prediction: Phishing
Phishing probability: 0.9923157691955566

Features:
entropy             : 3.8833374604210062
url_length          : 29
num_www             : 1
ratio_digits_url    : 0.0
num_subdomains      : 1
length_hostname     : 14
num_dots            : 2
num_hyphens         : 0
ratio_digits_host   : 0.0
num_question_mark   : 0
num_eq              : 0

URL: https://www.google.com/search?q=python
Prediction: Phishing
Phishing probability: 0.9997850060462952

Features:
entropy             : 4.221097250057957
url_length          : 38
num_www             : 1
ratio

In [164]:
# ============================================
# CHECK URL PATH / QUERY DISTRIBUTION
# ============================================

import pandas as pd
from urllib.parse import urlparse


def get_path_length(url):
    try:
        return len(urlparse(url).path)
    except:
        return 0


def get_query_length(url):
    try:
        return len(urlparse(url).query)
    except:
        return 0


def has_path(url):
    try:
        return int(urlparse(url).path not in ["", "/"])
    except:
        return 0


def has_query(url):
    try:
        return int(bool(urlparse(url).query))
    except:
        return 0


analysis_df = df_clean.copy()

analysis_df["path_length"] = analysis_df["url"].apply(
    get_path_length
)

analysis_df["query_length"] = analysis_df["url"].apply(
    get_query_length
)

analysis_df["has_path"] = analysis_df["url"].apply(
    has_path
)

analysis_df["has_query"] = analysis_df["url"].apply(
    has_query
)


print("========================================")
print("PATH / QUERY DISTRIBUTION")
print("========================================")

print("\nPath presence:")
print(
    pd.crosstab(
        analysis_df["label"],
        analysis_df["has_path"],
        normalize="index"
    ) * 100
)

print("\nQuery presence:")
print(
    pd.crosstab(
        analysis_df["label"],
        analysis_df["has_query"],
        normalize="index"
    ) * 100
)


print("\nAverage path length:")
print(
    analysis_df.groupby("label")["path_length"].mean()
)


print("\nAverage query length:")
print(
    analysis_df.groupby("label")["query_length"].mean()
)

PATH / QUERY DISTRIBUTION

Path presence:
has_path             0          1
label                            
legitimate  100.000000   0.000000
phishing     33.560098  66.439902

Query presence:
has_query            0          1
label                            
legitimate  100.000000   0.000000
phishing     79.804177  20.195823

Average path length:
label
legitimate     0.000000
phishing      22.365547
Name: path_length, dtype: float64

Average query length:
label
legitimate     0.000000
phishing      13.368185
Name: query_length, dtype: float64


In [165]:
# ============================================
# CHECK FEATURE IMPORTANCE FOR PATH-RELATED
# LEGITIMATE URL PROBLEM
# ============================================

print("Current URL model feature importance:")
print()

importance_df = pd.DataFrame({
    "feature": url_features_final,
    "importance": final_xgb_model.feature_importances_
})

importance_df = importance_df.sort_values(
    "importance",
    ascending=False
)

display(importance_df)

Current URL model feature importance:



,feature,importance
1,url_length,0.186196
0,entropy,0.175654
2,num_www,0.152400
4,num_subdomains,0.119720
5,length_hostname,0.093645
3,ratio_digits_url,0.093307
10,num_eq,0.074939
6,num_dots,0.034461
8,ratio_digits_host,0.029617
9,num_question_mark,0.021410


In [169]:
import random
from urllib.parse import urlparse

random.seed(42)

# --------------------------------------------------
# 1. Get legitimate URLs from TRAINING data only
# --------------------------------------------------

train_legitimate = train_df[train_df["label"] == 0].copy()

print("Original legitimate training URLs:", len(train_legitimate))


# --------------------------------------------------
# 2. Extract domains from training legitimate URLs
# --------------------------------------------------

train_legitimate["domain"] = train_legitimate["url"].apply(
    lambda x: urlparse(x).netloc
)

legitimate_domains = (
    train_legitimate["domain"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

print("Unique legitimate training domains:", len(legitimate_domains))


# --------------------------------------------------
# 3. Templates for realistic legitimate URLs
# --------------------------------------------------

path_templates = [
    "/search",
    "/search/results",
    "/products",
    "/products/item",
    "/about",
    "/about-us",
    "/contact",
    "/help",
    "/support",
    "/blog",
    "/news",
    "/article",
    "/category",
    "/login",
    "/account",
    "/profile",
    "/settings",
    "/questions/12345",
    "/wiki/example",
]

query_templates = [
    "/search?q=python",
    "/search?q=machine+learning",
    "/search?query=test",
    "/search?page=2",
    "/products?id=12345",
    "/products?category=electronics",
    "/article?id=12345",
    "/questions?page=2",
    "/search?q=laptop&page=2",
]


# --------------------------------------------------
# 4. Generate augmented legitimate URLs
# --------------------------------------------------

augmented_urls = []

# Generate approximately 10,000 additional examples
target_count = 10000

while len(augmented_urls) < target_count:

    domain = random.choice(legitimate_domains)

    # 70% path URLs
    # 30% query URLs
    if random.random() < 0.70:
        path = random.choice(path_templates)
    else:
        path = random.choice(query_templates)

    new_url = "https://" + domain + path

    augmented_urls.append(new_url)


# Remove duplicates
augmented_urls = list(set(augmented_urls))

print("Augmented legitimate URLs:", len(augmented_urls))

# Show examples
augmented_urls[:20]



NameError: name 'train_df' is not defined

In [168]:
[name for name in globals().keys()
 if any(word in name.lower() for word in ["train", "test", "split", "domain"])]

['extract_domain',
 'domain_counts',
 'domain_label_counts',
 'conflicting_domains',
 'domain_label',
 'GroupShuffleSplit',
 'train_idx',
 'test_idx',
 'X_train',
 'X_test',
 'y_train',
 'y_test',
 'groups_train',
 'groups_test',
 'train_domains',
 'test_domains',
 'X_train_no_length',
 'X_test_no_length',
 'X_train_log',
 'X_test_log',
 'test_results',
 'test_url',
 'test_urls',
 'get_num_subdomains',
 'X_train_clean',
 'X_test_clean',
 'unseen_test_urls',
 'domain',
 'domain_exists',
 'domain_age_unknown',
 'X_train_dns_raw',
 'X_test_dns_raw',
 'X_train_dns',
 'X_test_dns',
 'X_train_combined_url',
 'X_test_combined_url',
 'X_train_combined_dns',
 'X_test_combined_dns',
 'X_train_dns_imputed',
 'X_test_dns_imputed',
 'X_train_combined',
 'X_test_combined',
 'test_dns_urls',
 'test_subdomain_urls',
 'get_domain_registration_date',
 'get_domain_age',
 'get_live_domain_intelligence']

In [170]:
import random
from urllib.parse import urlparse

random.seed(42)

# Use the EXISTING domain-based training split
train_data = df_clean.iloc[train_idx].copy()

# Keep only legitimate URLs
train_legitimate = train_data[train_data["label"] == 0].copy()

print("Original legitimate training URLs:", len(train_legitimate))


# Extract domains
train_legitimate["domain"] = train_legitimate["url"].apply(
    lambda x: urlparse(x).netloc
)

legitimate_domains = (
    train_legitimate["domain"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

print("Unique legitimate training domains:", len(legitimate_domains))


# URL structures that legitimate websites commonly use
path_templates = [
    "/search",
    "/search/results",
    "/products",
    "/products/item",
    "/about",
    "/about-us",
    "/contact",
    "/help",
    "/support",
    "/blog",
    "/news",
    "/article",
    "/category",
    "/login",
    "/account",
    "/profile",
    "/settings",
    "/questions/12345",
    "/wiki/example",
]

query_templates = [
    "/search?q=python",
    "/search?q=machine+learning",
    "/search?query=test",
    "/search?page=2",
    "/products?id=12345",
    "/products?category=electronics",
    "/article?id=12345",
    "/questions?page=2",
    "/search?q=laptop&page=2",
]


# Generate additional legitimate examples
augmented_urls = []

target_count = 10000

while len(augmented_urls) < target_count:

    domain = random.choice(legitimate_domains)

    if random.random() < 0.70:
        path = random.choice(path_templates)
    else:
        path = random.choice(query_templates)

    new_url = "https://" + domain + path

    augmented_urls.append(new_url)


# Remove duplicates
augmented_urls = list(set(augmented_urls))

print("Augmented legitimate URLs:", len(augmented_urls))

print("\nSample augmented URLs:")
for url in augmented_urls[:20]:
    print(url)

Original legitimate training URLs: 0
Unique legitimate training domains: 0


IndexError: Cannot choose from an empty sequence

In [172]:
import numpy as np

print("df_clean shape:", df_clean.shape)

print("\nX_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\ny_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\ny_train values:")
print(np.unique(y_train, return_counts=True))

print("\ny_test values:")
print(np.unique(y_test, return_counts=True))

print("\ntrain_idx length:", len(train_idx))
print("test_idx length:", len(test_idx))

print("\nFirst 10 train_idx:")
print(train_idx[:10])

print("\nFirst 10 df_clean indices:")
print(df_clean.index[:10].tolist())

print("\nFirst 10 X_train indices:")
print(X_train.index[:10].tolist())

df_clean shape: (59814, 23)

X_train shape: (47323, 13)
X_test shape: (12491, 13)

y_train shape: (47323,)
y_test shape: (12491,)

y_train values:
(array([0, 1]), array([22673, 24650]))

y_test values:
(array([0, 1]), array([5684, 6807]))

train_idx length: 47323
test_idx length: 12491

First 10 train_idx:
[ 0  2  3  4  5  6  7  8 10 11]

First 10 df_clean indices:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

First 10 X_train indices:
[0, 2, 3, 4, 5, 6, 7, 8, 10, 11]


In [173]:
print(df_clean["label"].value_counts())
print("\nLabel data types:")
print(df_clean["label"].map(type).value_counts())

print("\nSample labels:")
print(df_clean["label"].head(10).tolist())

label
phishing      31457
legitimate    28357
Name: count, dtype: int64

Label data types:
label
<class 'str'>    59814
Name: count, dtype: int64

Sample labels:
['legitimate', 'phishing', 'phishing', 'phishing', 'legitimate', 'legitimate', 'legitimate', 'legitimate', 'phishing', 'phishing']


In [174]:
# Get the exact training rows using the existing train_idx
train_data = df_clean.iloc[train_idx].copy()

# Select legitimate rows using the actual string label
train_legitimate = train_data[
    train_data["label"] == "legitimate"
].copy()

print("Original legitimate training URLs:", len(train_legitimate))

# Extract domains
train_legitimate["domain"] = train_legitimate["url"].apply(
    lambda x: urlparse(x).netloc
)

legitimate_domains = (
    train_legitimate["domain"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

print("Unique legitimate training domains:", len(legitimate_domains))

print("\nSample legitimate URLs:")
print(train_legitimate["url"].head(10).tolist())

Original legitimate training URLs: 22673
Unique legitimate training domains: 22673

Sample legitimate URLs:
['https://manua.ls', 'https://strato.de', 'https://techmeme.com', 'https://mahabhumi.gov.in', 'https://goldenfrog.com', 'https://interactivedns.com', 'https://brocardi.it', 'https://www.northwell.edu', 'https://pitch.com', 'https://www.post.at']


In [175]:
import random

random.seed(42)

# ---------------------------------------------
# Generate realistic legitimate URL structures
# ---------------------------------------------

path_templates = [
    "/search",
    "/search/results",
    "/products",
    "/products/item",
    "/about",
    "/about-us",
    "/contact",
    "/help",
    "/support",
    "/blog",
    "/news",
    "/article",
    "/category",
    "/login",
    "/account",
    "/profile",
    "/settings",
    "/questions/12345",
    "/wiki/example"
]

query_templates = [
    "/search?q=python",
    "/search?q=machine+learning",
    "/search?query=test",
    "/search?page=2",
    "/products?id=12345",
    "/products?category=electronics",
    "/article?id=12345",
    "/questions?page=2",
    "/search?q=laptop&page=2"
]

augmented_urls = set()

# Use 5,000 additional examples
target_count = 5000

while len(augmented_urls) < target_count:

    domain = random.choice(legitimate_domains)

    # 70% path URLs
    # 30% query URLs
    if random.random() < 0.70:
        suffix = random.choice(path_templates)
    else:
        suffix = random.choice(query_templates)

    new_url = "https://" + domain + suffix

    augmented_urls.add(new_url)

augmented_urls = list(augmented_urls)

print("Number of augmented legitimate URLs:", len(augmented_urls))

print("\nSample augmented URLs:")
for url in augmented_urls[:20]:
    print(url)

Number of augmented legitimate URLs: 5000

Sample augmented URLs:
https://holley.com/questions?page=2
https://anime-planet.com/search/results
https://hotstar.com/about
https://savethechildren.org.uk/products
https://aau.at/search?q=python
https://sogetel.net/about-us
https://xn--69-lqix0ea5a7kua2b8lc3gd.com/search?query=test
https://www.renttherunway.com/products/item
https://xnxx.place/help
https://ecma-international.org/search/results
https://excellmedia.net/wiki/example
https://boreus.de/blog
https://verdi.de/contact
https://plaud.ai/products/item
https://ut.ee/products/item
https://www.vectorstock.com/help
https://benesse.ne.jp/search?query=test
https://top.pl/help
https://clarodigital.net/products
https://tedra.es/questions?page=2


In [176]:
import pandas as pd

aug_df = pd.DataFrame({
    "url": augmented_urls,
    "label": "legitimate"
})

# ---------------------------------------------
# Check path/query distribution
# ---------------------------------------------

aug_df["has_path"] = aug_df["url"].apply(
    lambda x: "/" in x.split("://", 1)[1]
)

aug_df["has_query"] = aug_df["url"].apply(
    lambda x: "?" in x
)

aug_df["url_length"] = aug_df["url"].str.len()

aug_df["path_length"] = aug_df["url"].apply(
    lambda x: len(urlparse(x).path)
)

aug_df["query_length"] = aug_df["url"].apply(
    lambda x: len(urlparse(x).query)
)

print("Augmented dataset shape:", aug_df.shape)

print("\nPath distribution:")
print(aug_df["has_path"].value_counts())

print("\nQuery distribution:")
print(aug_df["has_query"].value_counts())

print("\nAverage URL length:")
print(round(aug_df["url_length"].mean(), 2))

print("\nAverage path length:")
print(round(aug_df["path_length"].mean(), 2))

print("\nAverage query length:")
print(round(aug_df["query_length"].mean(), 2))

print("\nExamples with queries:")
print(
    aug_df[aug_df["has_query"]]["url"].head(10).to_list()
)

Augmented dataset shape: (5000, 7)

Path distribution:
has_path
True    5000
Name: count, dtype: int64

Query distribution:
has_query
False    3567
True     1433
Name: count, dtype: int64

Average URL length:
32.92

Average path length:
8.59

Average query length:
3.16

Examples with queries:
['https://holley.com/questions?page=2', 'https://aau.at/search?q=python', 'https://xn--69-lqix0ea5a7kua2b8lc3gd.com/search?query=test', 'https://benesse.ne.jp/search?query=test', 'https://tedra.es/questions?page=2', 'https://forabank.ru/search?page=2', 'https://formstack.com/search?q=python', 'https://rambler-co.ru/article?id=12345', 'https://guitarplayer.com/questions?page=2', 'https://narvar.com/search?page=2']


In [177]:
print(repr(augmented_urls[0]))

print("\nFirst 5 raw URLs:")
for url in augmented_urls[:5]:
    print(repr(url))

'https://holley.com/questions?page=2'

First 5 raw URLs:
'https://holley.com/questions?page=2'
'https://anime-planet.com/search/results'
'https://hotstar.com/about'
'https://savethechildren.org.uk/products'
'https://aau.at/search?q=python'


In [178]:
import re

def clean_augmented_url(url):
    # If URL is stored as Markdown:
    # [https://example.com/path](https://example.com/path)
    match = re.match(r"^\[.*?\]\((https?://.*?)\)$", url)

    if match:
        return match.group(1)

    return url


# Clean all augmented URLs
augmented_urls = [
    clean_augmented_url(url)
    for url in augmented_urls
]

# Remove duplicates after cleaning
augmented_urls = list(set(augmented_urls))

print("Number of cleaned URLs:", len(augmented_urls))

print("\nFirst 5 cleaned URLs:")
for url in augmented_urls[:5]:
    print(repr(url))

Number of cleaned URLs: 5000

First 5 cleaned URLs:
'https://holley.com/questions?page=2'
'https://anime-planet.com/search/results'
'https://hotstar.com/about'
'https://savethechildren.org.uk/products'
'https://aau.at/search?q=python'


In [179]:
def clean_augmented_url(url):
    if "](" in url and url.endswith(")"):
        url = url.split("](", 1)[1]
        url = url[:-1]  # remove final )

    return url


augmented_urls = [
    clean_augmented_url(url)
    for url in augmented_urls
]

# Remove duplicates
augmented_urls = list(set(augmented_urls))

print("Number of cleaned URLs:", len(augmented_urls))

print("\nFirst 5 cleaned URLs:")
for url in augmented_urls[:5]:
    print(repr(url))

Number of cleaned URLs: 5000

First 5 cleaned URLs:
'https://holley.com/questions?page=2'
'https://anime-planet.com/search/results'
'https://hotstar.com/about'
'https://savethechildren.org.uk/products'
'https://aau.at/search?q=python'


In [180]:
url = augmented_urls[0]

print("RAW:")
print(url)

print("\nREPR:")
print(repr(url))

print("\nTYPE:")
print(type(url))

print("\nContains '](':", "](" in url)
print("Ends with ')':", url.endswith(")"))

print("\nCharacters around '](':")
pos = url.find("](")
print(pos)

if pos != -1:
    print(repr(url[pos:pos+20]))

RAW:
https://holley.com/questions?page=2

REPR:
'https://holley.com/questions?page=2'

TYPE:
<class 'str'>

Contains '](': False
Ends with ')': False

Characters around '](':
-1


In [181]:
url = augmented_urls[0]

print("Character-by-character:")
for i, char in enumerate(url):
    print(i, repr(char), ord(char))

Character-by-character:
0 'h' 104
1 't' 116
2 't' 116
3 'p' 112
4 's' 115
5 ':' 58
6 '/' 47
7 '/' 47
8 'h' 104
9 'o' 111
10 'l' 108
11 'l' 108
12 'e' 101
13 'y' 121
14 '.' 46
15 'c' 99
16 'o' 111
17 'm' 109
18 '/' 47
19 'q' 113
20 'u' 117
21 'e' 101
22 's' 115
23 't' 116
24 'i' 105
25 'o' 111
26 'n' 110
27 's' 115
28 '?' 63
29 'p' 112
30 'a' 97
31 'g' 103
32 'e' 101
33 '=' 61
34 '2' 50


In [182]:
# Extract the same 11 features used by the final XGBoost model

augmented_features = pd.DataFrame(
    [extract_url_features(url) for url in augmented_urls]
)

augmented_features = augmented_features[url_features_final]

print("Augmented feature shape:", augmented_features.shape)

print("\nAverage feature values:")
display(augmented_features.mean().round(4))

print("\nFeature ranges:")
display(
    pd.DataFrame({
        "min": augmented_features.min(),
        "max": augmented_features.max()
    }).round(4)
)

Augmented feature shape: (5000, 11)

Average feature values:


,0
entropy,4.0056
url_length,32.9188
num_www,0.1590
ratio_digits_url,0.0200
num_subdomains,0.1586
length_hostname,12.8798
num_dots,1.2336
num_hyphens,0.0956
ratio_digits_host,0.0096
num_question_mark,0.2866



Feature ranges:


,min,max
entropy,3.1158,4.9094
url_length,17.0000,68.0000
num_www,0.0000,1.0000
ratio_digits_url,0.0000,0.2500
num_subdomains,0.0000,1.0000
length_hostname,4.0000,32.0000
num_dots,1.0000,4.0000
num_hyphens,0.0000,3.0000
ratio_digits_host,0.0000,0.6250
num_question_mark,0.0000,1.0000


In [183]:
# Get original training data
train_data = df_clean.iloc[train_idx].copy()

# Extract URL features for the original training URLs
original_train_features = pd.DataFrame(
    [extract_url_features(url) for url in train_data["url"]]
)

original_train_features["label"] = train_data["label"].values

# Separate legitimate and phishing
original_legitimate_features = original_train_features[
    original_train_features["label"] == "legitimate"
].drop(columns="label")

original_phishing_features = original_train_features[
    original_train_features["label"] == "phishing"
].drop(columns="label")


# ---------------------------------------------
# Compare means
# ---------------------------------------------

comparison = pd.DataFrame({
    "Original Legitimate": original_legitimate_features.mean(),
    "Augmented Legitimate": augmented_features.mean(),
    "Original Phishing": original_phishing_features.mean()
})

display(comparison.round(4))

,Original Legitimate,Augmented Legitimate,Original Phishing
entropy,3.6999,4.0056,4.4399
url_length,20.8210,32.9188,68.4806
num_www,0.1598,0.1590,0.0293
ratio_digits_url,0.0056,0.0200,0.0827
num_subdomains,0.1594,0.1586,0.7701
length_hostname,12.8210,12.8798,20.3169
num_dots,1.2368,1.2336,2.0331
num_hyphens,0.0564,0.0956,0.8524
ratio_digits_host,0.0098,0.0096,0.0546
num_question_mark,0.0000,0.2866,0.2283


In [184]:
# Use the existing training split
train_data = df_clean.iloc[train_idx].copy()

# Check which labels each domain has
domain_label_check = (
    train_data
    .groupby("domain")["label"]
    .unique()
)

# Keep only domains whose training label is exclusively legitimate
pure_legitimate_domains = domain_label_check[
    domain_label_check.apply(
        lambda labels: len(labels) == 1 and labels[0] == "legitimate"
    )
].index.tolist()

print("Total training domains:",
      train_data["domain"].nunique())

print("Pure legitimate domains:",
      len(pure_legitimate_domains))

print("Ambiguous/mixed domains:",
      train_data["domain"].nunique() - len(pure_legitimate_domains))

print("\nSample pure legitimate domains:")
print(pure_legitimate_domains[:20])

Total training domains: 32723
Pure legitimate domains: 20330
Ambiguous/mixed domains: 12393

Sample pure legitimate domains:
['007names.net', '009.am', '00web.net', '0123tt.ru', '01net.com', '021.rs', '0aa0cf0637d66c0d.com', '1-grid.co.uk', '1-hostingservice.com', '1-ofd.ru', '1000logos.net', '10010.com', '10086.cn', '100lw.com', '100sp.ru', '101domain.com', '101hotels.com', '103.by', '10fastfingers.com', '10jqka.com.cn']


In [185]:
import random

random.seed(42)

# ---------------------------------------------
# Legitimate URL structures
# ---------------------------------------------

path_templates = [
    "/search",
    "/search/results",
    "/products",
    "/products/item",
    "/about",
    "/about-us",
    "/contact",
    "/help",
    "/support",
    "/blog",
    "/news",
    "/article",
    "/category",
    "/login",
    "/account",
    "/profile",
    "/settings",
    "/questions/12345",
    "/wiki/example"
]

query_templates = [
    "/search?q=python",
    "/search?q=machine+learning",
    "/search?query=test",
    "/search?page=2",
    "/products?id=12345",
    "/products?category=electronics",
    "/article?id=12345",
    "/questions?page=2",
    "/search?q=laptop&page=2"
]


# ---------------------------------------------
# Generate augmentation
# ---------------------------------------------

augmented_urls = set()

target_count = 5000

while len(augmented_urls) < target_count:

    domain = random.choice(pure_legitimate_domains)

    # 70% path-only
    # 30% query-containing
    if random.random() < 0.70:
        suffix = random.choice(path_templates)
    else:
        suffix = random.choice(query_templates)

    augmented_urls.add("https://" + domain + suffix)


augmented_urls = list(augmented_urls)

print("Augmented legitimate URLs:", len(augmented_urls))

print("\nFirst 10 URLs:")
for url in augmented_urls[:10]:
    print(url)

Augmented legitimate URLs: 5000

First 10 URLs:
https://g2288.com/search?q=python
https://buddy4study.com/category
https://scandinavianhosting.se/products?id=12345
https://ufrgs.br/products?id=12345
https://xhamsterflirt.com/questions?page=2
https://neworleans.com/products?id=12345
https://onlinetestpad.com/about
https://zalando-prive.it/login
https://portalinmobiliario.com/questions?page=2
https://getvim.com/about


In [186]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# --------------------------------------------------
# 1. Prepare original training data
# --------------------------------------------------

X_train_final = X_train[url_features_final].copy()

# --------------------------------------------------
# 2. Extract features from augmented URLs
# --------------------------------------------------

augmented_features = pd.DataFrame(
    [extract_url_features(url) for url in augmented_urls]
)

augmented_features = augmented_features[url_features_final]

# All augmented URLs are legitimate
augmented_labels = np.zeros(len(augmented_features), dtype=int)


# --------------------------------------------------
# 3. Combine original + augmented training data
# --------------------------------------------------

X_train_augmented = pd.concat(
    [
        X_train_final.reset_index(drop=True),
        augmented_features.reset_index(drop=True)
    ],
    ignore_index=True
)

y_train_augmented = np.concatenate(
    [
        y_train,
        augmented_labels
    ]
)

print("Original training samples:", len(X_train_final))
print("Augmented samples:", len(augmented_features))
print("Final training samples:", len(X_train_augmented))


# --------------------------------------------------
# 4. Train CANDIDATE XGBoost model
# --------------------------------------------------

augmented_xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

augmented_xgb_model.fit(
    X_train_augmented,
    y_train_augmented
)


# --------------------------------------------------
# 5. Evaluate on the ORIGINAL untouched test set
# --------------------------------------------------

X_test_final = X_test[url_features_final].copy()

current_pred = final_xgb_model.predict(X_test_final)
current_prob = final_xgb_model.predict_proba(X_test_final)[:, 1]

augmented_pred = augmented_xgb_model.predict(X_test_final)
augmented_prob = augmented_xgb_model.predict_proba(X_test_final)[:, 1]


# --------------------------------------------------
# 6. Compare both models
# --------------------------------------------------

comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC"
    ],

    "Current Model": [
        accuracy_score(y_test, current_pred),
        precision_score(y_test, current_pred),
        recall_score(y_test, current_pred),
        f1_score(y_test, current_pred),
        roc_auc_score(y_test, current_prob)
    ],

    "Augmented Model": [
        accuracy_score(y_test, augmented_pred),
        precision_score(y_test, augmented_pred),
        recall_score(y_test, augmented_pred),
        f1_score(y_test, augmented_pred),
        roc_auc_score(y_test, augmented_prob)
    ]
})

display(comparison.round(6))

Original training samples: 47323
Augmented samples: 5000
Final training samples: 52323


,Metric,Current Model,Augmented Model
0,Accuracy,0.993115,0.973661
1,Precision,0.998813,0.998001
2,Recall,0.988541,0.953577
3,F1,0.993650,0.975284
4,ROC-AUC,0.997955,0.997234


In [187]:
validation_urls = [
    # Legitimate URLs with paths / queries
    "https://www.google.com",
    "https://www.google.com/search",
    "https://www.google.com/search?q=python",
    "https://github.com",
    "https://github.com/explore",
    "https://en.wikipedia.org/wiki/Machine_learning",
    "https://stackoverflow.com/questions/12345",

    # Suspicious-looking URLs
    "http://paypal-account-security-check-48291.com/verify",
    "http://free-prize-winner-92731.com/claim",
    "http://secure-login-account-verification-12345.com/login"
]

validation_results = []

for url in validation_urls:

    features = extract_url_features(url)

    X_input = pd.DataFrame(
        [features],
        columns=url_features_final
    )

    current_probability = final_xgb_model.predict_proba(
        X_input
    )[0][1]

    augmented_probability = augmented_xgb_model.predict_proba(
        X_input
    )[0][1]

    validation_results.append({
        "URL": url,
        "Current Model %": round(current_probability * 100, 2),
        "Augmented Model %": round(augmented_probability * 100, 2)
    })

validation_results_df = pd.DataFrame(validation_results)

display(validation_results_df)

,URL,Current Model %,Augmented Model %
0,https://www.google.com,0.550000,0.620000
1,https://www.google.com/search,99.230003,69.449997
2,https://www.google.com/search?q=python,99.980003,98.379997
3,https://github.com,0.480000,0.510000
4,https://github.com/explore,99.790001,25.410000
5,https://en.wikipedia.org/wiki/Machine_learning,100.000000,99.989998
6,https://stackoverflow.com/questions/12345,100.000000,9.640000
7,http://paypal-account-security-check-48291.com...,88.290001,75.199997
8,http://free-prize-winner-92731.com/claim,82.680000,9.320000
9,http://secure-login-account-verification-12345...,93.720001,87.470001


In [188]:
# Create a much smaller augmentation set
# from our already generated 5,000 examples.

random.seed(42)

small_augmented_urls = random.sample(
    augmented_urls,
    500
)

print("Small augmentation size:", len(small_augmented_urls))

print("\nSample URLs:")
for url in small_augmented_urls[:10]:
    print(url)

Small augmentation size: 500

Sample URLs:
https://arvig.net/products?category=electronics
https://rechargeapps.com/support
https://sankuai.com/support
https://relativity.one/help
https://voxlink.ru/about
https://ayads.co/settings
https://im.cz/account
https://hdb.gov.sg/search?query=test
https://bnovo.ru/settings
https://orderly.network/profile


In [189]:
# --------------------------------------------------
# Prepare the 500 augmented examples
# --------------------------------------------------

small_augmented_features = pd.DataFrame(
    [extract_url_features(url) for url in small_augmented_urls]
)

small_augmented_features = small_augmented_features[
    url_features_final
]

# All are intended as legitimate examples
small_augmented_labels = np.zeros(
    len(small_augmented_features),
    dtype=int
)


# --------------------------------------------------
# Combine with ORIGINAL training data
# --------------------------------------------------

X_train_small_aug = pd.concat(
    [
        X_train[url_features_final].reset_index(drop=True),
        small_augmented_features.reset_index(drop=True)
    ],
    ignore_index=True
)

y_train_small_aug = np.concatenate(
    [
        y_train,
        small_augmented_labels
    ]
)

print("Original training samples:", len(X_train))
print("Small augmentation:", len(small_augmented_features))
print("Final training samples:", len(X_train_small_aug))


# --------------------------------------------------
# Train candidate model
# --------------------------------------------------

small_augmented_xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

small_augmented_xgb_model.fit(
    X_train_small_aug,
    y_train_small_aug
)

print("\n500-example augmented model trained successfully.")

Original training samples: 47323
Small augmentation: 500
Final training samples: 47823

500-example augmented model trained successfully.


In [190]:
# --------------------------------------------------
# Evaluate current model vs 500-example model
# on the SAME untouched test set
# --------------------------------------------------

X_test_final = X_test[url_features_final].copy()

# Current baseline
current_pred = final_xgb_model.predict(X_test_final)
current_prob = final_xgb_model.predict_proba(X_test_final)[:, 1]

# 500-example augmented model
small_aug_pred = small_augmented_xgb_model.predict(X_test_final)
small_aug_prob = small_augmented_xgb_model.predict_proba(X_test_final)[:, 1]


# --------------------------------------------------
# Comparison
# --------------------------------------------------

comparison_500 = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC"
    ],

    "Current Model": [
        accuracy_score(y_test, current_pred),
        precision_score(y_test, current_pred),
        recall_score(y_test, current_pred),
        f1_score(y_test, current_pred),
        roc_auc_score(y_test, current_prob)
    ],

    "500-Augmented Model": [
        accuracy_score(y_test, small_aug_pred),
        precision_score(y_test, small_aug_pred),
        recall_score(y_test, small_aug_pred),
        f1_score(y_test, small_aug_pred),
        roc_auc_score(y_test, small_aug_prob)
    ]
})

display(comparison_500.round(6))

,Metric,Current Model,500-Augmented Model
0,Accuracy,0.993115,0.989833
1,Precision,0.998813,0.998507
2,Recall,0.988541,0.982812
3,F1,0.993650,0.990597
4,ROC-AUC,0.997955,0.997908


In [191]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

threshold_results = []

for threshold in np.arange(0.10, 0.91, 0.05):

    predictions = (current_prob >= threshold).astype(int)

    threshold_results.append({
        "Threshold": round(threshold, 2),
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1": f1_score(y_test, predictions)
    })

threshold_df = pd.DataFrame(threshold_results)

display(threshold_df.round(4))

,Threshold,Accuracy,Precision,Recall,F1
0,0.10,0.9857,0.9813,0.9928,0.9870
1,0.15,0.9906,0.9908,0.9919,0.9913
2,0.20,0.9924,0.9945,0.9915,0.9930
3,0.25,0.9933,0.9962,0.9915,0.9938
4,0.30,0.9933,0.9970,0.9906,0.9938
5,0.35,0.9934,0.9979,0.9900,0.9940
6,0.40,0.9937,0.9987,0.9897,0.9942
7,0.45,0.9934,0.9987,0.9891,0.9939
8,0.50,0.9931,0.9988,0.9885,0.9937
9,0.55,0.9930,0.9993,0.9878,0.9935


In [192]:
comparison_urls = [
    "https://www.google.com",
    "https://www.google.com/search",
    "https://www.google.com/search?q=python",
    "https://www.github.com",
    "https://www.github.com/explore"
]

feature_comparison = pd.DataFrame(
    [
        extract_url_features(url)
        for url in comparison_urls
    ],
    index=comparison_urls
)

display(feature_comparison[url_features_final].round(4))

,entropy,url_length,num_www,ratio_digits_url,num_subdomains,length_hostname,num_dots,num_hyphens,ratio_digits_host,num_question_mark,num_eq
https://www.google.com,3.6635,22,1,0.0,1,14,2,0,0.0,0,0
https://www.google.com/search,3.8833,29,1,0.0,1,14,2,0,0.0,0,0
https://www.google.com/search?q=python,4.2211,38,1,0.0,1,14,2,0,0.0,1,1
https://www.github.com,3.7544,22,1,0.0,1,14,2,0,0.0,0,0
https://www.github.com/explore,4.0981,30,1,0.0,1,14,2,0,0.0,0,0


In [193]:
comparison_urls = [
    "https://www.google.com",
    "https://www.google.com/search",
    "https://www.google.com/search?q=python",
    "https://www.github.com",
    "https://www.github.com/explore"
]

# Extract features
comparison_features = pd.DataFrame(
    [
        extract_url_features(url)
        for url in comparison_urls
    ]
)[url_features_final]

# Calculate SHAP values
comparison_shap = shap_explainer.shap_values(comparison_features)

# Display SHAP contribution for each URL
for i, url in enumerate(comparison_urls):

    shap_df = pd.DataFrame({
        "Feature": url_features_final,
        "Feature Value": comparison_features.iloc[i].values,
        "SHAP Contribution": comparison_shap[i]
    })

    shap_df["Absolute SHAP"] = shap_df["SHAP Contribution"].abs()

    shap_df = shap_df.sort_values(
        "Absolute SHAP",
        ascending=False
    )

    print("\n" + "=" * 80)
    print(url)
    print("=" * 80)

    display(
        shap_df[
            ["Feature", "Feature Value", "SHAP Contribution"]
        ].head(6).round(4)
    )


https://www.google.com


,Feature,Feature Value,SHAP Contribution
1,url_length,22.0000,-3.6139
2,num_www,1.0000,-2.3671
4,num_subdomains,1.0000,1.0842
0,entropy,3.6635,-0.6997
5,length_hostname,14.0000,0.6853
3,ratio_digits_url,0.0000,-0.2808



https://www.google.com/search


,Feature,Feature Value,SHAP Contribution
5,length_hostname,14.0000,4.4622
1,url_length,29.0000,2.9921
2,num_www,1.0000,-2.6592
4,num_subdomains,1.0000,0.8632
0,entropy,3.8833,-0.4588
3,ratio_digits_url,0.0000,-0.2679



https://www.google.com/search?q=python


,Feature,Feature Value,SHAP Contribution
1,url_length,38.0000,5.4972
5,length_hostname,14.0000,3.5089
2,num_www,1.0000,-2.0065
4,num_subdomains,1.0000,0.9419
0,entropy,4.2211,0.7883
3,ratio_digits_url,0.0000,-0.2809



https://www.github.com


,Feature,Feature Value,SHAP Contribution
1,url_length,22.0000,-3.5975
2,num_www,1.0000,-2.5035
4,num_subdomains,1.0000,0.9993
5,length_hostname,14.0000,0.6805
0,entropy,3.7544,-0.6234
3,ratio_digits_url,0.0000,-0.2856



https://www.github.com/explore


,Feature,Feature Value,SHAP Contribution
5,length_hostname,14.0000,4.0358
2,num_www,1.0000,-2.6787
1,url_length,30.0000,2.6378
4,num_subdomains,1.0000,0.9994
0,entropy,4.0981,0.4781
3,ratio_digits_url,0.0000,-0.4044


In [194]:
# Calculate SHAP interaction values
interaction_values = shap_explainer.shap_interaction_values(
    comparison_features
)

important_features = [
    "url_length",
    "length_hostname",
    "entropy",
    "num_www"
]

for i, url in enumerate(comparison_urls):

    print("\n" + "=" * 80)
    print(url)
    print("=" * 80)

    interaction_matrix = pd.DataFrame(
        interaction_values[i],
        index=url_features_final,
        columns=url_features_final
    )

    # Show interactions among the important features
    display(
        interaction_matrix.loc[
            important_features,
            important_features
        ].round(4)
    )


https://www.google.com


,url_length,length_hostname,entropy,num_www
url_length,-1.5417,-2.2955,0.3151,-0.1793
length_hostname,-2.2955,2.8346,0.0255,0.1299
entropy,0.3151,0.0255,-1.3454,0.1992
num_www,-0.1793,0.1299,0.1992,-1.3135



https://www.google.com/search


,url_length,length_hostname,entropy,num_www
url_length,1.4450,1.4166,0.4778,-0.3403
length_hostname,1.4166,2.6760,0.1175,0.3673
entropy,0.4778,0.1175,-1.0881,-0.0026
num_www,-0.3403,0.3673,-0.0026,-1.3739



https://www.google.com/search?q=python


,url_length,length_hostname,entropy,num_www
url_length,5.3560,0.4914,-0.2531,0.0778
length_hostname,0.4914,2.6697,-0.0727,0.4535
entropy,-0.2531,-0.0727,0.9350,0.0904
num_www,0.0778,0.4535,0.0904,-1.6375



https://www.github.com


,url_length,length_hostname,entropy,num_www
url_length,-1.5241,-2.3047,0.3406,-0.1900
length_hostname,-2.3047,2.8450,0.0242,0.1234
entropy,0.3406,0.0242,-1.0707,0.0864
num_www,-0.1900,0.1234,0.0864,-1.2625



https://www.github.com/explore


,url_length,length_hostname,entropy,num_www
url_length,2.1223,1.1262,-0.3184,-0.2402
length_hostname,1.1262,2.8812,-0.1434,0.3189
entropy,-0.3184,-0.1434,1.1296,-0.1099
num_www,-0.2402,0.3189,-0.1099,-1.2931


In [195]:
# --------------------------------------------------
# Remove url_length for an ablation experiment
# --------------------------------------------------

url_features_no_length = [
    feature for feature in url_features_final
    if feature != "url_length"
]

print("Features being used:")
print(url_features_no_length)

print("\nNumber of features:", len(url_features_no_length))


# --------------------------------------------------
# Prepare training and test data
# --------------------------------------------------

X_train_no_length_xgb = X_train[url_features_no_length].copy()
X_test_no_length_xgb = X_test[url_features_no_length].copy()


# --------------------------------------------------
# Train experimental XGBoost
# --------------------------------------------------

xgb_no_length = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_no_length.fit(
    X_train_no_length_xgb,
    y_train
)


# --------------------------------------------------
# Evaluate
# --------------------------------------------------

no_length_pred = xgb_no_length.predict(
    X_test_no_length_xgb
)

no_length_prob = xgb_no_length.predict_proba(
    X_test_no_length_xgb
)[:, 1]


no_length_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC"
    ],

    "Current Model": [
        accuracy_score(y_test, current_pred),
        precision_score(y_test, current_pred),
        recall_score(y_test, current_pred),
        f1_score(y_test, current_pred),
        roc_auc_score(y_test, current_prob)
    ],

    "Without url_length": [
        accuracy_score(y_test, no_length_pred),
        precision_score(y_test, no_length_pred),
        recall_score(y_test, no_length_pred),
        f1_score(y_test, no_length_pred),
        roc_auc_score(y_test, no_length_prob)
    ]
})

display(no_length_results.round(6))

Features being used:
['entropy', 'num_www', 'ratio_digits_url', 'num_subdomains', 'length_hostname', 'num_dots', 'num_hyphens', 'ratio_digits_host', 'num_question_mark', 'num_eq']

Number of features: 10


,Metric,Current Model,Without url_length
0,Accuracy,0.993115,0.976703
1,Precision,0.998813,0.993936
2,Recall,0.988541,0.963126
3,F1,0.993650,0.978288
4,ROC-AUC,0.997955,0.993981


In [196]:
# --------------------------------------------------
# XGBoost experiment: shallower trees
# --------------------------------------------------

xgb_depth3 = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_depth3.fit(
    X_train[url_features_final],
    y_train
)


# --------------------------------------------------
# Evaluate on the same test set
# --------------------------------------------------

depth3_pred = xgb_depth3.predict(
    X_test[url_features_final]
)

depth3_prob = xgb_depth3.predict_proba(
    X_test[url_features_final]
)[:, 1]


depth3_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC"
    ],

    "Current depth=6": [
        accuracy_score(y_test, current_pred),
        precision_score(y_test, current_pred),
        recall_score(y_test, current_pred),
        f1_score(y_test, current_pred),
        roc_auc_score(y_test, current_prob)
    ],

    "depth=3": [
        accuracy_score(y_test, depth3_pred),
        precision_score(y_test, depth3_pred),
        recall_score(y_test, depth3_pred),
        f1_score(y_test, depth3_pred),
        roc_auc_score(y_test, depth3_prob)
    ]
})

display(depth3_results.round(6))

,Metric,Current depth=6,depth=3
0,Accuracy,0.993115,0.987831
1,Precision,0.998813,0.998054
2,Recall,0.988541,0.979580
3,F1,0.993650,0.988731
4,ROC-AUC,0.997955,0.997358


In [197]:
xgb_depth5 = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_depth5.fit(
    X_train[url_features_final],
    y_train
)

depth5_pred = xgb_depth5.predict(
    X_test[url_features_final]
)

depth5_prob = xgb_depth5.predict_proba(
    X_test[url_features_final]
)[:, 1]

depth5_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC"
    ],

    "Current depth=6": [
        accuracy_score(y_test, current_pred),
        precision_score(y_test, current_pred),
        recall_score(y_test, current_pred),
        f1_score(y_test, current_pred),
        roc_auc_score(y_test, current_prob)
    ],

    "depth=5": [
        accuracy_score(y_test, depth5_pred),
        precision_score(y_test, depth5_pred),
        recall_score(y_test, depth5_pred),
        f1_score(y_test, depth5_pred),
        roc_auc_score(y_test, depth5_prob)
    ]
})

display(depth5_results.round(6))

,Metric,Current depth=6,depth=5
0,Accuracy,0.993115,0.992234
1,Precision,0.998813,0.999107
2,Recall,0.988541,0.986631
3,F1,0.993650,0.992830
4,ROC-AUC,0.997955,0.997969


In [198]:
validation_results_depth = []

for url in validation_urls:

    features = extract_url_features(url)

    X_input = pd.DataFrame(
        [features],
        columns=url_features_final
    )

    depth6_probability = final_xgb_model.predict_proba(
        X_input
    )[0][1]

    depth5_probability = xgb_depth5.predict_proba(
        X_input
    )[0][1]

    validation_results_depth.append({
        "URL": url,
        "Depth 6 %": round(depth6_probability * 100, 2),
        "Depth 5 %": round(depth5_probability * 100, 2)
    })

depth_comparison_df = pd.DataFrame(
    validation_results_depth
)

display(depth_comparison_df)

,URL,Depth 6 %,Depth 5 %
0,https://www.google.com,0.550000,0.550000
1,https://www.google.com/search,99.230003,99.080002
2,https://www.google.com/search?q=python,99.980003,99.980003
3,https://github.com,0.480000,0.450000
4,https://github.com/explore,99.790001,99.720001
5,https://en.wikipedia.org/wiki/Machine_learning,100.000000,100.000000
6,https://stackoverflow.com/questions/12345,100.000000,100.000000
7,http://paypal-account-security-check-48291.com...,88.290001,78.379997
8,http://free-prize-winner-92731.com/claim,82.680000,72.930000
9,http://secure-login-account-verification-12345...,93.720001,89.110001


In [199]:
test_urls_complete = [
    # Legitimate websites
    "https://www.google.com",
    "https://www.google.com/search?q=python",
    "https://github.com/explore",
    "https://en.wikipedia.org/wiki/Machine_learning",
    "https://stackoverflow.com/questions/12345",

    # Suspicious-looking URLs
    "http://paypal-account-security-check-48291.com/verify",
    "http://free-prize-winner-92731.com/claim",
    "http://secure-login-account-verification-12345.com/login"
]

complete_results = []

for url in test_urls_complete:

    print("\n" + "=" * 90)
    print("Testing:", url)
    print("=" * 90)

    try:
        result = analyze_url_complete(url)

        complete_results.append(result)

        print("Prediction:", result["prediction"])
        print("Phishing Probability:", result["phishing_probability"], "%")
        print("Legitimate Probability:", result["legitimate_probability"], "%")
        print("Risk Level:", result["risk_level"])

        print("\nLive Domain Intelligence:")
        print("Domain Age:", result["domain_age_days"], "days")
        print("DNS Resolvable:", result["dns_resolvable"])
        print("Number of IPs:", result["num_ips"])
        print("TTL:", result["ttl"])
        print("Name Servers:", result["num_name_servers"])

        print("\nPhishing Reasons:")
        for reason in result["phishing_reasons"]:
            print("-", reason)

        print("\nLegitimate Reasons:")
        for reason in result["legitimate_reasons"]:
            print("-", reason)

    except Exception as e:
        print("ERROR:", e)


Testing: https://www.google.com
Prediction: Legitimate
Phishing Probability: 0.55 %
Legitimate Probability: 99.45 %
Risk Level: Low

Live Domain Intelligence:
Domain Age: 10606 days
DNS Resolvable: Yes
Number of IPs: 8
TTL: 300
Name Servers: 4

Phishing Reasons:
- Number of subdomains: 1 (contributed toward the phishing prediction)
- Hostname length: 14 (contributed toward the phishing prediction)
- Digit ratio in the hostname: 0.00% (contributed toward the phishing prediction)

Legitimate Reasons:
- URL length: 22 characters (contributed toward the legitimate prediction)
- Number of 'www' occurrences: 1 (contributed toward the legitimate prediction)
- URL character entropy: 3.66 (contributed toward the legitimate prediction)

Testing: https://www.google.com/search?q=python
Prediction: Phishing
Phishing Probability: 99.98 %
Legitimate Probability: 0.02 %
Risk Level: High

Live Domain Intelligence:
Domain Age: 10606 days
DNS Resolvable: Yes
Number of IPs: 8
TTL: 300
Name Servers: 4

Ph

In [200]:
def format_live_value(value, suffix=""):
    if value is None or value == -1:
        return "Unavailable"
    return f"{value}{suffix}"


def analyze_url_complete(url):
    result = analyze_url(url)

    phishing_probability = float(result["phishing_probability"])

    if phishing_probability >= 0.80:
        risk_level = "High"
    elif phishing_probability >= 0.50:
        risk_level = "Medium"
    else:
        risk_level = "Low"

    feature_explanation = generate_readable_feature_explanation(url)

    return {
        "url": url,
        "prediction": result["prediction"],

        "phishing_probability": round(
            float(result["phishing_probability"]) * 100, 2
        ),

        "legitimate_probability": round(
            float(result["legitimate_probability"]) * 100, 2
        ),

        "risk_level": risk_level,

        "phishing_reasons": feature_explanation["phishing_reasons"],
        "legitimate_reasons": feature_explanation["legitimate_reasons"],

        # Better handling of unavailable live data
        "domain_age_days": format_live_value(
            result["domain_age"], " days"
        ),

        "dns_resolvable": (
            "Yes"
            if result["dns_resolvable"] == 1
            else "No"
        ),

        "num_ips": format_live_value(
            result["num_ips"]
        ),

        "ttl": format_live_value(
            result["ttl"]
        ),

        "num_name_servers": format_live_value(
            result["num_name_servers"]
        )
    }


print("Updated analyze_url_complete() successfully.")

Updated analyze_url_complete() successfully.


In [201]:
test_result = analyze_url_complete(
    "http://free-prize-winner-92731.com/claim"
)

print(test_result)

{'url': 'http://free-prize-winner-92731.com/claim', 'prediction': 'Phishing', 'phishing_probability': 82.68, 'legitimate_probability': 17.32, 'risk_level': 'High', 'phishing_reasons': ['URL length: 40 characters (contributed toward the phishing prediction)', 'URL character entropy: 4.40 (contributed toward the phishing prediction)', 'Digit ratio in the URL: 12.50% (contributed toward the phishing prediction)'], 'legitimate_reasons': ['Digit ratio in the hostname: 18.52% (contributed toward the legitimate prediction)', 'Hostname length: 27 (contributed toward the legitimate prediction)', 'Number of dots in the URL: 1 (contributed toward the legitimate prediction)'], 'domain_age_days': 'Unavailable', 'dns_resolvable': 'No', 'num_ips': '0', 'ttl': 'Unavailable', 'num_name_servers': '0'}


In [202]:
import joblib
import json

# Save final trained model
joblib.dump(final_xgb_model, "phishing_xgb_model.pkl")

# Save feature names
joblib.dump(url_features_final, "feature_names.pkl")

# Save feature descriptions
joblib.dump(feature_descriptions, "feature_descriptions.pkl")

print("Files saved successfully:")
print("- phishing_xgb_model.pkl")
print("- feature_names.pkl")
print("- feature_descriptions.pkl")

Files saved successfully:
- phishing_xgb_model.pkl
- feature_names.pkl
- feature_descriptions.pkl


In [203]:
from google.colab import files

files.download("phishing_xgb_model.pkl")
files.download("feature_names.pkl")
files.download("feature_descriptions.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [204]:
# Save the XGBoost model in its native portable format
final_xgb_model.save_model("phishing_xgb_model.json")

print("Native XGBoost model saved successfully.")

Native XGBoost model saved successfully.


In [205]:
from google.colab import files

files.download("phishing_xgb_model.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [207]:
import random
import numpy as np
import pandas as pd
from urllib.parse import urlparse

random.seed(42)
np.random.seed(42)

print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))

Train rows: 47323
Test rows: 12491


In [209]:
train_data = df_clean.iloc[train_idx].copy()

domain_labels = train_data.groupby("domain")["label"].unique()

pure_legitimate_domains = domain_labels[
    domain_labels.apply(
        lambda x: len(x) == 1 and x[0] == "legitimate"
    )
].index.tolist()

print("Pure legitimate training domains:", len(pure_legitimate_domains))

Pure legitimate training domains: 20330


In [210]:
path_templates = [
    "/search",
    "/search/results",
    "/products",
    "/products/item",
    "/about",
    "/about-us",
    "/contact",
    "/help",
    "/support",
    "/blog",
    "/news",
    "/article",
    "/category",
    "/login",
    "/account",
    "/profile",
    "/settings",
    "/questions/12345",
    "/wiki/example",
    "/docs/getting-started",
    "/features",
    "/resources/article",
    "/customer-support",
    "/learn/python"
]

query_templates = [
    "/search?q=python",
    "/search?q=machine+learning",
    "/search?query=test",
    "/search?page=2",
    "/products?id=12345",
    "/products?category=electronics",
    "/article?id=12345",
    "/questions?page=2",
    "/search?q=laptop&page=2",
    "/catalog?page=3&sort=price"
]

In [211]:
augmented_urls = set()

target_count = 5000

while len(augmented_urls) < target_count:

    domain = random.choice(pure_legitimate_domains)

    if random.random() < 0.70:
        suffix = random.choice(path_templates)
    else:
        suffix = random.choice(query_templates)

    new_url = "https://" + domain + suffix

    augmented_urls.add(new_url)

augmented_urls = list(augmented_urls)

print("Augmented legitimate URLs:", len(augmented_urls))

Augmented legitimate URLs: 5000


In [212]:
aug_df = pd.DataFrame({
    "url": augmented_urls,
    "label": "legitimate"
})

aug_df["has_path"] = aug_df["url"].apply(
    lambda x: bool(urlparse(x).path)
)

aug_df["has_query"] = aug_df["url"].apply(
    lambda x: bool(urlparse(x).query)
)

print("Shape:", aug_df.shape)

print("\nPath distribution:")
print(aug_df["has_path"].value_counts(normalize=True) * 100)

print("\nQuery distribution:")
print(aug_df["has_query"].value_counts(normalize=True) * 100)

print("\nExamples:")
display(aug_df.head(10))

Shape: (5000, 4)

Path distribution:
has_path
True    100.0
Name: proportion, dtype: float64

Query distribution:
has_query
False    70.86
True     29.14
Name: proportion, dtype: float64

Examples:


,url,label,has_path,has_query
0,https://buddy4study.com/category,legitimate,True,False
1,https://ufrgs.br/products?id=12345,legitimate,True,True
2,https://epunjabschool.gov.in/docs/getting-started,legitimate,True,False
3,https://dailymail.co.uk/search/results,legitimate,True,False
4,https://xhamsterflirt.com/questions?page=2,legitimate,True,True
5,https://gtk.org/search?query=test,legitimate,True,True
6,https://shubt.net/login,legitimate,True,False
7,https://zalando-prive.it/login,legitimate,True,False
8,https://nmugroup.se/settings,legitimate,True,False
9,https://chiesacattolica.it/products/item,legitimate,True,False


In [213]:
augmented_features = pd.DataFrame(
    [
        extract_url_features(url)
        for url in augmented_urls
    ]
)[url_features_final]

augmented_labels = np.zeros(
    len(augmented_features),
    dtype=int
)

print("Augmented feature shape:", augmented_features.shape)

Augmented feature shape: (5000, 11)


In [214]:
X_train_base = X_train[url_features_final].reset_index(drop=True)
y_train_base = np.asarray(y_train)

X_train_structural = pd.concat(
    [
        X_train_base,
        augmented_features.reset_index(drop=True)
    ],
    ignore_index=True
)

y_train_structural = np.concatenate(
    [
        y_train_base,
        augmented_labels
    ]
)

print("Original training rows:", len(X_train_base))
print("Augmented rows:", len(augmented_features))
print("Final training rows:", len(X_train_structural))

Original training rows: 47323
Augmented rows: 5000
Final training rows: 52323


In [215]:
AUGMENTATION_WEIGHT = 0.35

sample_weights = np.concatenate([
    np.ones(len(X_train_base)),
    np.full(
        len(augmented_features),
        AUGMENTATION_WEIGHT
    )
])

print("Original sample weight: 1.0")
print("Synthetic sample weight:", AUGMENTATION_WEIGHT)

Original sample weight: 1.0
Synthetic sample weight: 0.35


In [216]:
from xgboost import XGBClassifier

structural_xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

structural_xgb_model.fit(
    X_train_structural,
    y_train_structural,
    sample_weight=sample_weights
)

print("Candidate structural model trained.")

Candidate structural model trained.


In [217]:
structural_test_prob = structural_xgb_model.predict_proba(
    X_test[url_features_final]
)[:, 1]

structural_test_pred = (
    structural_test_prob >= 0.5
).astype(int)

In [218]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("STRUCTURAL MODEL")
print("----------------")

print(
    "Accuracy:",
    accuracy_score(y_test, structural_test_pred)
)

print(
    "Precision:",
    precision_score(y_test, structural_test_pred)
)

print(
    "Recall:",
    recall_score(y_test, structural_test_pred)
)

print(
    "F1:",
    f1_score(y_test, structural_test_pred)
)

print(
    "ROC-AUC:",
    roc_auc_score(y_test, structural_test_prob)
)

STRUCTURAL MODEL
----------------
Accuracy: 0.9815867424545673
Precision: 0.9981820936221785
Recall: 0.9679741442632585
F1: 0.982846062052506
ROC-AUC: 0.9977230485817524


In [219]:
test_data = df_clean.iloc[test_idx].copy()

test_legitimate_domains = (
    test_data[
        test_data["label"] == "legitimate"
    ]["domain"]
    .drop_duplicates()
    .tolist()
)

print(
    "Legitimate test domains:",
    len(test_legitimate_domains)
)

Legitimate test domains: 5088


In [220]:
challenge_suffixes = [
    "/search",
    "/search?q=python",
    "/products/item",
    "/products?id=12345",
    "/article?page=2",
    "/questions/12345?answer=1",
    "/wiki/example",
    "/docs/getting-started",
    "/search?q=machine+learning&page=2"
]

In [221]:
challenge_urls = [
    "https://" + domain + random.choice(challenge_suffixes)
    for domain in test_legitimate_domains[
        :min(1000, len(test_legitimate_domains))
    ]
]

print("Challenge URLs:", len(challenge_urls))

Challenge URLs: 1000


In [222]:
challenge_features = pd.DataFrame(
    [
        extract_url_features(url)
        for url in challenge_urls
    ]
)[url_features_final]

current_challenge_prob = final_xgb_model.predict_proba(
    challenge_features
)[:, 1]

structural_challenge_prob = structural_xgb_model.predict_proba(
    challenge_features
)[:, 1]

In [224]:
challenge_results = pd.DataFrame({
    "url": challenge_urls,
    "current_probability": current_challenge_prob,
    "structural_probability": structural_challenge_prob
})

challenge_results.head(20)

,url,current_probability,structural_probability
0,https://viiayrmr.com/docs/getting-started,0.999957,0.269882
1,https://culturedcode.com/search,0.997557,0.316748
2,https://freshauto.ru/docs/getting-started,0.999933,0.128759
3,https://exaly.com/questions/12345?answer=1,0.999992,0.443376
4,https://pngegg.com/questions/12345?answer=1,0.999995,0.058689
5,https://lenovosoftware.com/search?q=python,0.999777,0.027834
6,https://upzero.net/questions/12345?answer=1,0.999993,0.040092
7,https://pubmatic.com/search?q=python,0.999897,0.024548
8,https://algolianet.com/products/item,0.999769,0.236015
9,https://swiftgram.app/products/item,0.999686,0.210698


In [225]:
print(
    "Current model average phishing probability:",
    challenge_results["current_probability"].mean()
)

print(
    "Structural model average phishing probability:",
    challenge_results["structural_probability"].mean()
)

print(
    "Current model predicted phishing:",
    (challenge_results["current_probability"] >= 0.5).mean() * 100,
    "%"
)

print(
    "Structural model predicted phishing:",
    (challenge_results["structural_probability"] >= 0.5).mean() * 100,
    "%"
)

Current model average phishing probability: 0.99945307
Structural model average phishing probability: 0.22555731
Current model predicted phishing: 100.0 %
Structural model predicted phishing: 12.1 %


In [226]:
test_urls = [
    "https://www.google.com",
    "https://www.google.com/search",
    "https://www.google.com/search?q=python",
    "https://github.com",
    "https://github.com/login",
    "https://en.wikipedia.org/wiki/Python",
    "https://stackoverflow.com/questions/12345",

    "http://secure-login-account-verification-12345.com/login",
    "http://paypal-account-security-check-48291.com/verify",
    "http://free-prize-winner-92731.com/claim"
]

test_features = pd.DataFrame(
    [
        extract_url_features(url)
        for url in test_urls
    ]
)[url_features_final]

old_probs = final_xgb_model.predict_proba(
    test_features
)[:, 1]

new_probs = structural_xgb_model.predict_proba(
    test_features
)[:, 1]

comparison = pd.DataFrame({
    "URL": test_urls,
    "Old_Model": old_probs,
    "Structural_Model": new_probs
})

comparison["Old_Prediction"] = np.where(
    comparison["Old_Model"] >= 0.5,
    "Phishing",
    "Legitimate"
)

comparison["New_Prediction"] = np.where(
    comparison["Structural_Model"] >= 0.5,
    "Phishing",
    "Legitimate"
)

display(comparison)

,URL,Old_Model,Structural_Model,Old_Prediction,New_Prediction
0,https://www.google.com,0.005526,0.006990,Legitimate,Legitimate
1,https://www.google.com/search,0.992316,0.832320,Phishing,Phishing
2,https://www.google.com/search?q=python,0.999785,0.984924,Phishing,Phishing
3,https://github.com,0.004833,0.005206,Legitimate,Legitimate
4,https://github.com/login,0.998372,0.336811,Phishing,Legitimate
5,https://en.wikipedia.org/wiki/Python,0.999992,0.999647,Phishing,Phishing
6,https://stackoverflow.com/questions/12345,0.999981,0.344011,Phishing,Legitimate
7,http://secure-login-account-verification-12345...,0.937206,0.890692,Phishing,Phishing
8,http://paypal-account-security-check-48291.com...,0.882909,0.737893,Phishing,Phishing
9,http://free-prize-winner-92731.com/claim,0.826796,0.193596,Phishing,Legitimate


In [227]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

structural_test_prob = structural_xgb_model.predict_proba(
    X_test[url_features_final]
)[:, 1]

structural_test_pred = (
    structural_test_prob >= 0.5
).astype(int)

print("STRUCTURAL MODEL — ORIGINAL TEST SET")
print("Accuracy :", accuracy_score(y_test, structural_test_pred))
print("Precision:", precision_score(y_test, structural_test_pred))
print("Recall   :", recall_score(y_test, structural_test_pred))
print("F1       :", f1_score(y_test, structural_test_pred))
print("ROC-AUC  :", roc_auc_score(y_test, structural_test_prob))

STRUCTURAL MODEL — ORIGINAL TEST SET
Accuracy : 0.9815867424545673
Precision: 0.9981820936221785
Recall   : 0.9679741442632585
F1       : 0.982846062052506
ROC-AUC  : 0.9977230485817524


In [228]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

weights_to_test = [0.10, 0.15, 0.20, 0.25]

weight_results = []

for weight in weights_to_test:

    sample_weights_test = np.concatenate([
        np.ones(len(X_train_base)),
        np.full(len(augmented_features), weight)
    ])

    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_structural,
        y_train_structural,
        sample_weight=sample_weights_test
    )

    test_prob = model.predict_proba(
        X_test[url_features_final]
    )[:, 1]

    test_pred = (test_prob >= 0.5).astype(int)

    weight_results.append({
        "weight": weight,
        "accuracy": accuracy_score(y_test, test_pred),
        "precision": precision_score(y_test, test_pred),
        "recall": recall_score(y_test, test_pred),
        "f1": f1_score(y_test, test_pred),
        "roc_auc": roc_auc_score(y_test, test_prob)
    })

weight_results_df = pd.DataFrame(weight_results)

display(
    weight_results_df.style.format({
        "accuracy": "{:.4f}",
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "f1": "{:.4f}",
        "roc_auc": "{:.4f}"
    })
)

,weight,accuracy,precision,recall,f1,roc_auc
0,0.100000,0.9903,0.9987,0.9835,0.9910,0.9979
1,0.150000,0.9869,0.9988,0.9771,0.9878,0.9979
2,0.200000,0.9850,0.9988,0.9737,0.9861,0.9979
3,0.250000,0.9836,0.9988,0.9711,0.9847,0.9979


In [229]:
challenge_urls = [
    "https://www.google.com",
    "https://www.google.com/search",
    "https://www.google.com/search?q=python",
    "https://github.com",
    "https://github.com/login",
    "https://en.wikipedia.org/wiki/Python",
    "https://stackoverflow.com/questions/12345",
    "http://secure-login-account-verification-12345.com/login",
    "http://paypal-account-security-check-48291.com/verify",
    "http://free-prize-winner-92731.com/claim"
]

challenge_features = pd.DataFrame(
    [extract_url_features(url) for url in challenge_urls]
)[url_features_final]

challenge_comparison = pd.DataFrame({
    "URL": challenge_urls,
    "Current_Model": final_xgb_model.predict_proba(
        challenge_features
    )[:, 1]
})

for weight in weights_to_test:

    sample_weights_test = np.concatenate([
        np.ones(len(X_train_base)),
        np.full(len(augmented_features), weight)
    ])

    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_structural,
        y_train_structural,
        sample_weight=sample_weights_test
    )

    challenge_comparison[f"Weight_{weight}"] = model.predict_proba(
        challenge_features
    )[:, 1]

display(
    challenge_comparison.style.format({
        "Current_Model": "{:.4f}",
        "Weight_0.1": "{:.4f}",
        "Weight_0.15": "{:.4f}",
        "Weight_0.2": "{:.4f}",
        "Weight_0.25": "{:.4f}"
    })
)

,URL,Current_Model,Weight_0.1,Weight_0.15,Weight_0.2,Weight_0.25
0,https://www.google.com,0.0055,0.0072,0.0065,0.0068,0.0064
1,https://www.google.com/search,0.9923,0.8169,0.8137,0.8405,0.7969
2,https://www.google.com/search?q=python,0.9998,0.9926,0.9908,0.9863,0.9884
3,https://github.com,0.0048,0.0049,0.0051,0.0044,0.0051
4,https://github.com/login,0.9984,0.6729,0.5734,0.4392,0.3750
5,https://en.wikipedia.org/wiki/Python,1.0000,0.9998,0.9997,0.9997,0.9997
6,https://stackoverflow.com/questions/12345,1.0000,0.6106,0.6123,0.5149,0.4994
7,http://secure-login-account-verification-12345.com/login,0.9372,0.9063,0.9429,0.9301,0.9387
8,http://paypal-account-security-check-48291.com/verify,0.8829,0.7649,0.7829,0.8597,0.7974
9,http://free-prize-winner-92731.com/claim,0.8268,0.3107,0.2725,0.3099,0.1913


In [231]:
import pandas as pd

external_df = pd.read_csv("url_features_extracted1.csv")

print(external_df.shape)
print(external_df.columns.tolist())
display(external_df.head())

(101219, 18)
['URL', 'url_length', 'has_ip_address', 'dot_count', 'https_flag', 'url_entropy', 'token_count', 'subdomain_count', 'query_param_count', 'tld_length', 'path_length', 'has_hyphen_in_domain', 'number_of_digits', 'tld_popularity', 'suspicious_file_extension', 'domain_name_length', 'percentage_numeric_chars', 'ClassLabel']


,URL,url_length,has_ip_address,dot_count,https_flag,url_entropy,token_count,subdomain_count,query_param_count,tld_length,path_length,has_hyphen_in_domain,number_of_digits,tld_popularity,suspicious_file_extension,domain_name_length,percentage_numeric_chars,ClassLabel
0,https://keraekken-loagginnusa.godaddysites.com/,47,0,2,1,4.250669,6,1,1,3,1,1,0,1,0,12,0.0,0.0
1,https://metamsk01lgiix.godaddysites.com/,40,0,2,1,4.196439,6,1,1,3,1,0,2,1,0,12,5.0,0.0
2,http://myglobaltech.in/,23,0,1,0,3.936180,5,0,1,2,1,0,0,0,0,12,0.0,0.0
3,http://djtool-for-spotify.com/,30,0,1,0,3.894740,5,0,1,3,1,1,0,1,0,18,0.0,0.0
4,https://scearmcoommunnlty.com/invent/freind/get,47,0,1,1,4.143127,7,0,1,3,18,0,0,1,0,17,0.0,0.0


In [232]:
# Show every column clearly
for i, col in enumerate(external_df.columns):
    print(i, "->", col)

print("\nDataset shape:", external_df.shape)

0 -> URL
1 -> url_length
2 -> has_ip_address
3 -> dot_count
4 -> https_flag
5 -> url_entropy
6 -> token_count
7 -> subdomain_count
8 -> query_param_count
9 -> tld_length
10 -> path_length
11 -> has_hyphen_in_domain
12 -> number_of_digits
13 -> tld_popularity
14 -> suspicious_file_extension
15 -> domain_name_length
16 -> percentage_numeric_chars
17 -> ClassLabel

Dataset shape: (101219, 18)


In [233]:
# Check possible label columns
for col in external_df.columns:
    print(f"\n--- {col} ---")
    print(external_df[col].value_counts(dropna=False).head(10))


--- URL ---
URL
https://www.forbesindia.com/article/explainers/most-visited-websites-world/92619/1                                       4
https://www.pinterest.com/pin/745486544557523958/                                                                        4
https://pressgazette.co.uk/media-audience-and-business-data/media_metrics/most-popular-websites-news-world-monthly-2/    4
https://www.thetoptens.com/internet/websites/                                                                            4
https://insightssuccess.com/top-9-most-popular-websites-worldwide/                                                       4
https://www.comparethemarket.com/broadband/news/the-worlds-most-popular-websites/                                        4
https://support.apple.com/guide/safari/see-your-favorite-websites-sfri22092/mac                                          4
https://time.com/4960202/most-influential-websites/                                                                      4

In [234]:
import tldextract

# Get URLs belonging to the untouched test set
test_urls = df_clean.iloc[test_idx]["url"].dropna().astype(str)

# Extract registered/apex domains
test_domains = set()

for url in test_urls:
    ext = tldextract.extract(url)
    domain = ext.top_domain_under_public_suffix

    if domain:
        test_domains.add(domain)

print("Number of test domains:", len(test_domains))
print("Example test domains:", list(test_domains)[:20])

Number of test domains: 6134
Example test domains: ['snow.me', 'wpmudev.org', 'boxberry.ru', 'ooredoo.qa', 'freedompay.com', 'tudelft.nl', 'ln.run', 'sc.com', 'virginia.gov', 'newsouthwalestourism.com', 'nebraska.edu', 'commscope.com', 'examsoft.com', 'monsterinsights.com', 'page.tl', 'runetki6.com', 'albany.edu', 'staples.com', 'nohu90.com', 'pari.ru']


In [235]:
external_legit = external_df[
    external_df["ClassLabel"] == 0
].copy()

print("Legitimate external URLs:", len(external_legit))

Legitimate external URLs: 63678


In [238]:
import tldextract

def get_registered_domain(url):
    try:
        ext = tldextract.extract(str(url))
        return ext.top_domain_under_public_suffix
    except Exception:
        return ""

In [240]:
external_legit["registered_domain"] = external_legit["URL"].apply(
    get_registered_domain
)

external_legit_no_overlap = external_legit[
    ~external_legit["registered_domain"].isin(test_domains)
].copy()

print("Before removing test domains:", len(external_legit))
print("After removing test domains:", len(external_legit_no_overlap))
print(
    "Remaining unique domains:",
    external_legit_no_overlap["registered_domain"].nunique()
)

Before removing test domains: 63678
After removing test domains: 62957
Remaining unique domains: 1507


In [241]:
from urllib.parse import urlparse

def has_path_or_query(url):
    try:
        parsed = urlparse(str(url))
        return (
            (parsed.path and parsed.path != "/")
            or bool(parsed.query)
        )
    except Exception:
        return False

external_legit_no_overlap["has_path_or_query"] = (
    external_legit_no_overlap["URL"].apply(has_path_or_query)
)

print(
    external_legit_no_overlap["has_path_or_query"].value_counts()
)

has_path_or_query
True     62869
False       88
Name: count, dtype: int64


In [242]:
usable_legit = external_legit_no_overlap[
    external_legit_no_overlap["has_path_or_query"]
].copy()

print("Usable legitimate URLs:", len(usable_legit))

display(
    usable_legit[
        ["URL", "registered_domain"]
    ].head(20)
)

Usable legitimate URLs: 62869


,URL,registered_domain
4,https://scearmcoommunnlty.com/invent/freind/get,scearmcoommunnlty.com
41,https://scearmcoommunnlty.com/activating/get/put,scearmcoommunnlty.com
44,https://flowcode.com/p/eDxkv9bHRZ?fc=0/,flowcode.com
53,http://safemessaging.org/landing/form/bed8cb3f...,safemessaging.org
65,https://scearmcoommunnlty.com/here/activate/get,scearmcoommunnlty.com
80,https://treinamento.jovemaprendizamericana.com...,jovemaprendizamericana.com.br
81,https://vinov-breyselislemim.homes/vakifbank/g...,vinov-breyselislemim.homes
82,https://helpy-ake.top/i/,helpy-ake.top
87,https://meta.case-page-appeal.eu/community-sta...,case-page-appeal.eu
93,https://greenhousepharmaceuticals.com/wp-inclu...,greenhousepharmaceuticals.com


In [243]:
external_legit = external_df[
    external_df["ClassLabel"] == 1
].copy()

print("Legitimate external URLs:", len(external_legit))

Legitimate external URLs: 37540


In [244]:
external_legit["registered_domain"] = external_legit["URL"].apply(
    get_registered_domain
)

external_legit_no_overlap = external_legit[
    ~external_legit["registered_domain"].isin(test_domains)
].copy()

print("Before removing test domains:", len(external_legit))
print("After removing test domains:", len(external_legit_no_overlap))
print(
    "Remaining unique domains:",
    external_legit_no_overlap["registered_domain"].nunique()
)

Before removing test domains: 37540
After removing test domains: 37167
Remaining unique domains: 36287


In [245]:
external_legit_no_overlap["has_path_or_query"] = (
    external_legit_no_overlap["URL"].apply(has_path_or_query)
)

print(
    external_legit_no_overlap["has_path_or_query"].value_counts()
)

has_path_or_query
False    36690
True       477
Name: count, dtype: int64


In [246]:
usable_legit = external_legit_no_overlap[
    external_legit_no_overlap["has_path_or_query"]
].copy()

print("Usable legitimate URLs:", len(usable_legit))

display(
    usable_legit[
        ["URL", "registered_domain"]
    ].head(20)
)

Usable legitimate URLs: 477


,URL,registered_domain
63678,https://stackoverflow.com/questions/tagged/php,stackoverflow.com
63679,https://stackoverflow.com/questions/tagged/mic...,stackoverflow.com
63680,https://stackoverflow.com/questions/tagged/sql,stackoverflow.com
63681,https://stackoverflow.com/questions/79030270/h...,stackoverflow.com
63682,https://stackoverflow.com/questions/tagged/java,stackoverflow.com
63683,https://stackoverflow.com/questions/tagged/iwyu,stackoverflow.com
63684,https://stackoverflow.com/questions/tagged/and...,stackoverflow.com
63685,https://stackoverflow.com/questions/tagged/dep...,stackoverflow.com
63686,https://stackoverflow.com/questions/tagged/mac...,stackoverflow.com
63687,https://stackoverflow.com/questions/tagged/build,stackoverflow.com


In [247]:
print("Legitimate external URLs:", len(external_legit))
print("After removing test domains:", len(external_legit_no_overlap))
print(
    "Remaining unique domains:",
    external_legit_no_overlap["registered_domain"].nunique()
)

print("\nPath/query distribution:")
print(
    external_legit_no_overlap["has_path_or_query"]
    .value_counts()
)

usable_legit = external_legit_no_overlap[
    external_legit_no_overlap["has_path_or_query"]
].copy()

print("\nUsable legitimate URLs:", len(usable_legit))

print("\nSample usable URLs:")
display(
    usable_legit[["URL", "registered_domain"]].head(20)
)

Legitimate external URLs: 37540
After removing test domains: 37167
Remaining unique domains: 36287

Path/query distribution:
has_path_or_query
False    36690
True       477
Name: count, dtype: int64

Usable legitimate URLs: 477

Sample usable URLs:


,URL,registered_domain
63678,https://stackoverflow.com/questions/tagged/php,stackoverflow.com
63679,https://stackoverflow.com/questions/tagged/mic...,stackoverflow.com
63680,https://stackoverflow.com/questions/tagged/sql,stackoverflow.com
63681,https://stackoverflow.com/questions/79030270/h...,stackoverflow.com
63682,https://stackoverflow.com/questions/tagged/java,stackoverflow.com
63683,https://stackoverflow.com/questions/tagged/iwyu,stackoverflow.com
63684,https://stackoverflow.com/questions/tagged/and...,stackoverflow.com
63685,https://stackoverflow.com/questions/tagged/dep...,stackoverflow.com
63686,https://stackoverflow.com/questions/tagged/mac...,stackoverflow.com
63687,https://stackoverflow.com/questions/tagged/build,stackoverflow.com


In [248]:
print("Legitimate external URLs:", len(external_legit))
print("After removing test domains:", len(external_legit_no_overlap))
print("Remaining unique domains:",
      external_legit_no_overlap["registered_domain"].nunique())

print("\nPath/query distribution:")
print(
    external_legit_no_overlap["has_path_or_query"].value_counts()
)

print("\nUsable legitimate URLs:", len(usable_legit))
print("Usable legitimate domains:",
      usable_legit["registered_domain"].nunique())

Legitimate external URLs: 37540
After removing test domains: 37167
Remaining unique domains: 36287

Path/query distribution:
has_path_or_query
False    36690
True       477
Name: count, dtype: int64

Usable legitimate URLs: 477
Usable legitimate domains: 113


In [250]:
print("Number of usable URLs:", len(usable_legit))
print("Unique URLs:", usable_legit["URL"].nunique())
print("Unique domains:", usable_legit["registered_domain"].nunique())

print("\nDuplicate URLs:")
print(
    usable_legit["URL"].duplicated().sum()
)

print("\nPath/query examples:")
display(
    usable_legit[
        ["URL", "registered_domain"]
    ].sample(
        min(30, len(usable_legit)),
        random_state=42
    )
)

Number of usable URLs: 477
Unique URLs: 362
Unique domains: 113

Duplicate URLs:
115

Path/query examples:


,URL,registered_domain
64265,https://www.nominus.com/en/dm/blog/hosting/wha...,nominus.com
63711,https://stackoverflow.com/questions/79030256/h...,stackoverflow.com
63815,https://en.wikipedia.org/wiki/Jimmie%27s_Millions,wikipedia.org
63750,https://stackoverflow.com/questions/79030258/h...,stackoverflow.com
63756,https://stackoverflow.com/questions/tagged/ani...,stackoverflow.com
63797,https://en.wikipedia.org/wiki/Daisy_Waterstone,wikipedia.org
63988,https://www.designrush.com/best-designs/websites,designrush.com
63869,https://en.wikipedia.org/wiki/Brachytrematidae,wikipedia.org
63962,https://www.youtube.com/watch?v=VU69Mr7e9XQ,youtube.com
63687,https://stackoverflow.com/questions/tagged/build,stackoverflow.com


In [253]:
from urllib.parse import urlparse

def get_url_structure(url):
    parsed = urlparse(str(url))

    has_path = bool(parsed.path and parsed.path != "/")
    has_query = bool(parsed.query)

    if has_path and has_query:
        return "path + query"
    elif has_path:
        return "path only"
    elif has_query:
        return "query only"
    else:
        return "none"

usable_legit["structure_type"] = (
    usable_legit["URL"].apply(get_url_structure)
)

print(
    usable_legit["structure_type"].value_counts()
)

structure_type
path only       428
path + query     48
query only        1
Name: count, dtype: int64


In [254]:
print("\nAverage URL length:",
      usable_legit["URL"].str.len().mean())

print("Median URL length:",
      usable_legit["URL"].str.len().median())

print("Maximum URL length:",
      usable_legit["URL"].str.len().max())


Average URL length: 59.18448637316562
Median URL length: 51.0
Maximum URL length: 178


In [255]:
original_urls = set(
    df_clean["url"]
    .dropna()
    .astype(str)
)

external_exact_overlap = (
    usable_legit["URL"].isin(original_urls).sum()
)

print(
    "Exact URL overlap with original dataset:",
    external_exact_overlap
)

Exact URL overlap with original dataset: 0


In [256]:
usable_legit_clean = (
    usable_legit
    .drop_duplicates(subset=["URL"])
    .copy()
)

print("Before deduplication:", len(usable_legit))
print("After deduplication:", len(usable_legit_clean))
print(
    "Unique domains:",
    usable_legit_clean["registered_domain"].nunique()
)

Before deduplication: 477
After deduplication: 362
Unique domains: 113


In [257]:
domain_counts = (
    usable_legit_clean["registered_domain"]
    .value_counts()
)

print(domain_counts.head(20))

registered_domain
wikipedia.org           102
stackoverflow.com       100
youtube.com              37
ahrefs.com                2
explodingtopics.com       2
reddit.com                2
statista.com              2
businessinsider.com       2
kaggle.com                2
ssa.gov                   2
cloudflare.com            2
forbes.com                2
socialmediatoday.com      2
apple.com                 2
pewresearch.org           2
un.org                    2
pressgazette.co.uk        1
domcop.com                1
hostinger.com             1
dataforseo.com            1
Name: count, dtype: int64


In [258]:
# Keep at most 5 legitimate path/query URLs from each domain
balanced_legit = (
    usable_legit_clean
    .groupby("registered_domain", group_keys=False)
    .apply(
        lambda x: x.sample(
            n=min(len(x), 5),
            random_state=42
        )
    )
    .reset_index(drop=True)
)

print("Balanced legitimate URLs:", len(balanced_legit))
print(
    "Unique domains:",
    balanced_legit["registered_domain"].nunique()
)

print("\nDomain distribution:")
print(
    balanced_legit["registered_domain"].value_counts().head(20)
)

Balanced legitimate URLs: 138
Unique domains: 113

Domain distribution:
registered_domain
youtube.com             5
wikipedia.org           5
stackoverflow.com       5
businessinsider.com     2
cloudflare.com          2
forbes.com              2
apple.com               2
explodingtopics.com     2
un.org                  2
pewresearch.org         2
reddit.com              2
socialmediatoday.com    2
ssa.gov                 2
kaggle.com              2
ahrefs.com              2
statista.com            2
affiliatefix.com        1
census.gov              1
c-sharpcorner.com       1
builtwith.com           1
Name: count, dtype: int64


/tmp/ipykernel_1303/3113588283.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [259]:
external_features = pd.DataFrame(
    [
        extract_url_features(url)
        for url in balanced_legit["URL"]
    ]
)[url_features_final]

external_labels = np.zeros(
    len(external_features),
    dtype=int
)

print("External feature shape:", external_features.shape)
print("External labels:", np.unique(external_labels, return_counts=True))

External feature shape: (138, 11)
External labels: (array([0]), array([138]))


In [260]:
display(external_features.head())

print("\nFeature columns:")
print(external_features.columns.tolist())

,entropy,url_length,num_www,ratio_digits_url,num_subdomains,length_hostname,num_dots,num_hyphens,ratio_digits_host,num_question_mark,num_eq
0,4.203365,46,0,0.065217,0,15,1,2,0.200000,0,0
1,3.985873,41,0,0.048780,0,8,1,1,0.250000,0,0
2,3.770942,28,0,0.142857,0,12,1,0,0.333333,0,0
3,4.497828,86,1,0.093023,1,20,3,7,0.000000,0,0
4,4.216651,45,0,0.000000,0,10,1,2,0.000000,0,0



Feature columns:
['entropy', 'url_length', 'num_www', 'ratio_digits_url', 'num_subdomains', 'length_hostname', 'num_dots', 'num_hyphens', 'ratio_digits_host', 'num_question_mark', 'num_eq']


In [261]:
# ---------------------------------------------------------
# REAL LEGITIMATE URL AUGMENTATION
# ---------------------------------------------------------

# Original training data
X_train_real = X_train[url_features_final].reset_index(drop=True)
y_train_real = np.asarray(y_train)

# External legitimate URLs
X_external_real = external_features.reset_index(drop=True)
y_external_real = np.zeros(
    len(X_external_real),
    dtype=int
)

# Combine original + real legitimate URLs
X_train_real_aug = pd.concat(
    [
        X_train_real,
        X_external_real
    ],
    ignore_index=True
)

y_train_real_aug = np.concatenate(
    [
        y_train_real,
        y_external_real
    ]
)

# Give real external legitimate URLs more influence
REAL_AUGMENTATION_WEIGHT = 5.0

real_sample_weights = np.concatenate(
    [
        np.ones(len(X_train_real)),
        np.full(
            len(X_external_real),
            REAL_AUGMENTATION_WEIGHT
        )
    ]
)

print("Original training samples:",
      len(X_train_real))

print("External legitimate samples:",
      len(X_external_real))

print("Combined training samples:",
      len(X_train_real_aug))

print("External sample weight:",
      REAL_AUGMENTATION_WEIGHT)

Original training samples: 47323
External legitimate samples: 138
Combined training samples: 47461
External sample weight: 5.0


In [262]:
real_aug_xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

real_aug_xgb_model.fit(
    X_train_real_aug,
    y_train_real_aug,
    sample_weight=real_sample_weights
)

print("Real-data augmented model trained.")

Real-data augmented model trained.


In [263]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

real_test_prob = real_aug_xgb_model.predict_proba(
    X_test[url_features_final]
)[:, 1]

real_test_pred = (
    real_test_prob >= 0.5
).astype(int)

print("REAL-DATA AUGMENTED MODEL — ORIGINAL TEST SET")
print("Accuracy :", accuracy_score(y_test, real_test_pred))
print("Precision:", precision_score(y_test, real_test_pred))
print("Recall   :", recall_score(y_test, real_test_pred))
print("F1       :", f1_score(y_test, real_test_pred))
print("ROC-AUC  :", roc_auc_score(y_test, real_test_prob))

REAL-DATA AUGMENTED MODEL — ORIGINAL TEST SET
Accuracy : 0.989192218397246
Precision: 0.9977618621307073
Recall   : 0.9823710885852799
F1       : 0.9900066622251832
ROC-AUC  : 0.9979488892865699


In [264]:
real_challenge_prob = real_aug_xgb_model.predict_proba(
    challenge_features
)[:, 1]

real_challenge_results = pd.DataFrame({
    "URL": challenge_urls,
    "Current_Model": final_xgb_model.predict_proba(
        challenge_features
    )[:, 1],
    "Real_Augmented_Model": real_challenge_prob
})

real_challenge_results["Current_Prediction"] = np.where(
    real_challenge_results["Current_Model"] >= 0.5,
    "Phishing",
    "Legitimate"
)

real_challenge_results["Real_Augmented_Prediction"] = np.where(
    real_challenge_results["Real_Augmented_Model"] >= 0.5,
    "Phishing",
    "Legitimate"
)

display(
    real_challenge_results.style.format({
        "Current_Model": "{:.4f}",
        "Real_Augmented_Model": "{:.4f}"
    })
)

,URL,Current_Model,Real_Augmented_Model,Current_Prediction,Real_Augmented_Prediction
0,https://www.google.com,0.0055,0.0046,Legitimate,Legitimate
1,https://www.google.com/search,0.9923,0.6741,Phishing,Phishing
2,https://www.google.com/search?q=python,0.9998,0.7541,Phishing,Phishing
3,https://github.com,0.0048,0.0043,Legitimate,Legitimate
4,https://github.com/login,0.9984,0.9909,Phishing,Phishing
5,https://en.wikipedia.org/wiki/Python,1.0000,0.9968,Phishing,Phishing
6,https://stackoverflow.com/questions/12345,1.0000,0.9498,Phishing,Phishing
7,http://secure-login-account-verification-12345.com/login,0.9372,0.7975,Phishing,Phishing
8,http://paypal-account-security-check-48291.com/verify,0.8829,0.5915,Phishing,Phishing
9,http://free-prize-winner-92731.com/claim,0.8268,0.5826,Phishing,Phishing


In [265]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

real_weights_to_test = [1.0, 2.0, 3.0]

real_weight_results = []

for weight in real_weights_to_test:

    sample_weights = np.concatenate([
        np.ones(len(X_train_real)),
        np.full(len(X_external_real), weight)
    ])

    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_real_aug,
        y_train_real_aug,
        sample_weight=sample_weights
    )

    prob = model.predict_proba(
        X_test[url_features_final]
    )[:, 1]

    pred = (prob >= 0.5).astype(int)

    real_weight_results.append({
        "weight": weight,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, prob)
    })

real_weight_results_df = pd.DataFrame(real_weight_results)

display(
    real_weight_results_df.style.format({
        "accuracy": "{:.4f}",
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "f1": "{:.4f}",
        "roc_auc": "{:.4f}"
    })
)

,weight,accuracy,precision,recall,f1,roc_auc
0,1.000000,0.9918,0.9984,0.9866,0.9925,0.9980
1,2.000000,0.9906,0.9985,0.9841,0.9913,0.9979
2,3.000000,0.9900,0.9982,0.9834,0.9907,0.9980


In [266]:
# Train the final candidate with real legitimate URLs at weight = 1

real_weight_1 = 1.0

sample_weights_1 = np.concatenate([
    np.ones(len(X_train_real)),
    np.full(len(X_external_real), real_weight_1)
])

real_aug_xgb_1 = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

real_aug_xgb_1.fit(
    X_train_real_aug,
    y_train_real_aug,
    sample_weight=sample_weights_1
)

print("Weight-1 model trained.")

Weight-1 model trained.


In [267]:
real_1_prob = real_aug_xgb_1.predict_proba(
    challenge_features
)[:, 1]

challenge_weight1 = pd.DataFrame({
    "URL": challenge_urls,
    "Current_Model": final_xgb_model.predict_proba(
        challenge_features
    )[:, 1],
    "Real_Weight_1": real_1_prob
})

challenge_weight1["Current_Prediction"] = np.where(
    challenge_weight1["Current_Model"] >= 0.5,
    "Phishing",
    "Legitimate"
)

challenge_weight1["Weight_1_Prediction"] = np.where(
    challenge_weight1["Real_Weight_1"] >= 0.5,
    "Phishing",
    "Legitimate"
)

display(
    challenge_weight1.style.format({
        "Current_Model": "{:.4f}",
        "Real_Weight_1": "{:.4f}"
    })
)

,URL,Current_Model,Real_Weight_1,Current_Prediction,Weight_1_Prediction
0,https://www.google.com,0.0055,0.0044,Legitimate,Legitimate
1,https://www.google.com/search,0.9923,0.9268,Phishing,Phishing
2,https://www.google.com/search?q=python,0.9998,0.8725,Phishing,Phishing
3,https://github.com,0.0048,0.0045,Legitimate,Legitimate
4,https://github.com/login,0.9984,0.9945,Phishing,Phishing
5,https://en.wikipedia.org/wiki/Python,1.0000,0.9989,Phishing,Phishing
6,https://stackoverflow.com/questions/12345,1.0000,0.9830,Phishing,Phishing
7,http://secure-login-account-verification-12345.com/login,0.9372,0.6624,Phishing,Phishing
8,http://paypal-account-security-check-48291.com/verify,0.8829,0.7064,Phishing,Phishing
9,http://free-prize-winner-92731.com/claim,0.8268,0.5231,Phishing,Phishing


In [268]:
diagnostic_features = [
    "entropy",
    "num_www",
    "ratio_digits_url",
    "num_subdomains",
    "length_hostname",
    "num_dots",
    "num_hyphens",
    "ratio_digits_host"
]

X_train_diag = X_train[diagnostic_features]
X_test_diag = X_test[diagnostic_features]

diagnostic_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

diagnostic_xgb.fit(
    X_train_diag,
    y_train
)

diag_prob = diagnostic_xgb.predict_proba(
    X_test_diag
)[:, 1]

diag_pred = (diag_prob >= 0.5).astype(int)

print("HOSTNAME-FOCUSED DIAGNOSTIC MODEL")
print("Accuracy :", accuracy_score(y_test, diag_pred))
print("Precision:", precision_score(y_test, diag_pred))
print("Recall   :", recall_score(y_test, diag_pred))
print("F1       :", f1_score(y_test, diag_pred))
print("ROC-AUC  :", roc_auc_score(y_test, diag_prob))

HOSTNAME-FOCUSED DIAGNOSTIC MODEL
Accuracy : 0.9767832839644544
Precision: 0.9939366378656965
Recall   : 0.9632731012193331
F1       : 0.9783646672635035
ROC-AUC  : 0.9940046116165345


In [269]:
challenge_diag_features = challenge_features[diagnostic_features]

diag_challenge_prob = diagnostic_xgb.predict_proba(
    challenge_diag_features
)[:, 1]

diagnostic_challenge_results = pd.DataFrame({
    "URL": challenge_urls,
    "Current_Model": final_xgb_model.predict_proba(
        challenge_features
    )[:, 1],
    "Diagnostic_Model": diag_challenge_prob
})

diagnostic_challenge_results["Diagnostic_Prediction"] = np.where(
    diagnostic_challenge_results["Diagnostic_Model"] >= 0.5,
    "Phishing",
    "Legitimate"
)

display(
    diagnostic_challenge_results.style.format({
        "Current_Model": "{:.4f}",
        "Diagnostic_Model": "{:.4f}"
    })
)

,URL,Current_Model,Diagnostic_Model,Diagnostic_Prediction
0,https://www.google.com,0.0055,0.0125,Legitimate
1,https://www.google.com/search,0.9923,0.0323,Legitimate
2,https://www.google.com/search?q=python,0.9998,0.9338,Phishing
3,https://github.com,0.0048,0.0227,Legitimate
4,https://github.com/login,0.9984,0.5960,Phishing
5,https://en.wikipedia.org/wiki/Python,1.0000,0.9994,Phishing
6,https://stackoverflow.com/questions/12345,1.0000,0.9996,Phishing
7,http://secure-login-account-verification-12345.com/login,0.9372,0.7710,Phishing
8,http://paypal-account-security-check-48291.com/verify,0.8829,0.8478,Phishing
9,http://free-prize-winner-92731.com/claim,0.8268,0.1959,Legitimate


In [270]:
# ---------------------------------------------------------
# SYSTEMATIC FEATURE ABLATION
# ---------------------------------------------------------

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

ablation_results = []

for feature_to_remove in url_features_final:

    features = [
        f for f in url_features_final
        if f != feature_to_remove
    ]

    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train[features],
        y_train
    )

    # Original untouched test set
    test_prob = model.predict_proba(
        X_test[features]
    )[:, 1]

    test_pred = (
        test_prob >= 0.5
    ).astype(int)

    # Challenge set
    challenge_prob = model.predict_proba(
        challenge_features[features]
    )[:, 1]

    ablation_results.append({
        "removed_feature": feature_to_remove,

        "accuracy": accuracy_score(
            y_test,
            test_pred
        ),

        "precision": precision_score(
            y_test,
            test_pred
        ),

        "recall": recall_score(
            y_test,
            test_pred
        ),

        "f1": f1_score(
            y_test,
            test_pred
        ),

        "roc_auc": roc_auc_score(
            y_test,
            test_prob
        ),

        "google_search": challenge_prob[1],

        "google_query": challenge_prob[2],

        "github_login": challenge_prob[4],

        "wikipedia": challenge_prob[5],

        "stackoverflow": challenge_prob[6]
    })

ablation_df = pd.DataFrame(ablation_results)

display(
    ablation_df.style.format({
        "accuracy": "{:.4f}",
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "f1": "{:.4f}",
        "roc_auc": "{:.4f}",
        "google_search": "{:.4f}",
        "google_query": "{:.4f}",
        "github_login": "{:.4f}",
        "wikipedia": "{:.4f}",
        "stackoverflow": "{:.4f}"
    })
)

,removed_feature,accuracy,precision,recall,f1,roc_auc,google_search,google_query,github_login,wikipedia,stackoverflow
0,entropy,0.9945,0.9994,0.9905,0.9949,0.9979,0.9978,0.9999,0.9977,1.0000,1.0000
1,url_length,0.9767,0.9939,0.9631,0.9783,0.9940,0.0386,0.9979,0.6174,0.9992,0.9991
2,num_www,0.9910,0.9964,0.9869,0.9917,0.9976,0.9996,1.0000,0.9924,0.9998,1.0000
3,ratio_digits_url,0.9937,0.9991,0.9893,0.9942,0.9980,0.9962,1.0000,0.9977,1.0000,1.0000
4,num_subdomains,0.9918,0.9967,0.9881,0.9924,0.9979,0.9978,1.0000,0.9947,0.9999,1.0000
5,length_hostname,0.9114,0.9867,0.8488,0.9126,0.9757,0.0467,0.9884,0.0721,0.9991,0.9998
6,num_dots,0.9938,0.9991,0.9894,0.9942,0.9980,0.9969,0.9999,0.9969,1.0000,1.0000
7,num_hyphens,0.9941,0.9993,0.9899,0.9945,0.9978,0.9976,1.0000,0.9974,1.0000,1.0000
8,ratio_digits_host,0.9938,0.9990,0.9896,0.9942,0.9979,0.9973,0.9999,0.9973,1.0000,1.0000
9,num_question_mark,0.9934,0.9990,0.9890,0.9939,0.9980,0.9963,0.9999,0.9969,1.0000,1.0000


In [271]:
# ---------------------------------------------------------
# REAL LEGITIMATE STRUCTURAL TEST
# ---------------------------------------------------------

real_legit_test_features = external_features[
    url_features_final
].reset_index(drop=True)

real_legit_prob = final_xgb_model.predict_proba(
    real_legit_test_features
)[:, 1]

real_legit_pred = (
    real_legit_prob >= 0.5
).astype(int)

print("REAL LEGITIMATE STRUCTURAL TEST")
print("--------------------------------")

print("Number of URLs:",
      len(real_legit_prob))

print(
    "Predicted Legitimate:",
    (real_legit_pred == 0).sum()
)

print(
    "Predicted Phishing:",
    (real_legit_pred == 1).sum()
)

print(
    "False Positive Rate:",
    (real_legit_pred == 1).mean()
)

print(
    "Average phishing probability:",
    real_legit_prob.mean()
)

print(
    "Median phishing probability:",
    np.median(real_legit_prob)
)

REAL LEGITIMATE STRUCTURAL TEST
--------------------------------
Number of URLs: 138
Predicted Legitimate: 0
Predicted Phishing: 138
False Positive Rate: 1.0
Average phishing probability: 0.99524057
Median phishing probability: 0.9999478


In [272]:
real_legit_results = pd.DataFrame({
    "URL": balanced_legit["URL"].values,
    "phishing_probability": real_legit_prob,
    "prediction": np.where(
        real_legit_pred == 1,
        "Phishing",
        "Legitimate"
    )
})

display(
    real_legit_results.sort_values(
        "phishing_probability",
        ascending=False
    ).head(30).style.format({
        "phishing_probability": "{:.4f}"
    })
)

,URL,phishing_probability,prediction
133,https://m.youtube.com/shorts/FU8HQNWJWLM,1.0000,Phishing
32,https://geek.digit.in/community/threads/most-frequently-visited-websites-not-what-you%E2%80%99d-expect-usa.55705/?srsltid=AfmBOoqgwTBlrRZE-NrTJFaTF_tM6QItczAsd5cKt3JOVVsru5kSE6uv,1.0000,Phishing
124,https://en.wikipedia.org/wiki/Hotcakes_(album),1.0000,Phishing
131,https://today.yougov.com/ratings/entertainment/popularity/news-websites/all,1.0000,Phishing
6,https://support.apple.com/guide/safari/go-back-to-websites-you-already-visited-ibrw1009/mac,1.0000,Phishing
7,https://support.apple.com/guide/safari/see-your-favorite-websites-sfri22092/mac,1.0000,Phishing
125,https://en.wikipedia.org/wiki/Pak_Jong-hyeon,1.0000,Phishing
14,https://bm.ge/en/news/the-worlds-most-popular-websites/85920,1.0000,Phishing
52,https://blog.hubspot.com/marketing/best-website-designs-list,1.0000,Phishing
31,https://dev.to/souviktests/most-popular-websites-1996-2020-based-on-traffic-popularity-1e1i,1.0000,Phishing


In [273]:
print("Total legitimate structural URLs:", len(real_legit_results))

print(
    "Predicted Legitimate:",
    (real_legit_results["prediction"] == "Legitimate").sum()
)

print(
    "Predicted Phishing:",
    (real_legit_results["prediction"] == "Phishing").sum()
)

print(
    "False Positive Rate:",
    (
        real_legit_results["prediction"] == "Phishing"
    ).mean()
)

print(
    "Average phishing probability:",
    real_legit_results["phishing_probability"].mean()
)

print(
    "Median phishing probability:",
    real_legit_results["phishing_probability"].median()
)

print(
    "Probability < 0.5:",
    (
        real_legit_results["phishing_probability"] < 0.5
    ).sum()
)

print(
    "Probability >= 0.5:",
    (
        real_legit_results["phishing_probability"] >= 0.5
    ).sum()
)

Total legitimate structural URLs: 138
Predicted Legitimate: 0
Predicted Phishing: 138
False Positive Rate: 1.0
Average phishing probability: 0.99524057
Median phishing probability: 0.9999477863311768
Probability < 0.5: 0
Probability >= 0.5: 138


In [274]:
import os

print(os.listdir("/content"))

['.config', 'label_encoder.pkl', 'Dataset.csv', 'feature_names.pkl', 'phishing_xgb_model.pkl', 'url_features_extracted1.csv', 'phishing_xgb_model.json', 'merged_features_final.xlsx', 'feature_descriptions.pkl', 'sample_data']


In [275]:
import pandas as pd

dataset = pd.read_csv("/content/Dataset.csv")

print("Shape:", dataset.shape)
print("\nColumns:")
print(dataset.columns.tolist())

print("\nFirst 5 rows:")
display(dataset.head())

Shape: (116600, 26)

Columns:
['url', 'url_len', 'dom', 'dom_len', 'is_ip', 'tld', 'tld_len', 'subdom_cnt', 'letter_cnt', 'digit_cnt', 'special_cnt', 'eq_cnt', 'qm_cnt', 'amp_cnt', 'dot_cnt', 'dash_cnt', 'under_cnt', 'letter_ratio', 'digit_ratio', 'spec_ratio', 'is_https', 'slash_cnt', 'entropy', 'path_len', 'query_len', 'label']

First 5 rows:


,url,url_len,dom,dom_len,is_ip,tld,tld_len,subdom_cnt,letter_cnt,digit_cnt,...,under_cnt,letter_ratio,digit_ratio,spec_ratio,is_https,slash_cnt,entropy,path_len,query_len,label
0,https://www.rmit.edu.au/,24,rmit.edu.au,11,0,edu.au,6,1,17,0,...,0,0.708333,0.0,0.291667,1,3,3.709148,1,0,0
1,http://www.latrobe.edu.au/,26,latrobe.edu.au,14,0,edu.au,6,1,19,0,...,0,0.730769,0.0,0.269231,0,3,3.738149,1,0,0
2,https://www.cqu.edu.au/,23,cqu.edu.au,10,0,edu.au,6,1,16,0,...,0,0.695652,0.0,0.304348,1,3,3.609668,1,0,0
3,http://bond.edu.au/,19,bond.edu.au,11,0,edu.au,6,0,13,0,...,0,0.684211,0.0,0.315789,0,3,3.576618,1,0,0
4,http://www.csu.edu.au/,22,csu.edu.au,10,0,edu.au,6,1,15,0,...,0,0.681818,0.0,0.318182,0,3,3.503998,1,0,0


In [276]:
print("Label counts:")
print(dataset["label"].value_counts())

print("\nLabel percentages:")
print(dataset["label"].value_counts(normalize=True) * 100)

print("\nSample URLs for each label:")

print("\nLabel 0:")
display(dataset[dataset["label"] == 0][["url", "path_len", "query_len", "label"]].head(10))

print("\nLabel 1:")
display(dataset[dataset["label"] == 1][["url", "path_len", "query_len", "label"]].head(10))

Label counts:
label
0    100000
1     16600
Name: count, dtype: int64

Label percentages:
label
0    85.763293
1    14.236707
Name: proportion, dtype: float64

Sample URLs for each label:

Label 0:


,url,path_len,query_len,label
0,https://www.rmit.edu.au/,1,0,0
1,http://www.latrobe.edu.au/,1,0,0
2,https://www.cqu.edu.au/,1,0,0
3,http://bond.edu.au/,1,0,0
4,http://www.csu.edu.au/,1,0,0
5,http://research.mater.org.au/,1,0,0
6,http://www.hri.org.au/,1,0,0
7,https://www.epipsi.gr/,1,0,0
8,http://www.riotinto.com/,1,0,0
9,http://www.arup.com/Global_locations/USA.aspx,26,0,0



Label 1:


,url,path_len,query_len,label
100000,https://pay.pagcorefeiraosetembro.site/2wq7Gr7...,16,83,1
100001,https://chillicancorne.com/,1,0,1
100002,https://chillicancorne.com/client,7,0,1
100003,https://nycmydreamx.com/,1,0,1
100004,https://nycmydreamx.com/client,7,0,1
100005,https://infernoworldofart.com/,1,0,1
100006,https://fala-clien-es.webcindario.com/,1,0,1
100007,https://infernoworldofart.com/client/,8,0,1
100008,https://acquittement-ulys.com/,1,0,1
100009,https://acquittement-ulys.com/as.php,7,0,1


In [277]:
# Check path/query structure by label

dataset["has_path"] = dataset["path_len"] > 1
dataset["has_query"] = dataset["query_len"] > 0

print("PATH DISTRIBUTION")
print(
    pd.crosstab(
        dataset["label"],
        dataset["has_path"],
        normalize="index"
    ) * 100
)

print("\nQUERY DISTRIBUTION")
print(
    pd.crosstab(
        dataset["label"],
        dataset["has_query"],
        normalize="index"
    ) * 100
)

print("\nAVERAGE PATH LENGTH")
print(dataset.groupby("label")["path_len"].mean())

print("\nAVERAGE QUERY LENGTH")
print(dataset.groupby("label")["query_len"].mean())

PATH DISTRIBUTION
has_path      False      True 
label                         
0         76.640000  23.360000
1         44.168675  55.831325

QUERY DISTRIBUTION
has_query      False      True 
label                          
0          97.998000   2.002000
1          85.451807  14.548193

AVERAGE PATH LENGTH
label
0     4.604890
1    17.153373
Name: path_len, dtype: float64

AVERAGE QUERY LENGTH
label
0    0.309240
1    9.379699
Name: query_len, dtype: float64


In [278]:
import tldextract

def get_registered_domain(url):
    ext = tldextract.extract(url)
    return ext.top_domain_under_public_suffix

# Domains in the NEW dataset
dataset["registered_domain"] = dataset["url"].apply(get_registered_domain)

# Domains in your ORIGINAL untouched test set
test_urls = df_clean.iloc[test_idx]["url"]
test_domains = set(test_urls.apply(get_registered_domain))

# Check overlap
new_dataset_domains = set(dataset["registered_domain"])

overlap_domains = new_dataset_domains.intersection(test_domains)

print("New dataset unique domains:", len(new_dataset_domains))
print("Original test domains:", len(test_domains))
print("Overlapping domains:", len(overlap_domains))

New dataset unique domains: 90846
Original test domains: 6135
Overlapping domains: 846


In [279]:
# Remove domains that overlap with the ORIGINAL test set
safe_dataset = dataset[
    ~dataset["registered_domain"].isin(test_domains)
].copy()

print("Original new dataset rows:", len(dataset))
print("Safe new dataset rows:", len(safe_dataset))

print("\nRows removed because of test-domain overlap:",
      len(dataset) - len(safe_dataset))

# Legitimate portion only
safe_legit = safe_dataset[safe_dataset["label"] == 0].copy()

print("\nSafe legitimate URLs:", len(safe_legit))

print("\nLegitimate path/query distribution:")
print(
    pd.crosstab(
        safe_legit["has_path"],
        safe_legit["has_query"]
    )
)

print("\nLegitimate URLs with path OR query:",
      ((safe_legit["has_path"]) | (safe_legit["has_query"])).sum())

print("\nUnique legitimate domains with path/query:",
      safe_legit[
          safe_legit["has_path"] | safe_legit["has_query"]
      ]["registered_domain"].nunique())

Original new dataset rows: 116600
Safe new dataset rows: 105712

Rows removed because of test-domain overlap: 10888

Safe legitimate URLs: 98830

Legitimate path/query distribution:
has_query  False  True 
has_path               
False      75446    541
True       21393   1450

Legitimate URLs with path OR query: 23384

Unique legitimate domains with path/query: 17626


In [280]:
# Look at how many path/query URLs each legitimate domain contributes

safe_legit_path_query = safe_legit[
    safe_legit["has_path"] | safe_legit["has_query"]
].copy()

domain_counts = (
    safe_legit_path_query
    .groupby("registered_domain")
    .size()
    .sort_values(ascending=False)
)

print("Number of domains:", len(domain_counts))

print("\nTop 20 domains by number of URLs:")
display(domain_counts.head(20))

print("\nDomain-count statistics:")
print(domain_counts.describe())

print("\nHow many domains have:")
print("1 URL :", (domain_counts == 1).sum())
print("2 URLs:", (domain_counts == 2).sum())
print("3 URLs:", (domain_counts == 3).sum())
print("5+ URLs:", (domain_counts >= 5).sum())
print("10+ URLs:", (domain_counts >= 10).sum())
print("50+ URLs:", (domain_counts >= 50).sum())

Number of domains: 17626

Top 20 domains by number of URLs:


,0
registered_domain,
www.nhs.uk,111
usda.gov,87
energy.gov,84
usgs.gov,70
va.gov,60
drdo.gov.in,55
www.gov.uk,54
cnr.it,47
mpg.de,45



Domain-count statistics:
count    17626.000000
mean         1.326677
std          2.276778
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max        111.000000
dtype: float64

How many domains have:
1 URL : 15976
2 URLs: 792
3 URLs: 325
5+ URLs: 387
10+ URLs: 122
50+ URLs: 7


In [281]:
import pandas as pd

# Candidate legitimate URLs containing a path or query
candidates = safe_legit[
    safe_legit["has_path"] | safe_legit["has_query"]
].copy()

# Shuffle reproducibly
candidates = candidates.sample(frac=1, random_state=42)

# Maximum 5 URLs from any one domain
selected_legit = (
    candidates
    .groupby("registered_domain", group_keys=False)
    .head(5)
    .copy()
)

# Take at most 10,000 URLs
selected_legit = selected_legit.head(10000).copy()

print("Selected URLs:", len(selected_legit))
print("Unique domains:", selected_legit["registered_domain"].nunique())

print("\nPath/query distribution:")
print(
    pd.crosstab(
        selected_legit["has_path"],
        selected_legit["has_query"]
    )
)

print("\nLabel distribution:")
print(selected_legit["label"].value_counts())

Selected URLs: 10000
Unique domains: 8616

Path/query distribution:
has_query  False  True 
has_path               
False          0    246
True        9198    556

Label distribution:
label
0    10000
Name: count, dtype: int64


In [282]:
# Convert the 10,000 selected URLs to the same 11 features
new_features = pd.DataFrame(
    selected_legit["url"].apply(extract_url_features).tolist()
)

# Make absolutely sure the columns are in the model's order
new_features = new_features[url_features_final]

# All selected URLs are legitimate
new_labels = pd.Series(
    0,
    index=new_features.index,
    name="label"
)

print("Feature shape:", new_features.shape)
print("\nFeatures:")
print(new_features.columns.tolist())

print("\nFirst 5 feature rows:")
display(new_features.head())

print("\nLabel count:")
print(new_labels.value_counts())

Feature shape: (10000, 11)

Features:
['entropy', 'url_length', 'num_www', 'ratio_digits_url', 'num_subdomains', 'length_hostname', 'num_dots', 'num_hyphens', 'ratio_digits_host', 'num_question_mark', 'num_eq']

First 5 feature rows:


,entropy,url_length,num_www,ratio_digits_url,num_subdomains,length_hostname,num_dots,num_hyphens,ratio_digits_host,num_question_mark,num_eq
0,3.739878,35,0,0.000000,0,16,1,1,0.0,0,0
1,4.162480,67,1,0.000000,1,13,2,7,0.0,0,0
2,4.336338,61,1,0.016393,1,12,3,1,0.0,0,0
3,4.080415,37,1,0.000000,1,21,3,0,0.0,0,0
4,3.989898,30,1,0.000000,1,19,2,1,0.0,0,0



Label count:
label
0    10000
Name: count, dtype: int64


In [283]:
# Test the CURRENT model on the new legitimate URLs
new_probabilities = final_xgb_model.predict_proba(new_features)[:, 1]

new_predictions = (new_probabilities >= 0.5).astype(int)

print("Total legitimate URLs:", len(new_predictions))
print("Predicted Legitimate:", (new_predictions == 0).sum())
print("Predicted Phishing:", (new_predictions == 1).sum())

print("\nFalse Positive Rate:",
      (new_predictions == 1).mean())

print("\nAverage phishing probability:",
      new_probabilities.mean())

print("Median phishing probability:",
      pd.Series(new_probabilities).median())

print("\nProbability summary:")
print(pd.Series(new_probabilities).describe())

Total legitimate URLs: 10000
Predicted Legitimate: 135
Predicted Phishing: 9865

False Positive Rate: 0.9865

Average phishing probability: 0.95394385
Median phishing probability: 0.9986234903335571

Probability summary:
count    10000.000000
mean         0.953944
std          0.107781
min          0.088588
25%          0.969808
50%          0.998623
75%          0.999854
max          0.999999
dtype: float64


In [286]:
# Prepare the original training data correctly

X_train_original = X_train[url_features_final].copy()

# Your original y_train is already encoded as 0 = legitimate, 1 = phishing
y_train_original = pd.Series(
    y_train,
    index=X_train_original.index,
    name="label"
).astype(int)

print("Original training shape:", X_train_original.shape)

print("\nOriginal training label counts:")
print(y_train_original.value_counts().sort_index())

print("\nOriginal training label percentages:")
print(
    y_train_original
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

Original training shape: (47323, 11)

Original training label counts:
label
0    22673
1    24650
Name: count, dtype: int64

Original training label percentages:
label
0    47.911164
1    52.088836
Name: proportion, dtype: float64


In [287]:
# Combine original training data + new legitimate URLs

X_combined = pd.concat(
    [
        X_train_original.reset_index(drop=True),
        new_features.reset_index(drop=True)
    ],
    ignore_index=True
)

y_combined = pd.concat(
    [
        y_train_original.reset_index(drop=True),
        new_labels.reset_index(drop=True)
    ],
    ignore_index=True
).astype(int)

# Give every example equal weight initially
sample_weights = pd.Series(
    1.0,
    index=X_combined.index
)

print("Combined training shape:", X_combined.shape)

print("\nCombined label counts:")
print(y_combined.value_counts().sort_index())

print("\nCombined label percentages:")
print(
    y_combined.value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

print("\nSample weights:")
print(sample_weights.value_counts())

Combined training shape: (57323, 11)

Combined label counts:
label
0    32673
1    24650
Name: count, dtype: int64

Combined label percentages:
label
0    56.998064
1    43.001936
Name: proportion, dtype: float64

Sample weights:
1.0    57323
Name: count, dtype: int64


In [288]:
from xgboost import XGBClassifier

corrected_xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

corrected_xgb_model.fit(
    X_combined,
    y_combined,
    sample_weight=sample_weights
)

print("Corrected XGBoost model trained successfully.")

Corrected XGBoost model trained successfully.


In [289]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Predictions on the ORIGINAL untouched test set
test_prob = corrected_xgb_model.predict_proba(X_test[url_features_final])[:, 1]
test_pred = (test_prob >= 0.5).astype(int)

print("=== ORIGINAL UNTOUCHED TEST SET ===")

print("Accuracy :", accuracy_score(y_test, test_pred))
print("Precision:", precision_score(y_test, test_pred))
print("Recall   :", recall_score(y_test, test_pred))
print("F1 Score :", f1_score(y_test, test_pred))
print("ROC-AUC  :", roc_auc_score(y_test, test_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred))

=== ORIGINAL UNTOUCHED TEST SET ===
Accuracy : 0.9490032823633016
Precision: 0.9977412068409164
Recall   : 0.9084765682385779
F1 Score : 0.9510188389081122
ROC-AUC  : 0.9946563654564726

Confusion Matrix:
[[5670   14]
 [ 623 6184]]


In [290]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

weights_to_test = [0.10, 0.25, 0.50]

results = []

for new_weight in weights_to_test:

    # Original training examples = 1.0
    # New legitimate examples = selected weight
    weights = pd.concat([
        pd.Series(1.0, index=range(len(X_train_original))),
        pd.Series(new_weight, index=range(len(new_features)))
    ]).reset_index(drop=True)

    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_combined,
        y_combined,
        sample_weight=weights
    )

    # Original untouched test
    prob = model.predict_proba(X_test[url_features_final])[:, 1]
    pred = (prob >= 0.5).astype(int)

    # New legitimate URLs
    new_prob = model.predict_proba(new_features)[:, 1]
    new_pred = (new_prob >= 0.5).astype(int)

    results.append({
        "new_weight": new_weight,
        "test_accuracy": accuracy_score(y_test, pred),
        "test_precision": precision_score(y_test, pred),
        "test_recall": recall_score(y_test, pred),
        "test_f1": f1_score(y_test, pred),
        "test_roc_auc": roc_auc_score(y_test, prob),
        "new_legit_fpr": new_pred.mean(),
        "new_legit_avg_prob": new_prob.mean()
    })

results_df = pd.DataFrame(results)

display(results_df)

,new_weight,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,new_legit_fpr,new_legit_avg_prob
0,0.10,0.984869,0.998043,0.974144,0.985949,0.997699,0.1769,0.246731
1,0.25,0.975662,0.997856,0.957397,0.977208,0.997191,0.1236,0.169491
2,0.50,0.958770,0.997470,0.926693,0.960780,0.996245,0.0864,0.123540


In [291]:
# Recreate the 0.10-weight model so we can inspect its errors

weights_010 = pd.concat([
    pd.Series(1.0, index=range(len(X_train_original))),
    pd.Series(0.10, index=range(len(new_features)))
]).reset_index(drop=True)

model_010 = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

model_010.fit(
    X_combined,
    y_combined,
    sample_weight=weights_010
)

# Predict on the 10,000 legitimate URLs
new_prob_010 = model_010.predict_proba(new_features)[:, 1]

fp_mask = new_prob_010 >= 0.5

fp_examples = selected_legit.loc[
    fp_mask,
    ["url", "path_len", "query_len", "registered_domain"]
].copy()

fp_examples["phishing_probability"] = new_prob_010[fp_mask]

print("False positives:", len(fp_examples))
print("False positive rate:", len(fp_examples) / len(selected_legit))

print("\nHighest-confidence false positives:")
display(
    fp_examples
    .sort_values("phishing_probability", ascending=False)
    .head(30)
)

False positives: 1769
False positive rate: 0.1769

Highest-confidence false positives:


,url,path_len,query_len,registered_domain,phishing_probability
97265,https://facdedroit.univ-lyon3.fr/crdms,6,0,univ-lyon3.fr,0.998835
75405,https://iasr2015.wordpress.com/english/,9,0,wordpress.com,0.998770
81182,https://physicians.columbia-stmarys.org/Clinic...,21,0,columbia-stmarys.org,0.998749
61579,https://matsuldalhumanitarianfondation.wordpre...,6,0,wordpress.com,0.998557
9519,http://imaginlab.images-et-reseaux.com/en,3,0,images-et-reseaux.com,0.998505
35415,http://heggset.rowlab.no/Hjem/Vis/Article/94/S...,39,0,rowlab.no,0.998499
15714,http://washingtonstatechinese.apps-1and1.com/e...,13,0,apps-1and1.com,0.998351
94603,http://portal-chsj.min-saude.pt/pages/2,8,0,min-saude.pt,0.998295
23308,https://enac.cl/web2018/,9,0,enac.cl,0.998186
91848,https://healthcare.ascension.org/Locations/Mic...,84,57,ascension.org,0.998152


In [292]:
import pandas as pd

importance_df = pd.DataFrame({
    "feature": url_features_final,
    "importance": model_010.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("Feature importance for the 0.10 augmented model:")
display(importance_df)

Feature importance for the 0.10 augmented model:


,feature,importance
2,num_www,0.230619
0,entropy,0.205558
1,url_length,0.154019
4,num_subdomains,0.124981
3,ratio_digits_url,0.115120
5,length_hostname,0.075624
8,ratio_digits_host,0.034212
6,num_dots,0.031424
7,num_hyphens,0.017185
10,num_eq,0.006933


In [294]:
# Compare feature values between:
# 1. legitimate URLs correctly classified by model_010
# 2. legitimate URLs incorrectly classified as phishing

fp_mask = new_prob_010 >= 0.5

correct_legit_features = new_features.loc[~fp_mask].copy()
false_positive_features = new_features.loc[fp_mask].copy()

comparison = pd.DataFrame({
    "correct_legitimate_mean": correct_legit_features.mean(),
    "false_positive_mean": false_positive_features.mean()
})

comparison["difference"] = (
    comparison["false_positive_mean"]
    - comparison["correct_legitimate_mean"]
)

display(comparison)

,correct_legitimate_mean,false_positive_mean,difference
entropy,3.975666,4.017889,0.042224
url_length,40.341392,41.310910,0.969518
num_www,0.970356,0.198982,-0.771373
ratio_digits_url,0.005630,0.017161,0.011531
num_subdomains,1.029644,0.538722,-0.490922
length_hostname,16.394849,16.305257,-0.089592
num_dots,2.693233,2.074053,-0.619180
num_hyphens,0.610132,0.624081,0.013949
ratio_digits_host,0.001384,0.004111,0.002727
num_question_mark,0.078241,0.089316,0.011075


In [295]:
import shap
import numpy as np
import pandas as pd

# Create SHAP explainer for the 0.10 model
explainer_010 = shap.TreeExplainer(model_010)

# Calculate SHAP values for the 10,000 legitimate URLs
shap_values = explainer_010.shap_values(new_features)

# Separate correctly classified legitimate URLs and false positives
correct_mask = new_prob_010 < 0.5
fp_mask = new_prob_010 >= 0.5

# Average SHAP contribution for each group
shap_comparison = pd.DataFrame({
    "feature": url_features_final,
    "correct_legitimate_mean_shap": shap_values[correct_mask].mean(axis=0),
    "false_positive_mean_shap": shap_values[fp_mask].mean(axis=0)
})

shap_comparison["difference"] = (
    shap_comparison["false_positive_mean_shap"]
    - shap_comparison["correct_legitimate_mean_shap"]
)

shap_comparison["abs_difference"] = (
    shap_comparison["difference"].abs()
)

shap_comparison = shap_comparison.sort_values(
    "abs_difference",
    ascending=False
)

display(shap_comparison)

,feature,correct_legitimate_mean_shap,false_positive_mean_shap,difference,abs_difference
2,num_www,-2.818604,-0.177010,2.641595,2.641595
5,length_hostname,0.118427,0.714650,0.596222,0.596222
1,url_length,1.025501,1.554471,0.528970,0.528970
6,num_dots,-0.589445,-0.199632,0.389813,0.389813
3,ratio_digits_url,-0.354382,-0.216168,0.138213,0.138213
0,entropy,-0.090498,0.031567,0.122065,0.122065
4,num_subdomains,0.376567,0.258205,-0.118362,0.118362
7,num_hyphens,-0.075772,-0.007769,0.068002,0.068002
8,ratio_digits_host,-0.010859,0.014515,0.025373,0.025373
9,num_question_mark,0.004240,0.008763,0.004523,0.004523


In [296]:
# Compare www usage in the original training data
# versus the new legitimate URLs

original_www = X_train_original["num_www"]
new_www = new_features["num_www"]

print("ORIGINAL TRAINING DATA")
print(original_www.value_counts().sort_index())
print("\nPercentage:")
print(
    original_www.value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

print("\nNEW LEGITIMATE DATA")
print(new_www.value_counts().sort_index())
print("\nPercentage:")
print(
    new_www.value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

print("\nORIGINAL TRAINING — www by class")
original_www_class = pd.DataFrame({
    "label": y_train_original.values,
    "num_www": original_www.values
})

print(
    pd.crosstab(
        original_www_class["label"],
        original_www_class["num_www"],
        normalize="index"
    ).mul(100)
)

ORIGINAL TRAINING DATA
num_www
0    42996
1     4307
2       20
Name: count, dtype: int64

Percentage:
num_www
0    90.856455
1     9.101283
2     0.042263
Name: proportion, dtype: float64

NEW LEGITIMATE DATA
num_www
0    1679
1    8303
2      18
Name: count, dtype: int64

Percentage:
num_www
0    16.79
1    83.03
2     0.18
Name: proportion, dtype: float64

ORIGINAL TRAINING — www by class
num_www          0          1         2
label                                  
0        84.016231  15.983769  0.000000
1        97.148073   2.770791  0.081136


In [297]:
# Legitimate path/query URLs WITHOUT "www"
non_www_candidates = safe_legit[
    (safe_legit["has_path"] | safe_legit["has_query"]) &
    (safe_legit["url"].str.lower().str.count("www") == 0)
].copy()

# Legitimate path/query URLs WITH "www"
www_candidates = safe_legit[
    (safe_legit["has_path"] | safe_legit["has_query"]) &
    (safe_legit["url"].str.lower().str.count("www") > 0)
].copy()

print("Non-www legitimate path/query URLs:",
      len(non_www_candidates))

print("Non-www unique domains:",
      non_www_candidates["registered_domain"].nunique())

print("\nWWW legitimate path/query URLs:",
      len(www_candidates))

print("WWW unique domains:",
      www_candidates["registered_domain"].nunique())


Non-www legitimate path/query URLs: 3847
Non-www unique domains: 3039

WWW legitimate path/query URLs: 19537
WWW unique domains: 14841


In [298]:
# Shuffle both groups reproducibly
non_www_candidates = non_www_candidates.sample(
    frac=1,
    random_state=42
)

www_candidates = www_candidates.sample(
    frac=1,
    random_state=42
)

# Maximum 1 URL per domain
non_www_selected = (
    non_www_candidates
    .groupby("registered_domain", group_keys=False)
    .head(1)
    .head(3000)
    .copy()
)

www_selected = (
    www_candidates
    .groupby("registered_domain", group_keys=False)
    .head(1)
    .head(3000)
    .copy()
)

balanced_legit = pd.concat(
    [non_www_selected, www_selected],
    ignore_index=True
)

print("Total selected:", len(balanced_legit))

print("\nUnique domains:",
      balanced_legit["registered_domain"].nunique())

print("\nWWW distribution:")
print(
    balanced_legit["url"]
    .str.lower()
    .str.count("www")
    .value_counts()
)

print("\nPath/query distribution:")
print(
    pd.crosstab(
        balanced_legit["has_path"],
        balanced_legit["has_query"]
    )
)

Total selected: 6000

Unique domains: 5903

WWW distribution:
url
0    3000
1    2990
2      10
Name: count, dtype: int64

Path/query distribution:
has_query  False  True 
has_path               
False          0    202
True        5484    314


In [299]:
# Convert the balanced legitimate URLs
# to the exact 11 features used by the production model

balanced_features = pd.DataFrame(
    balanced_legit["url"]
    .apply(extract_url_features)
    .tolist()
)

balanced_features = balanced_features[url_features_final]

balanced_labels = pd.Series(
    0,
    index=balanced_features.index,
    name="label"
)

print("Feature shape:", balanced_features.shape)

print("\nFeatures:")
print(balanced_features.columns.tolist())

print("\nLabel count:")
print(balanced_labels.value_counts())

print("\nFirst 5 rows:")
display(balanced_features.head())

Feature shape: (6000, 11)

Features:
['entropy', 'url_length', 'num_www', 'ratio_digits_url', 'num_subdomains', 'length_hostname', 'num_dots', 'num_hyphens', 'ratio_digits_host', 'num_question_mark', 'num_eq']

Label count:
label
0    6000
Name: count, dtype: int64

First 5 rows:


,entropy,url_length,num_www,ratio_digits_url,num_subdomains,length_hostname,num_dots,num_hyphens,ratio_digits_host,num_question_mark,num_eq
0,3.943465,25,0,0.000000,0,7,1,1,0.0,0,0
1,4.492014,99,0,0.060606,1,28,2,4,0.0,0,0
2,3.516028,22,0,0.000000,0,11,1,0,0.0,0,0
3,4.030639,32,0,0.000000,0,13,2,0,0.0,0,0
4,3.893661,25,0,0.000000,0,14,1,0,0.0,0,0


In [300]:
# Combine original training data with balanced legitimate augmentation

X_balanced_combined = pd.concat(
    [
        X_train_original.reset_index(drop=True),
        balanced_features.reset_index(drop=True)
    ],
    ignore_index=True
)

y_balanced_combined = pd.concat(
    [
        y_train_original.reset_index(drop=True),
        balanced_labels.reset_index(drop=True)
    ],
    ignore_index=True
).astype(int)

# Original data = 1.0
# New balanced legitimate data = 0.10
balanced_weights = pd.concat([
    pd.Series(1.0, index=range(len(X_train_original))),
    pd.Series(0.10, index=range(len(balanced_features)))
]).reset_index(drop=True)

print("Combined shape:", X_balanced_combined.shape)

print("\nLabels:")
print(y_balanced_combined.value_counts().sort_index())

print("\nWeights:")
print(balanced_weights.value_counts())

Combined shape: (53323, 11)

Labels:
label
0    28673
1    24650
Name: count, dtype: int64

Weights:
1.0    47323
0.1     6000
Name: count, dtype: int64


In [301]:
from xgboost import XGBClassifier

balanced_xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

balanced_xgb_model.fit(
    X_balanced_combined,
    y_balanced_combined,
    sample_weight=balanced_weights
)

print("Balanced augmentation XGBoost trained successfully.")

Balanced augmentation XGBoost trained successfully.


In [302]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Original untouched test set
balanced_test_prob = balanced_xgb_model.predict_proba(
    X_test[url_features_final]
)[:, 1]

balanced_test_pred = (
    balanced_test_prob >= 0.5
).astype(int)

print("=== BALANCED AUGMENTATION MODEL ===")
print("=== ORIGINAL UNTOUCHED TEST SET ===")

print("Accuracy :", accuracy_score(y_test, balanced_test_pred))
print("Precision:", precision_score(y_test, balanced_test_pred))
print("Recall   :", recall_score(y_test, balanced_test_pred))
print("F1 Score :", f1_score(y_test, balanced_test_pred))
print("ROC-AUC  :", roc_auc_score(y_test, balanced_test_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, balanced_test_pred))

=== BALANCED AUGMENTATION MODEL ===
=== ORIGINAL UNTOUCHED TEST SET ===
Accuracy : 0.9843887599071331
Precision: 0.9983418751884233
Recall   : 0.9729690024974291
F1 Score : 0.9854921508816308
ROC-AUC  : 0.9978130953905854

Confusion Matrix:
[[5673   11]
 [ 184 6623]]


In [303]:
# Test the balanced model on the 6,000 NEW legitimate URLs

balanced_legit_prob = balanced_xgb_model.predict_proba(
    balanced_features
)[:, 1]

balanced_legit_pred = (
    balanced_legit_prob >= 0.5
).astype(int)

print("=== NEW BALANCED LEGITIMATE SET ===")

print("Total legitimate URLs:", len(balanced_legit_pred))
print("Predicted Legitimate:",
      (balanced_legit_pred == 0).sum())
print("Predicted Phishing:",
      (balanced_legit_pred == 1).sum())

print("\nFalse Positive Rate:",
      balanced_legit_pred.mean())

print("\nAverage phishing probability:",
      balanced_legit_prob.mean())

print("Median phishing probability:",
      pd.Series(balanced_legit_prob).median())

print("\nProbability summary:")
print(pd.Series(balanced_legit_prob).describe())

=== NEW BALANCED LEGITIMATE SET ===
Total legitimate URLs: 6000
Predicted Legitimate: 3467
Predicted Phishing: 2533

False Positive Rate: 0.4221666666666667

Average phishing probability: 0.44544557
Median phishing probability: 0.38669008016586304

Probability summary:
count    6000.000000
mean        0.445446
std         0.325502
min         0.004081
25%         0.136015
50%         0.386690
75%         0.747624
max         0.998991
dtype: float64


In [304]:
balanced_results = balanced_legit[
    ["url", "registered_domain", "has_path", "has_query"]
].copy()

balanced_results["phishing_probability"] = balanced_legit_prob
balanced_results["prediction"] = np.where(
    balanced_results["phishing_probability"] >= 0.5,
    "Phishing",
    "Legitimate"
)

false_positives = balanced_results[
    balanced_results["prediction"] == "Phishing"
].copy()

print("False positives:", len(false_positives))

print("\n=== False positives by URL structure ===")
print(
    pd.crosstab(
        false_positives["has_path"],
        false_positives["has_query"]
    )
)

print("\n=== All legitimate URLs by URL structure ===")
print(
    pd.crosstab(
        balanced_results["has_path"],
        balanced_results["has_query"]
    )
)

print("\n=== Highest-confidence false positives ===")
display(
    false_positives[
        ["url", "phishing_probability"]
    ]
    .sort_values("phishing_probability", ascending=False)
    .head(20)
)

False positives: 2533

=== False positives by URL structure ===
has_query  False  True 
has_path               
False          0     94
True        2303    136

=== All legitimate URLs by URL structure ===
has_query  False  True 
has_path               
False          0    202
True        5484    314

=== Highest-confidence false positives ===


,url,phishing_probability
2335,https://physicians.columbia-stmarys.org/Clinic...,0.998991
629,https://healthcare.ascension.org/Locations/Mic...,0.998676
1756,http://cpanel33.proisp.no/~geneshxc/,0.998508
224,http://imaginlab.images-et-reseaux.com/en,0.998385
921,http://ptc10.jrscomware.com/nisigaoka/,0.998359
2891,https://facdedroit.univ-lyon3.fr/crdms,0.998333
171,https://sites-recherche.univ-rennes2.fr/ace/,0.998236
1208,https://nwcancerclinic.21co.com/local,0.998172
2680,http://washingtonstatechinese.apps-1and1.com/e...,0.998134
2525,http://procareodessa.com/?lipi=urn%3Ali%3Apage...,0.998071


In [305]:
from urllib.parse import urlparse
import re
import numpy as np
import pandas as pd


def extract_path_query_features(url):
    parsed = urlparse(url)

    path = parsed.path or ""
    query = parsed.query or ""

    # Remove leading/trailing slashes for tokenization
    path_clean = path.strip("/")

    # Path tokens
    path_tokens = [
        token for token in re.split(r"[/_\-]+", path_clean)
        if token
    ]

    # Query parameters
    query_params = [
        token for token in query.split("&")
        if token
    ]

    # File extension
    last_path_part = path_clean.split("/")[-1] if path_clean else ""

    if "." in last_path_part:
        extension = last_path_part.rsplit(".", 1)[-1].lower()
    else:
        extension = ""

    return {
        "path_length": len(path),
        "query_length": len(query),
        "path_depth": len([p for p in path.split("/") if p]),
        "path_token_count": len(path_tokens),
        "path_digit_count": sum(c.isdigit() for c in path),
        "path_letter_count": sum(c.isalpha() for c in path),
        "path_special_count": sum(
            not c.isalnum() and c not in "/"
            for c in path
        ),
        "query_param_count": len(query_params),
        "query_digit_count": sum(c.isdigit() for c in query),
        "query_special_count": sum(
            not c.isalnum() and c not in "&"
            for c in query
        ),
        "has_file_extension": int(bool(extension)),
        "path_has_login": int(
            any(
                word in path.lower()
                for word in [
                    "login",
                    "signin",
                    "sign-in",
                    "authenticate",
                    "authentication"
                ]
            )
        ),
        "path_has_account": int(
            any(
                word in path.lower()
                for word in [
                    "account",
                    "verify",
                    "verification",
                    "secure"
                ]
            )
        )
    }

In [306]:
balanced_path_features = pd.DataFrame(
    balanced_legit["url"]
    .apply(extract_path_query_features)
    .tolist()
)

print("Shape:", balanced_path_features.shape)

display(
    balanced_path_features.describe().T
)

Shape: (6000, 13)


,count,mean,std,min,25%,50%,75%,max
path_length,6000.0,13.630667,14.652498,1.0,4.0,10.0,17.0,220.0
query_length,6000.0,1.229833,5.837324,0.0,0.0,0.0,0.0,116.0
path_depth,6000.0,1.485000,0.939285,0.0,1.0,1.0,2.0,11.0
path_token_count,6000.0,1.920667,1.755435,0.0,1.0,1.0,2.0,21.0
path_digit_count,6000.0,0.223667,1.518889,0.0,0.0,0.0,0.0,56.0
path_letter_count,6000.0,10.639167,12.539983,0.0,2.0,7.0,13.0,168.0
path_special_count,6000.0,0.803167,1.511998,0.0,0.0,0.0,1.0,44.0
query_param_count,6000.0,0.111333,0.417094,0.0,0.0,0.0,0.0,5.0
query_digit_count,6000.0,0.190833,1.513098,0.0,0.0,0.0,0.0,48.0
query_special_count,6000.0,0.159167,0.719897,0.0,0.0,0.0,0.0,14.0


In [308]:
balanced_diagnostic = balanced_results.copy()

balanced_diagnostic = pd.concat(
    [
        balanced_diagnostic.reset_index(drop=True),
        balanced_path_features.reset_index(drop=True)
    ],
    axis=1
)

print("=== FALSE POSITIVE FEATURE MEANS ===")

display(
    balanced_diagnostic[
        balanced_diagnostic["prediction"] == "Phishing"
    ][balanced_path_features.columns]
    .mean()
)

print("\n=== CORRECT LEGITIMATE FEATURE MEANS ===")

display(
    balanced_diagnostic[
        balanced_diagnostic["prediction"] == "Legitimate"
    ][balanced_path_features.columns]
    .mean()
)

=== FALSE POSITIVE FEATURE MEANS ===


,0
path_length,14.739045
query_length,1.673904
path_depth,1.559811
path_token_count,1.998816
path_digit_count,0.393210
path_letter_count,11.466640
path_special_count,0.808922
query_param_count,0.126332
query_digit_count,0.289775
query_special_count,0.210028



=== CORRECT LEGITIMATE FEATURE MEANS ===


,0
path_length,12.820883
query_length,0.905394
path_depth,1.430343
path_token_count,1.863571
path_digit_count,0.099798
path_letter_count,10.034612
path_special_count,0.798962
query_param_count,0.100375
query_digit_count,0.118546
query_special_count,0.122007


In [309]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Use the ORIGINAL training URLs
train_urls = df_clean.iloc[train_idx]["url"].astype(str)

# Original test URLs
test_urls = df_clean.iloc[test_idx]["url"].astype(str)

# Corresponding labels
train_labels = y_train_original.values
test_labels = np.asarray(y_test).astype(int)

# Character-level TF-IDF
tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=3,
    max_features=50000,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(train_urls)
X_test_tfidf = tfidf.transform(test_urls)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

tfidf_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

tfidf_model.fit(
    X_train_tfidf,
    train_labels
)

tfidf_test_prob = tfidf_model.predict_proba(
    X_test_tfidf
)[:, 1]

tfidf_test_pred = (
    tfidf_test_prob >= 0.5
).astype(int)

print("\n=== TF-IDF URL MODEL ===")
print("Accuracy :", accuracy_score(test_labels, tfidf_test_pred))
print("Precision:", precision_score(test_labels, tfidf_test_pred))
print("Recall   :", recall_score(test_labels, tfidf_test_pred))
print("F1 Score :", f1_score(test_labels, tfidf_test_pred))
print("ROC-AUC  :", roc_auc_score(test_labels, tfidf_test_prob))

Train TF-IDF shape: (47323, 50000)
Test TF-IDF shape: (12491, 50000)

=== TF-IDF URL MODEL ===
Accuracy : 0.9598110639660555
Precision: 0.9996829925503249
Recall   : 0.926546202438666
F1 Score : 0.9617261360170783
ROC-AUC  : 0.9972445779880317


In [311]:
# Transform the 6,000 realistic legitimate URLs
balanced_legit_urls = balanced_legit["url"].astype(str)

X_balanced_legit_tfidf = tfidf.transform(
    balanced_legit_urls
)

balanced_legit_tfidf_prob = tfidf_model.predict_proba(
    X_balanced_legit_tfidf
)[:, 1]

balanced_legit_tfidf_pred = (
    balanced_legit_tfidf_prob >= 0.5
).astype(int)

print("=== TF-IDF MODEL — LEGITIMATE CHALLENGE ===")

print("Total:", len(balanced_legit_tfidf_pred))

print(
    "Predicted Legitimate:",
    (balanced_legit_tfidf_pred == 0).sum()
)

print(
    "Predicted Phishing:",
    (balanced_legit_tfidf_pred == 1).sum()
)

print(
    "\nFalse Positive Rate:",
    balanced_legit_tfidf_pred.mean()
)

print(
    "Average phishing probability:",
    balanced_legit_tfidf_prob.mean()
)

print(
    "Median phishing probability:",
    np.median(balanced_legit_tfidf_prob)
)

print("\nProbability summary:")
print(pd.Series(balanced_legit_tfidf_prob).describe())

=== TF-IDF MODEL — LEGITIMATE CHALLENGE ===
Total: 6000
Predicted Legitimate: 901
Predicted Phishing: 5099

False Positive Rate: 0.8498333333333333
Average phishing probability: 0.8157128841369058
Median phishing probability: 0.9415229714272595

Probability summary:
count    6000.000000
mean        0.815713
std         0.263599
min         0.006427
25%         0.799636
50%         0.941523
75%         0.980571
max         0.999876
dtype: float64


In [312]:
# ============================================================
# FINAL MODEL REBUILD — STEP 1
# Load and inspect URL-Phish v2
# ============================================================

import pandas as pd
import numpy as np
import tldextract

# Load dataset
dataset = pd.read_csv("/content/Dataset.csv")

print("Dataset shape:", dataset.shape)

print("\nColumns:")
print(dataset.columns.tolist())

print("\nLabel distribution:")
print(dataset["label"].value_counts().sort_index())

print("\nLabel percentages:")
print(
    dataset["label"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("\nMissing values:")
print(dataset.isnull().sum())

print("\nFirst 5 rows:")
display(dataset.head())

Dataset shape: (116600, 26)

Columns:
['url', 'url_len', 'dom', 'dom_len', 'is_ip', 'tld', 'tld_len', 'subdom_cnt', 'letter_cnt', 'digit_cnt', 'special_cnt', 'eq_cnt', 'qm_cnt', 'amp_cnt', 'dot_cnt', 'dash_cnt', 'under_cnt', 'letter_ratio', 'digit_ratio', 'spec_ratio', 'is_https', 'slash_cnt', 'entropy', 'path_len', 'query_len', 'label']

Label distribution:
label
0    100000
1     16600
Name: count, dtype: int64

Label percentages:
label
0    85.76
1    14.24
Name: proportion, dtype: float64

Missing values:
url              0
url_len          0
dom              0
dom_len          0
is_ip            0
tld             14
tld_len          0
subdom_cnt       0
letter_cnt       0
digit_cnt        0
special_cnt      0
eq_cnt           0
qm_cnt           0
amp_cnt          0
dot_cnt          0
dash_cnt         0
under_cnt        0
letter_ratio     0
digit_ratio      0
spec_ratio       0
is_https         0
slash_cnt        0
entropy          0
path_len         0
query_len        0
label     

,url,url_len,dom,dom_len,is_ip,tld,tld_len,subdom_cnt,letter_cnt,digit_cnt,...,under_cnt,letter_ratio,digit_ratio,spec_ratio,is_https,slash_cnt,entropy,path_len,query_len,label
0,https://www.rmit.edu.au/,24,rmit.edu.au,11,0,edu.au,6,1,17,0,...,0,0.708333,0.0,0.291667,1,3,3.709148,1,0,0
1,http://www.latrobe.edu.au/,26,latrobe.edu.au,14,0,edu.au,6,1,19,0,...,0,0.730769,0.0,0.269231,0,3,3.738149,1,0,0
2,https://www.cqu.edu.au/,23,cqu.edu.au,10,0,edu.au,6,1,16,0,...,0,0.695652,0.0,0.304348,1,3,3.609668,1,0,0
3,http://bond.edu.au/,19,bond.edu.au,11,0,edu.au,6,0,13,0,...,0,0.684211,0.0,0.315789,0,3,3.576618,1,0,0
4,http://www.csu.edu.au/,22,csu.edu.au,10,0,edu.au,6,1,15,0,...,0,0.681818,0.0,0.318182,0,3,3.503998,1,0,0


In [313]:
# ============================================================
# FINAL MODEL REBUILD — STEP 2
# Registered-domain overlap removal
# ============================================================

def get_registered_domain(url):
    ext = tldextract.extract(str(url))
    return ext.top_domain_under_public_suffix


# Get registered domains from the ORIGINAL untouched test set
original_test_urls = df_clean.iloc[test_idx]["url"].astype(str)

original_test_domains = set(
    original_test_urls.apply(get_registered_domain)
)

# Remove empty values just in case
original_test_domains.discard("")

print("Original test registered domains:",
      len(original_test_domains))


# Get registered domains in URL-Phish v2
dataset["registered_domain"] = (
    dataset["url"]
    .apply(get_registered_domain)
)

print(
    "URL-Phish v2 unique registered domains:",
    dataset["registered_domain"].nunique()
)


# Find overlap
overlapping_domains = (
    set(dataset["registered_domain"])
    & original_test_domains
)

print(
    "Overlapping registered domains:",
    len(overlapping_domains)
)


# Remove overlapping domains
safe_dataset = dataset[
    ~dataset["registered_domain"].isin(original_test_domains)
].copy()

print("\nOriginal URL-Phish v2 rows:", len(dataset))
print("Removed rows:", len(dataset) - len(safe_dataset))
print("Safe dataset rows:", len(safe_dataset))

print("\nSafe dataset labels:")
print(
    safe_dataset["label"]
    .value_counts()
    .sort_index()
)

print("\nSafe dataset unique domains:",
      safe_dataset["registered_domain"].nunique())

Original test registered domains: 6134
URL-Phish v2 unique registered domains: 90846
Overlapping registered domains: 845

Original URL-Phish v2 rows: 116600
Removed rows: 10872
Safe dataset rows: 105728

Safe dataset labels:
label
0    98839
1     6889
Name: count, dtype: int64

Safe dataset unique domains: 90001


In [314]:
# ============================================================
# FINAL MODEL REBUILD — STEP 3
# Domain-separated train/test split
# ============================================================

from sklearn.model_selection import train_test_split

# Unique registered domains
unique_domains = safe_dataset["registered_domain"].unique()

print("Total unique domains:", len(unique_domains))

# 80% domains → training
# 20% domains → testing
train_domains, test_domains_new = train_test_split(
    unique_domains,
    test_size=0.20,
    random_state=42
)

print("Training domains:", len(train_domains))
print("Testing domains:", len(test_domains_new))

# Create sets for fast lookup
train_domain_set = set(train_domains)
test_domain_set = set(test_domains_new)

# Split URLs according to their registered domain
new_train_df = safe_dataset[
    safe_dataset["registered_domain"].isin(train_domain_set)
].copy()

new_test_df = safe_dataset[
    safe_dataset["registered_domain"].isin(test_domain_set)
].copy()

print("\nTraining rows:", len(new_train_df))
print("Testing rows:", len(new_test_df))

print("\nTraining labels:")
print(
    new_train_df["label"]
    .value_counts()
    .sort_index()
)

print("\nTesting labels:")
print(
    new_test_df["label"]
    .value_counts()
    .sort_index()
)

# Verify there is absolutely no domain overlap
train_domain_check = set(new_train_df["registered_domain"])
test_domain_check = set(new_test_df["registered_domain"])

overlap_check = train_domain_check & test_domain_check

print("\nDomain overlap:", len(overlap_check))

Total unique domains: 90001
Training domains: 72000
Testing domains: 18001

Training rows: 84890
Testing rows: 20838

Training labels:
label
0    79106
1     5784
Name: count, dtype: int64

Testing labels:
label
0    19733
1     1105
Name: count, dtype: int64

Domain overlap: 0


In [315]:
# ============================================================
# FINAL MODEL REBUILD — STEP 4
# Check path/query distribution
# ============================================================

def show_url_structure(df, name):
    temp = df.copy()

    temp["has_path"] = temp["path_len"] > 1
    temp["has_query"] = temp["query_len"] > 0

    print(f"\n{'=' * 60}")
    print(name)
    print(f"{'=' * 60}")

    print("\nTotal URLs:", len(temp))

    print("\nPath distribution:")
    print(
        temp.groupby("label")["has_path"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )

    print("\nQuery distribution:")
    print(
        temp.groupby("label")["has_query"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )

    print("\nAverage path length:")
    print(
        temp.groupby("label")["path_len"]
        .mean()
    )

    print("\nAverage query length:")
    print(
        temp.groupby("label")["query_len"]
        .mean()
    )


show_url_structure(new_train_df, "NEW TRAINING SET")
show_url_structure(new_test_df, "NEW TEST SET")


NEW TRAINING SET

Total URLs: 84890

Path distribution:
label  has_path
0      False       76.91
       True        23.09
1      True        71.72
       False       28.28
Name: proportion, dtype: float64

Query distribution:
label  has_query
0      False        97.95
       True          2.05
1      False        91.84
       True          8.16
Name: proportion, dtype: float64

Average path length:
label
0     4.554294
1    13.197441
Name: path_len, dtype: float64

Average query length:
label
0    0.314022
1    7.131224
Name: query_len, dtype: float64

NEW TEST SET

Total URLs: 20838

Path distribution:
label  has_path
0      False       76.77
       True        23.23
1      True        68.14
       False       31.86
Name: proportion, dtype: float64

Query distribution:
label  has_query
0      False        98.11
       True          1.89
1      False        90.23
       True          9.77
Name: proportion, dtype: float64

Average path length:
label
0     4.482947
1    14.098643
Name: 

In [316]:
# ============================================================
# FINAL MODEL REBUILD — STEP 5
# Extract the exact 11 production features
# ============================================================

# Extract features from NEW TRAINING URLs
X_new_train = pd.DataFrame(
    new_train_df["url"]
    .apply(extract_url_features)
    .tolist()
)

# Keep exactly the production feature order
X_new_train = X_new_train[url_features_final]

# Extract features from NEW TEST URLs
X_new_test = pd.DataFrame(
    new_test_df["url"]
    .apply(extract_url_features)
    .tolist()
)

# Keep exactly the production feature order
X_new_test = X_new_test[url_features_final]

# Labels
y_new_train = new_train_df["label"].astype(int).reset_index(drop=True)
y_new_test = new_test_df["label"].astype(int).reset_index(drop=True)

# Reset feature indexes
X_new_train = X_new_train.reset_index(drop=True)
X_new_test = X_new_test.reset_index(drop=True)

print("Training feature shape:", X_new_train.shape)
print("Testing feature shape :", X_new_test.shape)

print("\nFeatures:")
print(X_new_train.columns.tolist())

print("\nTraining labels:")
print(y_new_train.value_counts().sort_index())

print("\nTesting labels:")
print(y_new_test.value_counts().sort_index())

print("\nMissing values in training:")
print(X_new_train.isnull().sum().sum())

print("\nMissing values in testing:")
print(X_new_test.isnull().sum().sum())

Training feature shape: (84890, 11)
Testing feature shape : (20838, 11)

Features:
['entropy', 'url_length', 'num_www', 'ratio_digits_url', 'num_subdomains', 'length_hostname', 'num_dots', 'num_hyphens', 'ratio_digits_host', 'num_question_mark', 'num_eq']

Training labels:
label
0    79106
1     5784
Name: count, dtype: int64

Testing labels:
label
0    19733
1     1105
Name: count, dtype: int64

Missing values in training:
0

Missing values in testing:
0


In [317]:
# ============================================================
# FINAL MODEL REBUILD — STEP 6
# Train XGBoost on the new domain-separated dataset
# ============================================================

from xgboost import XGBClassifier

final_rebuilt_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

final_rebuilt_xgb.fit(
    X_new_train,
    y_new_train
)

print("Final rebuilt XGBoost trained successfully.")

Final rebuilt XGBoost trained successfully.


In [318]:
# ============================================================
# FINAL MODEL REBUILD — STEP 7
# Evaluate on completely unseen domains
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Predictions
rebuilt_test_prob = final_rebuilt_xgb.predict_proba(
    X_new_test
)[:, 1]

rebuilt_test_pred = (
    rebuilt_test_prob >= 0.5
).astype(int)

# Metrics
accuracy = accuracy_score(
    y_new_test,
    rebuilt_test_pred
)

precision = precision_score(
    y_new_test,
    rebuilt_test_pred
)

recall = recall_score(
    y_new_test,
    rebuilt_test_pred
)

f1 = f1_score(
    y_new_test,
    rebuilt_test_pred
)

roc_auc = roc_auc_score(
    y_new_test,
    rebuilt_test_prob
)

print("==============================================")
print("FINAL REBUILT XGBOOST")
print("UNSEEN-DOMAIN TEST SET")
print("==============================================")

print(f"Accuracy : {accuracy:.6f}")
print(f"Precision: {precision:.6f}")
print(f"Recall   : {recall:.6f}")
print(f"F1 Score : {f1:.6f}")
print(f"ROC-AUC  : {roc_auc:.6f}")

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_new_test,
    rebuilt_test_pred
))

print("\nClassification Report:")
print(
    classification_report(
        y_new_test,
        rebuilt_test_pred,
        target_names=["Legitimate", "Phishing"]
    )
)

FINAL REBUILT XGBOOST
UNSEEN-DOMAIN TEST SET
Accuracy : 0.973654
Precision: 0.814480
Recall   : 0.651584
F1 Score : 0.723982
ROC-AUC  : 0.975383

Confusion Matrix:
[[19569   164]
 [  385   720]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate       0.98      0.99      0.99     19733
    Phishing       0.81      0.65      0.72      1105

    accuracy                           0.97     20838
   macro avg       0.90      0.82      0.86     20838
weighted avg       0.97      0.97      0.97     20838



In [319]:
# ============================================================
# FINAL MODEL REBUILD — STEP 8
# Class-weighted XGBoost
# ============================================================

from xgboost import XGBClassifier

weighted_rebuilt_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=3
)

weighted_rebuilt_xgb.fit(
    X_new_train,
    y_new_train
)

print("Class-weighted XGBoost trained successfully.")

Class-weighted XGBoost trained successfully.


In [320]:
# ============================================================
# FINAL MODEL REBUILD — STEP 9
# Evaluate class-weighted XGBoost
# ============================================================

weighted_test_prob = weighted_rebuilt_xgb.predict_proba(
    X_new_test
)[:, 1]

weighted_test_pred = (
    weighted_test_prob >= 0.5
).astype(int)

print("==============================================")
print("CLASS-WEIGHTED REBUILT XGBOOST")
print("UNSEEN-DOMAIN TEST SET")
print("==============================================")

print(
    f"Accuracy : {accuracy_score(y_new_test, weighted_test_pred):.6f}"
)

print(
    f"Precision: {precision_score(y_new_test, weighted_test_pred):.6f}"
)

print(
    f"Recall   : {recall_score(y_new_test, weighted_test_pred):.6f}"
)

print(
    f"F1 Score : {f1_score(y_new_test, weighted_test_pred):.6f}"
)

print(
    f"ROC-AUC  : {roc_auc_score(y_new_test, weighted_test_prob):.6f}"
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_new_test,
        weighted_test_pred
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_new_test,
        weighted_test_pred,
        target_names=["Legitimate", "Phishing"]
    )
)

CLASS-WEIGHTED REBUILT XGBOOST
UNSEEN-DOMAIN TEST SET
Accuracy : 0.970487
Precision: 0.691108
Recall   : 0.801810
F1 Score : 0.742354
ROC-AUC  : 0.976178

Confusion Matrix:
[[19337   396]
 [  219   886]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate       0.99      0.98      0.98     19733
    Phishing       0.69      0.80      0.74      1105

    accuracy                           0.97     20838
   macro avg       0.84      0.89      0.86     20838
weighted avg       0.97      0.97      0.97     20838



In [321]:
# ============================================================
# FINAL MODEL REBUILD — STEP 10
# Threshold analysis
# ============================================================

thresholds = [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80
]

threshold_results = []

for threshold in thresholds:

    pred = (
        weighted_test_prob >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_new_test,
        pred
    ).ravel()

    threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy_score(
            y_new_test, pred
        ),
        "precision": precision_score(
            y_new_test, pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_new_test, pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_new_test, pred,
            zero_division=0
        ),
        "false_positive_rate": fp / (fp + tn),
        "false_negative_rate": fn / (fn + tp),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    })

threshold_df = pd.DataFrame(threshold_results)

display(
    threshold_df.round(4)
)

,threshold,accuracy,precision,recall,f1,false_positive_rate,false_negative_rate,TP,FP,FN,TN
0,0.10,0.9190,0.3892,0.9249,0.5478,0.0813,0.0751,1022,1604,83,18129
1,0.15,0.9402,0.4669,0.9050,0.6160,0.0579,0.0950,1000,1142,105,18591
2,0.20,0.9506,0.5201,0.8887,0.6562,0.0459,0.1113,982,906,123,18827
3,0.25,0.9561,0.5549,0.8742,0.6788,0.0393,0.1258,966,775,139,18958
4,0.30,0.9600,0.5830,0.8643,0.6963,0.0346,0.1357,955,683,150,19050
5,0.35,0.9634,0.6111,0.8534,0.7122,0.0304,0.1466,943,600,162,19133
6,0.40,0.9660,0.6367,0.8344,0.7223,0.0267,0.1656,922,526,183,19207
7,0.45,0.9685,0.6647,0.8199,0.7342,0.0232,0.1801,906,457,199,19276
8,0.50,0.9705,0.6911,0.8018,0.7424,0.0201,0.1982,886,396,219,19337
9,0.55,0.9724,0.7232,0.7756,0.7485,0.0166,0.2244,857,328,248,19405


In [322]:
# ============================================================
# FINAL MODEL REBUILD — STEP 11
# Threshold analysis on realistic legitimate URLs
# ============================================================

challenge_thresholds = [
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80
]

challenge_results = []

# Get probabilities from the weighted rebuilt model
weighted_challenge_prob = weighted_rebuilt_xgb.predict_proba(
    balanced_features
)[:, 1]

for threshold in challenge_thresholds:

    challenge_pred = (
        weighted_challenge_prob >= threshold
    ).astype(int)

    false_positives = challenge_pred.sum()

    fpr = false_positives / len(challenge_pred)

    challenge_results.append({
        "threshold": threshold,
        "false_positives": false_positives,
        "correct_legitimate": len(challenge_pred) - false_positives,
        "false_positive_rate": fpr,
        "average_probability": weighted_challenge_prob.mean(),
        "median_probability": np.median(weighted_challenge_prob)
    })

challenge_threshold_df = pd.DataFrame(
    challenge_results
)

display(
    challenge_threshold_df.round(4)
)

,threshold,false_positives,correct_legitimate,false_positive_rate,average_probability,median_probability
0,0.50,879,5121,0.1465,0.177,0.0351
1,0.55,764,5236,0.1273,0.177,0.0351
2,0.60,622,5378,0.1037,0.177,0.0351
3,0.65,500,5500,0.0833,0.177,0.0351
4,0.70,402,5598,0.0670,0.177,0.0351
5,0.75,297,5703,0.0495,0.177,0.0351
6,0.80,210,5790,0.0350,0.177,0.0351


In [323]:
# ============================================================
# FINAL MODEL REBUILD — STEP 12
# Real-world-style URL validation
# ============================================================

realworld_urls = [
    "https://www.google.com",
    "https://www.google.com/search",
    "https://www.google.com/search?q=python",
    "https://github.com",
    "https://github.com/user/repository",
    "https://github.com/login",
    "https://en.wikipedia.org/wiki/Python",
    "https://stackoverflow.com/questions/12345",
    "https://www.nhs.uk/conditions/",
    "https://www.microsoft.com/en-us/download",
    "https://www.apple.com/support/",
    "https://www.amazon.com/s?k=laptop",
    "https://secure-login-account-verification-12345.com/login",
    "http://paypal-account-security-check-48291.com/verify",
    "http://free-prize-winner-92731.com/claim"
]

realworld_features = pd.DataFrame(
    [extract_url_features(url) for url in realworld_urls]
)[url_features_final]

realworld_prob = weighted_rebuilt_xgb.predict_proba(
    realworld_features
)[:, 1]

realworld_results = pd.DataFrame({
    "url": realworld_urls,
    "phishing_probability": realworld_prob
})

# Show predictions at 0.70
realworld_results["prediction_0.70"] = np.where(
    realworld_results["phishing_probability"] >= 0.70,
    "Phishing",
    "Legitimate"
)

# Also show 0.65 and 0.75 so we can compare
realworld_results["prediction_0.65"] = np.where(
    realworld_results["phishing_probability"] >= 0.65,
    "Phishing",
    "Legitimate"
)

realworld_results["prediction_0.75"] = np.where(
    realworld_results["phishing_probability"] >= 0.75,
    "Phishing",
    "Legitimate"
)

display(
    realworld_results.round(4)
)

,url,phishing_probability,prediction_0.70,prediction_0.65,prediction_0.75
0,https://www.google.com,0.0007,Legitimate,Legitimate,Legitimate
1,https://www.google.com/search,0.0138,Legitimate,Legitimate,Legitimate
2,https://www.google.com/search?q=python,0.0016,Legitimate,Legitimate,Legitimate
3,https://github.com,0.0116,Legitimate,Legitimate,Legitimate
4,https://github.com/user/repository,0.8459,Phishing,Phishing,Phishing
5,https://github.com/login,0.7209,Phishing,Phishing,Legitimate
6,https://en.wikipedia.org/wiki/Python,0.5735,Legitimate,Legitimate,Legitimate
7,https://stackoverflow.com/questions/12345,0.9759,Phishing,Phishing,Phishing
8,https://www.nhs.uk/conditions/,0.0074,Legitimate,Legitimate,Legitimate
9,https://www.microsoft.com/en-us/download,0.0282,Legitimate,Legitimate,Legitimate


In [324]:
# ============================================================
# FINAL MODEL REBUILD — STEP 13
# SHAP diagnostic on real-world-style URLs
# ============================================================

import shap

# Create SHAP explainer for the rebuilt weighted model
rebuilt_shap_explainer = shap.TreeExplainer(
    weighted_rebuilt_xgb
)

# Calculate SHAP values
realworld_shap_values = rebuilt_shap_explainer.shap_values(
    realworld_features
)

# Convert to DataFrame
realworld_shap_df = pd.DataFrame(
    realworld_shap_values,
    columns=url_features_final
)

# Show the most important features for each URL
for i, url in enumerate(realworld_urls):

    print("\n" + "=" * 80)
    print(url)
    print("Phishing probability:",
          round(realworld_prob[i], 4))

    shap_row = realworld_shap_df.iloc[i]

    top_features = (
        shap_row
        .abs()
        .sort_values(ascending=False)
        .head(6)
        .index
    )

    explanation = pd.DataFrame({
        "feature": top_features,
        "feature_value": [
            realworld_features.iloc[i][f]
            for f in top_features
        ],
        "shap_value": [
            shap_row[f]
            for f in top_features
        ]
    })

    display(
        explanation
        .sort_values(
            "shap_value",
            key=lambda x: x.abs(),
            ascending=False
        )
        .round(4)
    )


https://www.google.com
Phishing probability: 0.0007


,feature,feature_value,shap_value
0,num_www,1.0000,-2.4925
1,url_length,22.0000,-1.5983
2,entropy,3.6635,-0.6839
3,ratio_digits_url,0.0000,-0.3395
4,num_dots,2.0000,-0.2341
5,num_hyphens,0.0000,-0.1850



https://www.google.com/search
Phishing probability: 0.0138


,feature,feature_value,shap_value
0,num_www,1.0,-2.8923
1,length_hostname,14.0,0.6582
2,ratio_digits_url,0.0,-0.3081
3,num_hyphens,0.0,-0.1407
4,url_length,29.0,0.1038
5,num_dots,2.0,-0.0702



https://www.google.com/search?q=python
Phishing probability: 0.0016


...

,feature,feature_value,shap_value
0,entropy,4.4234,1.5602
1,ratio_digits_url,0.0877,1.2872
2,num_www,0.0000,0.8531
3,num_hyphens,4.0000,-0.8492
4,ratio_digits_host,0.1163,0.6310
5,url_length,57.0000,0.6201



http://paypal-account-security-check-48291.com/verify
Phishing probability: 0.9436


,feature,feature_value,shap_value
0,entropy,4.4961,1.5980
1,ratio_digits_url,0.0943,1.2300
2,num_www,0.0000,0.7998
3,num_hyphens,4.0000,-0.7613
4,ratio_digits_host,0.1282,0.6044
5,num_subdomains,0.0000,0.5673



http://free-prize-winner-92731.com/claim
Phishing probability: 0.9797


,feature,feature_value,shap_value
0,ratio_digits_url,0.1250,1.2229
1,entropy,4.3964,1.1767
2,num_subdomains,0.0000,0.6840
3,num_www,0.0000,0.6191
4,url_length,40.0000,0.3968
5,num_dots,1.0000,0.3928


In [325]:
# ============================================================
# FINAL MODEL REBUILD — STEP 14
# Add path-specific features
# ============================================================

from urllib.parse import urlparse
from collections import Counter
import math
import re


def calculate_text_entropy(text):
    if not text:
        return 0

    counter = Counter(text)
    length = len(text)

    entropy = 0

    for count in counter.values():
        p = count / length
        entropy -= p * math.log2(p)

    return entropy


def extract_path_features(url):

    parsed = urlparse(url)

    path = parsed.path or ""

    path_clean = path.strip("/")

    path_parts = [
        p for p in path.split("/")
        if p
    ]

    path_tokens = [
        token
        for token in re.split(r"[/_\-]+", path_clean)
        if token
    ]

    path_digits = sum(
        c.isdigit()
        for c in path
    )

    path_letters = sum(
        c.isalpha()
        for c in path
    )

    path_length = len(path)

    return {
        "path_length": path_length,

        "path_depth": len(path_parts),

        "path_token_count": len(path_tokens),

        "path_digit_ratio": (
            path_digits / path_length
            if path_length > 0 else 0
        ),

        "path_entropy": calculate_text_entropy(path),

        "path_has_extension": int(
            bool(
                re.search(
                    r"\.[a-zA-Z0-9]{2,6}$",
                    path_clean
                )
            )
        )
    }


# Extract for training
path_train_features = pd.DataFrame(
    new_train_df["url"]
    .apply(extract_path_features)
    .tolist()
)

# Extract for testing
path_test_features = pd.DataFrame(
    new_test_df["url"]
    .apply(extract_path_features)
    .tolist()
)

# Combine with existing 11 features
X_expanded_train = pd.concat(
    [
        X_new_train.reset_index(drop=True),
        path_train_features.reset_index(drop=True)
    ],
    axis=1
)

X_expanded_test = pd.concat(
    [
        X_new_test.reset_index(drop=True),
        path_test_features.reset_index(drop=True)
    ],
    axis=1
)

print("Expanded training shape:",
      X_expanded_train.shape)

print("Expanded testing shape:",
      X_expanded_test.shape)

print("\nNew features:")
print(path_train_features.columns.tolist())

print("\nAll features:")
print(X_expanded_train.columns.tolist())

print("\nMissing values:")
print(
    X_expanded_train.isnull().sum().sum(),
    X_expanded_test.isnull().sum().sum()
)

Expanded training shape: (84890, 17)
Expanded testing shape: (20838, 17)

New features:
['path_length', 'path_depth', 'path_token_count', 'path_digit_ratio', 'path_entropy', 'path_has_extension']

All features:
['entropy', 'url_length', 'num_www', 'ratio_digits_url', 'num_subdomains', 'length_hostname', 'num_dots', 'num_hyphens', 'ratio_digits_host', 'num_question_mark', 'num_eq', 'path_length', 'path_depth', 'path_token_count', 'path_digit_ratio', 'path_entropy', 'path_has_extension']

Missing values:
0 0


In [326]:
# ============================================================
# FINAL MODEL REBUILD — STEP 15
# Train 17-feature XGBoost
# ============================================================

expanded_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=3
)

expanded_xgb.fit(
    X_expanded_train,
    y_new_train
)

print("17-feature XGBoost trained successfully.")

17-feature XGBoost trained successfully.


In [327]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Predictions
expanded_test_prob = expanded_xgb.predict_proba(
    X_expanded_test
)[:, 1]

expanded_test_pred = (
    expanded_test_prob >= 0.5
).astype(int)

# Metrics
expanded_accuracy = accuracy_score(
    y_new_test,
    expanded_test_pred
)

expanded_precision = precision_score(
    y_new_test,
    expanded_test_pred
)

expanded_recall = recall_score(
    y_new_test,
    expanded_test_pred
)

expanded_f1 = f1_score(
    y_new_test,
    expanded_test_pred
)

expanded_auc = roc_auc_score(
    y_new_test,
    expanded_test_prob
)

print("17-FEATURE MODEL — UNSEEN DOMAIN TEST")
print("=" * 50)

print(f"Accuracy : {expanded_accuracy:.6f}")
print(f"Precision: {expanded_precision:.6f}")
print(f"Recall   : {expanded_recall:.6f}")
print(f"F1 Score : {expanded_f1:.6f}")
print(f"ROC-AUC  : {expanded_auc:.6f}")

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_new_test,
    expanded_test_pred
))

17-FEATURE MODEL — UNSEEN DOMAIN TEST
Accuracy : 0.973510
Precision: 0.723525
Recall   : 0.809955
F1 Score : 0.764304
ROC-AUC  : 0.979207

Confusion Matrix:
[[19391   342]
 [  210   895]]


In [328]:
# ============================================================
# STEP 17 — REAL-WORLD STYLE VALIDATION
# ============================================================

realworld_expanded_features = pd.concat(
    [
        realworld_features.reset_index(drop=True),
        pd.DataFrame(
            realworld_urls
        ).iloc[:, 0].apply(
            extract_path_features
        ).apply(pd.Series).reset_index(drop=True)
    ],
    axis=1
)

realworld_expanded_prob = expanded_xgb.predict_proba(
    realworld_expanded_features
)[:, 1]

print("=" * 90)
print("17-FEATURE MODEL — REAL-WORLD STYLE URL TEST")
print("=" * 90)

for url, prob in zip(
    realworld_urls,
    realworld_expanded_prob
):
    print(
        f"{prob:.4f}  |  {url}"
    )

17-FEATURE MODEL — REAL-WORLD STYLE URL TEST
0.0018  |  https://www.google.com
0.0323  |  https://www.google.com/search
0.0342  |  https://www.google.com/search?q=python
0.0226  |  https://github.com
0.5874  |  https://github.com/user/repository
0.8928  |  https://github.com/login
0.5392  |  https://en.wikipedia.org/wiki/Python
0.9877  |  https://stackoverflow.com/questions/12345
0.0103  |  https://www.nhs.uk/conditions/
0.0147  |  https://www.microsoft.com/en-us/download
0.0039  |  https://www.apple.com/support/
0.0040  |  https://www.amazon.com/s?k=laptop
0.9787  |  https://secure-login-account-verification-12345.com/login
0.9904  |  http://paypal-account-security-check-48291.com/verify
0.9851  |  http://free-prize-winner-92731.com/claim


In [329]:
suspicious_path_words = [
    "login",
    "signin",
    "sign-in",
    "verify",
    "verification",
    "secure",
    "account",
    "authenticate",
    "authentication",
    "password",
    "credential",
    "payment",
    "billing",
    "confirm",
    "confirmation",
    "update",
    "unlock",
    "recover",
    "reset",
    "claim",
    "prize",
    "wallet"
]

normal_path_words = [
    "search",
    "wiki",
    "questions",
    "question",
    "user",
    "users",
    "repository",
    "repositories",
    "blog",
    "news",
    "article",
    "articles",
    "products",
    "product",
    "support",
    "help",
    "about",
    "contact",
    "download",
    "downloads",
    "conditions",
    "profile",
    "settings"
]

In [330]:
# ============================================================
# STEP 18 — PATH TOKEN ANALYSIS
# ============================================================

from collections import Counter
from urllib.parse import urlparse
import re


suspicious_path_words = [
    "login",
    "signin",
    "sign-in",
    "verify",
    "verification",
    "secure",
    "account",
    "authenticate",
    "authentication",
    "password",
    "credential",
    "payment",
    "billing",
    "confirm",
    "confirmation",
    "update",
    "unlock",
    "recover",
    "reset",
    "claim",
    "prize",
    "wallet"
]

normal_path_words = [
    "search",
    "wiki",
    "questions",
    "question",
    "user",
    "users",
    "repository",
    "repositories",
    "blog",
    "news",
    "article",
    "articles",
    "products",
    "product",
    "support",
    "help",
    "about",
    "contact",
    "download",
    "downloads",
    "conditions",
    "profile",
    "settings"
]


def get_path_tokens(url):

    path = urlparse(str(url)).path.lower()

    tokens = [
        token
        for token in re.split(r"[/_\-]+", path)
        if token
    ]

    return tokens


# Analyze the TRAINING DATA ONLY
token_stats = []

for url, label in zip(
    new_train_df["url"],
    y_new_train
):

    tokens = set(get_path_tokens(url))

    for token in tokens:

        if (
            token in suspicious_path_words
            or token in normal_path_words
        ):

            token_stats.append({
                "token": token,
                "label": label
            })


token_df = pd.DataFrame(token_stats)


if len(token_df) == 0:

    print("No matching path tokens found.")

else:

    summary = (
        token_df
        .groupby("token")["label"]
        .agg(
            occurrences="count",
            phishing_count="sum"
        )
        .reset_index()
    )

    summary["phishing_rate"] = (
        summary["phishing_count"]
        / summary["occurrences"]
    )

    summary = summary.sort_values(
        "occurrences",
        ascending=False
    )

    print("PATH TOKEN STATISTICS")
    print("=" * 70)

    display(
        summary.round(4)
    )

PATH TOKEN STATISTICS


,token,occurrences,phishing_count,phishing_rate
0,about,278,0,0.0000
10,login,66,64,0.9697
8,contact,42,3,0.0714
11,news,33,1,0.0303
15,profile,25,6,0.2400
22,user,24,11,0.4583
19,signin,23,23,1.0000
2,article,18,1,0.0556
17,search,18,11,0.6111
7,confirmation,12,12,1.0000


In [331]:
# ============================================================
# STEP 19 — SEMANTIC PATH FEATURES
# ============================================================

suspicious_path_words = [
    "login",
    "signin",
    "sign-in",
    "confirmation",
    "update",
    "authentication",
    "verification",
    "account",
    "verify",
    "payment",
    "claim",
    "wallet"
]

benign_path_words = [
    "about",
    "wiki",
    "article",
    "articles",
    "news",
    "blog",
    "product",
    "repository"
]


def extract_semantic_path_features(url):

    tokens = set(get_path_tokens(url))

    return {

        "path_has_suspicious_word": int(
            any(
                word in tokens
                for word in suspicious_path_words
            )
        ),

        "path_has_benign_word": int(
            any(
                word in tokens
                for word in benign_path_words
            )
        ),

        "path_suspicious_word_count": sum(
            word in tokens
            for word in suspicious_path_words
        ),

        "path_benign_word_count": sum(
            word in tokens
            for word in benign_path_words
        )
    }


# Training semantic features
semantic_train_features = pd.DataFrame(
    new_train_df["url"]
    .apply(extract_semantic_path_features)
    .tolist()
)

# Testing semantic features
semantic_test_features = pd.DataFrame(
    new_test_df["url"]
    .apply(extract_semantic_path_features)
    .tolist()
)


# Add to existing 17 features
X_semantic_train = pd.concat(
    [
        X_expanded_train.reset_index(drop=True),
        semantic_train_features.reset_index(drop=True)
    ],
    axis=1
)

X_semantic_test = pd.concat(
    [
        X_expanded_test.reset_index(drop=True),
        semantic_test_features.reset_index(drop=True)
    ],
    axis=1
)


print("Training shape:",
      X_semantic_train.shape)

print("Testing shape:",
      X_semantic_test.shape)

print("\nSemantic features:")
print(
    semantic_train_features.columns.tolist()
)

print("\nMissing values:")
print(
    X_semantic_train.isnull().sum().sum(),
    X_semantic_test.isnull().sum().sum()
)

Training shape: (84890, 21)
Testing shape: (20838, 21)

Semantic features:
['path_has_suspicious_word', 'path_has_benign_word', 'path_suspicious_word_count', 'path_benign_word_count']

Missing values:
0 0


In [332]:
# ============================================================
# STEP 20 — TRAIN 21-FEATURE XGBOOST
# ============================================================

semantic_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=3
)

semantic_xgb.fit(
    X_semantic_train,
    y_new_train
)

print("21-feature XGBoost trained successfully.")

21-feature XGBoost trained successfully.


In [333]:
# ============================================================
# STEP 21 — EVALUATE 21-FEATURE MODEL
# ============================================================

semantic_test_prob = semantic_xgb.predict_proba(
    X_semantic_test
)[:, 1]

semantic_test_pred = (
    semantic_test_prob >= 0.5
).astype(int)

semantic_accuracy = accuracy_score(
    y_new_test,
    semantic_test_pred
)

semantic_precision = precision_score(
    y_new_test,
    semantic_test_pred
)

semantic_recall = recall_score(
    y_new_test,
    semantic_test_pred
)

semantic_f1 = f1_score(
    y_new_test,
    semantic_test_pred
)

semantic_auc = roc_auc_score(
    y_new_test,
    semantic_test_prob
)

print("21-FEATURE MODEL — UNSEEN DOMAIN TEST")
print("=" * 55)

print(f"Accuracy : {semantic_accuracy:.6f}")
print(f"Precision: {semantic_precision:.6f}")
print(f"Recall   : {semantic_recall:.6f}")
print(f"F1 Score : {semantic_f1:.6f}")
print(f"ROC-AUC  : {semantic_auc:.6f}")

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_new_test,
        semantic_test_pred
    )
)

21-FEATURE MODEL — UNSEEN DOMAIN TEST
Accuracy : 0.974278
Precision: 0.729621
Recall   : 0.818100
F1 Score : 0.771331
ROC-AUC  : 0.979407

Confusion Matrix:
[[19398   335]
 [  201   904]]


In [334]:
# ============================================================
# STEP 22 — 21-FEATURE REAL-WORLD VALIDATION
# ============================================================

realworld_semantic_features = pd.concat(
    [
        realworld_expanded_features.reset_index(drop=True),
        pd.DataFrame(
            realworld_urls
        ).iloc[:, 0].apply(
            extract_semantic_path_features
        ).apply(pd.Series).reset_index(drop=True)
    ],
    axis=1
)

realworld_semantic_prob = semantic_xgb.predict_proba(
    realworld_semantic_features
)[:, 1]

print("=" * 90)
print("21-FEATURE MODEL — REAL-WORLD STYLE URL TEST")
print("=" * 90)

for url, prob in zip(
    realworld_urls,
    realworld_semantic_prob
):
    print(f"{prob:.4f}  |  {url}")

21-FEATURE MODEL — REAL-WORLD STYLE URL TEST
0.0022  |  https://www.google.com
0.0329  |  https://www.google.com/search
0.0303  |  https://www.google.com/search?q=python
0.0231  |  https://github.com
0.0608  |  https://github.com/user/repository
0.9914  |  https://github.com/login
0.0444  |  https://en.wikipedia.org/wiki/Python
0.9918  |  https://stackoverflow.com/questions/12345
0.0128  |  https://www.nhs.uk/conditions/
0.0123  |  https://www.microsoft.com/en-us/download
0.0064  |  https://www.apple.com/support/
0.0028  |  https://www.amazon.com/s?k=laptop
0.9960  |  https://secure-login-account-verification-12345.com/login
0.9968  |  http://paypal-account-security-check-48291.com/verify
0.9978  |  http://free-prize-winner-92731.com/claim


In [335]:
# ============================================================
# STEP 23 — SHAP DIAGNOSTIC FOR 21-FEATURE MODEL
# ============================================================

semantic_shap_explainer = shap.TreeExplainer(
    semantic_xgb
)

semantic_shap_values = (
    semantic_shap_explainer.shap_values(
        realworld_semantic_features
    )
)

semantic_shap_df = pd.DataFrame(
    semantic_shap_values,
    columns=X_semantic_train.columns
)

diagnostic_urls = [
    "https://github.com/user/repository",
    "https://github.com/login",
    "https://en.wikipedia.org/wiki/Python",
    "https://stackoverflow.com/questions/12345",
    "https://www.amazon.com/s?k=laptop",
    "https://secure-login-account-verification-12345.com/login"
]

print("=" * 90)
print("21-FEATURE SHAP DIAGNOSTIC")
print("=" * 90)

for url in diagnostic_urls:

    i = realworld_urls.index(url)

    shap_row = semantic_shap_df.iloc[i]

    top_features = (
        shap_row.abs()
        .sort_values(ascending=False)
        .head(10)
        .index
    )

    explanation = pd.DataFrame({
        "feature": top_features,
        "feature_value": [
            realworld_semantic_features.iloc[i][f]
            for f in top_features
        ],
        "shap_value": [
            shap_row[f]
            for f in top_features
        ]
    })

    print("\n" + "=" * 90)
    print(url)
    print(
        "Phishing probability:",
        round(realworld_semantic_prob[i], 4)
    )

    display(
        explanation
        .sort_values(
            "shap_value",
            key=lambda x: x.abs(),
            ascending=False
        )
        .round(4)
    )

21-FEATURE SHAP DIAGNOSTIC

https://github.com/user/repository
Phishing probability: 0.0608


,feature,feature_value,shap_value
0,path_has_benign_word,1.0000,-2.1052
1,num_subdomains,0.0000,0.8772
2,num_www,0.0000,0.8371
3,num_dots,1.0000,0.8352
4,path_token_count,2.0000,-0.4921
5,path_length,16.0000,0.4920
6,path_benign_word_count,1.0000,-0.4390
7,ratio_digits_url,0.0000,-0.4057
8,entropy,3.9032,-0.3496
9,length_hostname,10.0000,-0.1726



https://github.com/login
Phishing probability: 0.9914


,feature,feature_value,shap_value
0,path_has_suspicious_word,1.000,2.1452
1,num_www,0.000,1.2587
2,path_entropy,2.585,0.9970
3,num_dots,1.000,0.9289
4,num_subdomains,0.000,0.7299
5,url_length,24.000,-0.6894
6,path_suspicious_word_count,1.000,0.5769
7,path_length,6.000,0.4296
8,path_token_count,1.000,0.3723
9,ratio_digits_url,0.000,-0.2373



https://en.wikipedia.org/wiki/Python
Phishing probability: 0.0444


...

,feature,feature_value,shap_value
0,entropy,4.4795,1.4934
1,num_www,0.0000,1.1531
2,ratio_digits_url,0.1220,1.0480
3,path_digit_ratio,0.3125,0.9663
4,num_subdomains,0.0000,0.9250
5,num_dots,1.0000,0.6866
6,path_length,16.0000,0.4108
7,path_token_count,2.0000,-0.2488
8,path_has_extension,0.0000,0.1029
9,length_hostname,17.0000,-0.0974



https://www.amazon.com/s?k=laptop
Phishing probability: 0.0028


,feature,feature_value,shap_value
0,num_www,1.0,-2.5832
1,num_question_mark,1.0,-0.5273
2,num_eq,1.0,-0.4783
3,url_length,33.0,-0.4034
4,num_hyphens,0.0,-0.3025
5,path_entropy,1.0,0.3023
6,ratio_digits_url,0.0,-0.2183
7,num_dots,2.0,-0.1475
8,path_token_count,1.0,0.1432
9,path_length,2.0,-0.1427



https://secure-login-account-verification-12345.com/login
Phishing probability: 0.996


,feature,feature_value,shap_value
0,path_has_suspicious_word,1.0000,1.9952
1,entropy,4.4234,1.2410
2,url_length,57.0000,0.8473
3,num_www,0.0000,0.7128
4,num_subdomains,0.0000,0.6626
5,path_entropy,2.5850,0.4933
6,path_suspicious_word_count,1.0000,0.4107
7,ratio_digits_host,0.1163,0.4015
8,length_hostname,43.0000,-0.3756
9,ratio_digits_url,0.0877,0.3194


In [336]:
# ============================================================
# STEP 24 — REMOVE PATH DIGIT RATIO
# ============================================================

features_without_path_digit = [
    col
    for col in X_semantic_train.columns
    if col != "path_digit_ratio"
]

X_20_train = X_semantic_train[
    features_without_path_digit
].copy()

X_20_test = X_semantic_test[
    features_without_path_digit
].copy()

print("Training shape:", X_20_train.shape)
print("Testing shape:", X_20_test.shape)

print("\nRemoved feature:")
print("path_digit_ratio")

print("\nRemaining features:")
print(features_without_path_digit)

Training shape: (84890, 20)
Testing shape: (20838, 20)

Removed feature:
path_digit_ratio

Remaining features:
['entropy', 'url_length', 'num_www', 'ratio_digits_url', 'num_subdomains', 'length_hostname', 'num_dots', 'num_hyphens', 'ratio_digits_host', 'num_question_mark', 'num_eq', 'path_length', 'path_depth', 'path_token_count', 'path_entropy', 'path_has_extension', 'path_has_suspicious_word', 'path_has_benign_word', 'path_suspicious_word_count', 'path_benign_word_count']


In [337]:
# ============================================================
# STEP 25 — TRAIN 20-FEATURE XGBOOST
# ============================================================

xgb_20 = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=3
)

xgb_20.fit(
    X_20_train,
    y_new_train
)

print("20-feature XGBoost trained successfully.")

20-feature XGBoost trained successfully.


In [338]:
# ============================================================
# STEP 26 — EVALUATE 20-FEATURE MODEL
# ============================================================

xgb_20_prob = xgb_20.predict_proba(
    X_20_test
)[:, 1]

xgb_20_pred = (
    xgb_20_prob >= 0.5
).astype(int)

accuracy_20 = accuracy_score(
    y_new_test,
    xgb_20_pred
)

precision_20 = precision_score(
    y_new_test,
    xgb_20_pred
)

recall_20 = recall_score(
    y_new_test,
    xgb_20_pred
)

f1_20 = f1_score(
    y_new_test,
    xgb_20_pred
)

auc_20 = roc_auc_score(
    y_new_test,
    xgb_20_prob
)

print("20-FEATURE MODEL — UNSEEN DOMAIN TEST")
print("=" * 55)

print(f"Accuracy : {accuracy_20:.6f}")
print(f"Precision: {precision_20:.6f}")
print(f"Recall   : {recall_20:.6f}")
print(f"F1 Score : {f1_20:.6f}")
print(f"ROC-AUC  : {auc_20:.6f}")

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_new_test,
        xgb_20_pred
    )
)

20-FEATURE MODEL — UNSEEN DOMAIN TEST
Accuracy : 0.974374
Precision: 0.732303
Recall   : 0.814480
F1 Score : 0.771208
ROC-AUC  : 0.979260

Confusion Matrix:
[[19404   329]
 [  205   900]]


In [339]:
# ============================================================
# STEP 27 — 20-FEATURE REAL-WORLD VALIDATION
# ============================================================

realworld_20_features = realworld_semantic_features[
    features_without_path_digit
].copy()

realworld_20_prob = xgb_20.predict_proba(
    realworld_20_features
)[:, 1]

print("=" * 90)
print("20-FEATURE MODEL — REAL-WORLD STYLE URL TEST")
print("=" * 90)

for url, prob in zip(
    realworld_urls,
    realworld_20_prob
):
    print(f"{prob:.4f}  |  {url}")

20-FEATURE MODEL — REAL-WORLD STYLE URL TEST
0.0026  |  https://www.google.com
0.0445  |  https://www.google.com/search
0.0132  |  https://www.google.com/search?q=python
0.0212  |  https://github.com
0.0485  |  https://github.com/user/repository
0.9901  |  https://github.com/login
0.0482  |  https://en.wikipedia.org/wiki/Python
0.9856  |  https://stackoverflow.com/questions/12345
0.0063  |  https://www.nhs.uk/conditions/
0.0179  |  https://www.microsoft.com/en-us/download
0.0075  |  https://www.apple.com/support/
0.0022  |  https://www.amazon.com/s?k=laptop
0.9984  |  https://secure-login-account-verification-12345.com/login
0.9981  |  http://paypal-account-security-check-48291.com/verify
0.9978  |  http://free-prize-winner-92731.com/claim


In [342]:
# ============================================================
# STEP 28 — LEGITIMATE PATH/QUERY COVERAGE
# FIXED INDEX VERSION
# ============================================================

train_audit = new_train_df.copy()

# Attach labels using positional alignment
train_audit["model_label"] = np.asarray(
    y_new_train
)

train_audit["has_path"] = (
    train_audit["url"]
    .apply(
        lambda x: bool(
            urlparse(str(x)).path.strip("/")
        )
    )
)

train_audit["has_query"] = (
    train_audit["url"]
    .apply(
        lambda x: bool(
            urlparse(str(x)).query
        )
    )
)

# Separate using the label column we just added
legit_train = train_audit[
    train_audit["model_label"] == 0
].copy()

phishing_train = train_audit[
    train_audit["model_label"] == 1
].copy()


# ============================================================
# LEGITIMATE
# ============================================================

print("LEGITIMATE TRAINING URL COVERAGE")
print("=" * 60)

print(
    "Total legitimate URLs:",
    len(legit_train)
)

print(
    "Legitimate URLs with path:",
    legit_train["has_path"].sum()
)

print(
    "Legitimate URLs with query:",
    legit_train["has_query"].sum()
)

print(
    "Legitimate URLs with path OR query:",
    (
        legit_train["has_path"]
        | legit_train["has_query"]
    ).sum()
)

print("\nPercentages:")

print(
    "Path:",
    round(
        legit_train["has_path"].mean() * 100,
        2
    ),
    "%"
)

print(
    "Query:",
    round(
        legit_train["has_query"].mean() * 100,
        2
    ),
    "%"
)

print(
    "Path OR query:",
    round(
        (
            legit_train["has_path"]
            | legit_train["has_query"]
        ).mean() * 100,
        2
    ),
    "%"
)


# ============================================================
# PHISHING
# ============================================================

print("\nPHISHING TRAINING URL COVERAGE")
print("=" * 60)

print(
    "Total phishing URLs:",
    len(phishing_train)
)

print(
    "Phishing URLs with path:",
    phishing_train["has_path"].sum()
)

print(
    "Phishing URLs with query:",
    phishing_train["has_query"].sum()
)

print(
    "Phishing URLs with path OR query:",
    (
        phishing_train["has_path"]
        | phishing_train["has_query"]
    ).sum()
)

print("\nPercentages:")

print(
    "Path:",
    round(
        phishing_train["has_path"].mean() * 100,
        2
    ),
    "%"
)

print(
    "Query:",
    round(
        phishing_train["has_query"].mean() * 100,
        2
    ),
    "%"
)

print(
    "Path OR query:",
    round(
        (
            phishing_train["has_path"]
            | phishing_train["has_query"]
        ).mean() * 100,
        2
    ),
    "%"
)

LEGITIMATE TRAINING URL COVERAGE
Total legitimate URLs: 79106
Legitimate URLs with path: 18249
Legitimate URLs with query: 1620
Legitimate URLs with path OR query: 18694

Percentages:
Path: 23.07 %
Query: 2.05 %
Path OR query: 23.63 %

PHISHING TRAINING URL COVERAGE
Total phishing URLs: 5784
Phishing URLs with path: 4147
Phishing URLs with query: 472
Phishing URLs with path OR query: 4228

Percentages:
Path: 71.7 %
Query: 8.16 %
Path OR query: 73.1 %


In [343]:
# ============================================================
# STEP 29 — LEGITIMATE PATH/QUERY DOMAIN DIVERSITY
# ============================================================

legit_path_query = legit_train[
    legit_train["has_path"]
    | legit_train["has_query"]
].copy()

# Get registered domains
legit_path_query["registered_domain"] = (
    legit_path_query["url"]
    .apply(get_registered_domain)
)

print("LEGITIMATE PATH/QUERY DOMAIN DIVERSITY")
print("=" * 65)

print(
    "URLs with path/query:",
    len(legit_path_query)
)

print(
    "Unique registered domains:",
    legit_path_query["registered_domain"].nunique()
)

print(
    "Average URLs per domain:",
    round(
        len(legit_path_query)
        / legit_path_query["registered_domain"].nunique(),
        2
    )
)

print("\nTop 30 domains by number of path/query URLs:")
print("=" * 65)

top_domains = (
    legit_path_query["registered_domain"]
    .value_counts()
    .head(30)
)

display(top_domains.to_frame("url_count"))

LEGITIMATE PATH/QUERY DOMAIN DIVERSITY
URLs with path/query: 18694
Unique registered domains: 14085
Average URLs per domain: 1.33

Top 30 domains by number of path/query URLs:


,url_count
registered_domain,
www.nhs.uk,111
usda.gov,87
energy.gov,84
va.gov,60
drdo.gov.in,55
www.gov.uk,54
mpg.de,45
inrae.fr,43
pc.gc.ca,38


In [344]:
# ============================================================
# STEP 30 — REAL-WORLD DOMAIN REPRESENTATION IN TRAINING DATA
# ============================================================

target_domains = [
    "github.com",
    "stackoverflow.com",
    "wikipedia.org",
    "amazon.com",
    "google.com",
    "microsoft.com",
    "apple.com",
    "nhs.uk"
]

# Work only with the legitimate training URLs
legit_urls = legit_train.copy()

legit_urls["registered_domain"] = (
    legit_urls["url"]
    .apply(get_registered_domain)
)

print("REAL-WORLD DOMAIN REPRESENTATION")
print("=" * 75)

for domain in target_domains:

    matches = legit_urls[
        legit_urls["registered_domain"] == domain
    ]

    print("\n" + "-" * 75)
    print(domain)

    print(
        "Legitimate training URLs:",
        len(matches)
    )

    if len(matches) > 0:

        print(
            "Unique URLs:",
            matches["url"].nunique()
        )

        print(
            "Examples:"
        )

        for url in matches["url"].head(5):
            print("  ", url)


# ============================================================
# PATH TOKEN COVERAGE FOR THESE DOMAINS
# ============================================================

print("\n\nPATH TOKEN COVERAGE")
print("=" * 75)

for domain in target_domains:

    matches = legit_urls[
        legit_urls["registered_domain"] == domain
    ]

    if len(matches) == 0:
        continue

    all_tokens = []

    for url in matches["url"]:
        all_tokens.extend(
            get_path_tokens(url)
        )

    token_counts = (
        pd.Series(all_tokens)
        .value_counts()
        .head(15)
    )

    print("\n" + "-" * 75)
    print(domain)

    if len(token_counts) == 0:
        print("No path tokens found.")

    else:
        display(
            token_counts.to_frame(
                "count"
            )
        )

REAL-WORLD DOMAIN REPRESENTATION

---------------------------------------------------------------------------
github.com
Legitimate training URLs: 1
Unique URLs: 1
Examples:
   https://github.com/CICE-Consortium

---------------------------------------------------------------------------
stackoverflow.com
Legitimate training URLs: 0

---------------------------------------------------------------------------
wikipedia.org
Legitimate training URLs: 1
Unique URLs: 1
Examples:
   https://en.wikipedia.org/wiki/WestGrid

---------------------------------------------------------------------------
amazon.com
Legitimate training URLs: 0

---------------------------------------------------------------------------
google.com
Legitimate training URLs: 0

---------------------------------------------------------------------------
microsoft.com
Legitimate training URLs: 0

---------------------------------------------------------------------------
apple.com
Legitimate training URLs: 4
Unique URLs: 

,count
cice,1
consortium,1



---------------------------------------------------------------------------
wikipedia.org


,count
wiki,1
westgrid,1



---------------------------------------------------------------------------
apple.com


,count
de,1
il,1
uk,1


In [345]:
# ============================================================
# STEP 31 — AUDIT FULL EXTERNAL LEGITIMATE PATH/QUERY SET
# ============================================================

# LegitPhish dataset
external_legit = external_df[
    external_df["ClassLabel"] == 1
].copy()

# Extract registered domain
external_legit["registered_domain"] = (
    external_legit["URL"]
    .apply(get_registered_domain)
)

# Detect path/query
external_legit["has_path"] = (
    external_legit["URL"]
    .apply(
        lambda x: bool(
            urlparse(str(x)).path.strip("/")
        )
    )
)

external_legit["has_query"] = (
    external_legit["URL"]
    .apply(
        lambda x: bool(
            urlparse(str(x)).query
        )
    )
)

external_path_query = external_legit[
    external_legit["has_path"]
    | external_legit["has_query"]
].copy()

print("FULL EXTERNAL LEGITIMATE PATH/QUERY SET")
print("=" * 70)

print(
    "Total legitimate URLs:",
    len(external_legit)
)

print(
    "Legitimate path/query URLs:",
    len(external_path_query)
)

print(
    "Unique registered domains:",
    external_path_query["registered_domain"].nunique()
)

print(
    "Duplicate URLs:",
    external_path_query["URL"].duplicated().sum()
)

print("\nTop domains:")
print("=" * 70)

display(
    external_path_query[
        "registered_domain"
    ]
    .value_counts()
    .head(40)
    .to_frame("url_count")
)

FULL EXTERNAL LEGITIMATE PATH/QUERY SET
Total legitimate URLs: 37540
Legitimate path/query URLs: 534
Unique registered domains: 135
Duplicate URLs: 138

Top domains:


,url_count
registered_domain,
wikipedia.org,104
stackoverflow.com,100
youtube.com,52
google.com,9
fbi.gov,6
apple.com,6
yahoo.com,5
forbes.com,5
w3techs.com,4


In [346]:
# ============================================================
# STEP 32 — SAFE EXTERNAL LEGITIMATE DATA AUDIT
# ============================================================

# Copy external legitimate path/query URLs
external_safe_candidate = external_path_query.copy()

# ------------------------------------------------------------
# 1. Remove duplicate URLs
# ------------------------------------------------------------

external_safe_candidate = (
    external_safe_candidate
    .drop_duplicates(subset="URL")
    .copy()
)

# ------------------------------------------------------------
# 2. Domains used by our NEW unseen-domain test set
# ------------------------------------------------------------

new_test_domains = set(
    new_test_df["url"]
    .apply(get_registered_domain)
)

# ------------------------------------------------------------
# 3. Original test domains
# ------------------------------------------------------------

original_test_domains_set = set(
    original_test_domains
)

# ------------------------------------------------------------
# 4. Remove any domain appearing in either test set
# ------------------------------------------------------------

external_safe_candidate = (
    external_safe_candidate[
        ~external_safe_candidate["registered_domain"]
        .isin(new_test_domains)
    ]
    .copy()
)

external_safe_candidate = (
    external_safe_candidate[
        ~external_safe_candidate["registered_domain"]
        .isin(original_test_domains_set)
    ]
    .copy()
)

# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

print("SAFE EXTERNAL LEGITIMATE DATA")
print("=" * 70)

print(
    "Safe URLs:",
    len(external_safe_candidate)
)

print(
    "Safe unique domains:",
    external_safe_candidate[
        "registered_domain"
    ].nunique()
)

print(
    "Domains overlapping NEW test set:",
    len(
        set(
            external_path_query[
                "registered_domain"
            ]
        )
        & new_test_domains
    )
)

print(
    "Domains overlapping ORIGINAL test set:",
    len(
        set(
            external_path_query[
                "registered_domain"
            ]
        )
        & original_test_domains_set
    )
)

# ------------------------------------------------------------
# TARGET DOMAIN CHECK
# ------------------------------------------------------------

target_domains = [
    "github.com",
    "stackoverflow.com",
    "wikipedia.org",
    "amazon.com",
    "google.com",
    "microsoft.com",
    "apple.com",
    "nhs.uk",
    "youtube.com",
    "reddit.com",
    "kaggle.com",
    "cloudflare.com",
    "owasp.org"
]

print("\nTARGET DOMAIN AVAILABILITY")
print("=" * 70)

for domain in target_domains:

    matches = external_safe_candidate[
        external_safe_candidate[
            "registered_domain"
        ] == domain
    ]

    print(
        f"{domain:25s} : {len(matches)} URLs"
    )

# ------------------------------------------------------------
# DOMAIN DISTRIBUTION
# ------------------------------------------------------------

print("\nTOP SAFE DOMAINS")
print("=" * 70)

display(
    external_safe_candidate[
        "registered_domain"
    ]
    .value_counts()
    .head(30)
    .to_frame("url_count")
)

SAFE EXTERNAL LEGITIMATE DATA
Safe URLs: 359
Safe unique domains: 110
Domains overlapping NEW test set: 3
Domains overlapping ORIGINAL test set: 22

TARGET DOMAIN AVAILABILITY
github.com                : 0 URLs
stackoverflow.com         : 100 URLs
wikipedia.org             : 102 URLs
amazon.com                : 0 URLs
google.com                : 0 URLs
microsoft.com             : 0 URLs
apple.com                 : 2 URLs
nhs.uk                    : 0 URLs
youtube.com               : 37 URLs
reddit.com                : 2 URLs
kaggle.com                : 2 URLs
cloudflare.com            : 2 URLs
owasp.org                 : 1 URLs

TOP SAFE DOMAINS


,url_count
registered_domain,
wikipedia.org,102
stackoverflow.com,100
youtube.com,37
explodingtopics.com,2
reddit.com,2
statista.com,2
ssa.gov,2
forbes.com,2
cloudflare.com,2


In [347]:
# ============================================================
# STEP 33 — BALANCED EXTERNAL LEGITIMATE AUGMENTATION
# ============================================================

MAX_URLS_PER_DOMAIN = 5

external_augmented = (
    external_safe_candidate
    .groupby(
        "registered_domain",
        group_keys=False
    )
    .apply(
        lambda x: x.sample(
            n=min(
                len(x),
                MAX_URLS_PER_DOMAIN
            ),
            random_state=42
        )
    )
    .reset_index(drop=True)
)

print("BALANCED EXTERNAL AUGMENTATION")
print("=" * 70)

print(
    "URLs selected:",
    len(external_augmented)
)

print(
    "Unique domains:",
    external_augmented[
        "registered_domain"
    ].nunique()
)

print(
    "Average URLs/domain:",
    round(
        len(external_augmented)
        / external_augmented[
            "registered_domain"
        ].nunique(),
        2
    )
)

print("\nDomain distribution:")
print("=" * 70)

display(
    external_augmented[
        "registered_domain"
    ]
    .value_counts()
    .head(30)
    .to_frame("url_count")
)

BALANCED EXTERNAL AUGMENTATION
URLs selected: 135
Unique domains: 110
Average URLs/domain: 1.23

Domain distribution:


/tmp/ipykernel_1303/1092616100.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,url_count
registered_domain,
wikipedia.org,5
youtube.com,5
stackoverflow.com,5
businessinsider.com,2
ahrefs.com,2
apple.com,2
ssa.gov,2
reddit.com,2
pewresearch.org,2


In [348]:
# ============================================================
# STEP 34 — EXTRACT OUR PRODUCTION FEATURES
# ============================================================

external_aug_features = pd.DataFrame(
    external_augmented["URL"]
    .apply(extract_url_features)
    .tolist()
)

# Add the 6 generic path features
external_aug_path_features = pd.DataFrame(
    external_augmented["URL"]
    .apply(extract_path_features)
    .tolist()
)

# Add the 4 semantic path features
external_aug_semantic_features = pd.DataFrame(
    external_augmented["URL"]
    .apply(extract_semantic_path_features)
    .tolist()
)

# Combine everything
external_aug_X = pd.concat(
    [
        external_aug_features.reset_index(drop=True),
        external_aug_path_features.reset_index(drop=True),
        external_aug_semantic_features.reset_index(drop=True)
    ],
    axis=1
)

# All are legitimate
external_aug_y = np.zeros(
    len(external_aug_X),
    dtype=int
)

print("External feature shape:")
print(external_aug_X.shape)

print("\nExternal labels:")
print(
    np.unique(
        external_aug_y,
        return_counts=True
    )
)

print("\nMissing values:")
print(
    external_aug_X.isnull().sum().sum()
)

print("\nFeature order matches training:")
print(
    list(external_aug_X.columns)
    == list(X_semantic_train.columns)
)

External feature shape:
(135, 21)

External labels:
(array([0]), array([135]))

Missing values:
0

Feature order matches training:
True


In [349]:
# ============================================================
# STEP 35 — TRAIN AUGMENTED 21-FEATURE MODEL
# ============================================================

# Combine original training data + external legitimate URLs
X_augmented_train = pd.concat(
    [
        X_semantic_train.reset_index(drop=True),
        external_aug_X.reset_index(drop=True)
    ],
    axis=0,
    ignore_index=True
)

y_augmented_train = np.concatenate(
    [
        np.asarray(y_new_train),
        external_aug_y
    ]
)

print("Augmented training shape:")
print(X_augmented_train.shape)

print("\nAugmented labels:")
print(
    np.unique(
        y_augmented_train,
        return_counts=True
    )
)


# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

augmented_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=3
)

augmented_xgb.fit(
    X_augmented_train,
    y_augmented_train
)

print("\nAugmented 21-feature XGBoost trained successfully.")

Augmented training shape:
(85025, 21)

Augmented labels:
(array([0, 1]), array([79241,  5784]))

Augmented 21-feature XGBoost trained successfully.


In [350]:
# ============================================================
# STEP 36 — EVALUATE AUGMENTED MODEL
# ============================================================

augmented_test_prob = augmented_xgb.predict_proba(
    X_semantic_test
)[:, 1]

augmented_test_pred = (
    augmented_test_prob >= 0.5
).astype(int)

aug_accuracy = accuracy_score(
    y_new_test,
    augmented_test_pred
)

aug_precision = precision_score(
    y_new_test,
    augmented_test_pred
)

aug_recall = recall_score(
    y_new_test,
    augmented_test_pred
)

aug_f1 = f1_score(
    y_new_test,
    augmented_test_pred
)

aug_auc = roc_auc_score(
    y_new_test,
    augmented_test_prob
)

print("AUGMENTED 21-FEATURE MODEL")
print("=" * 60)

print(f"Accuracy : {aug_accuracy:.6f}")
print(f"Precision: {aug_precision:.6f}")
print(f"Recall   : {aug_recall:.6f}")
print(f"F1 Score : {aug_f1:.6f}")
print(f"ROC-AUC  : {aug_auc:.6f}")

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_new_test,
        augmented_test_pred
    )
)

AUGMENTED 21-FEATURE MODEL
Accuracy : 0.974758
Precision: 0.738272
Recall   : 0.811765
F1 Score : 0.773276
ROC-AUC  : 0.979430

Confusion Matrix:
[[19415   318]
 [  208   897]]


In [351]:
# ============================================================
# STEP 37 — AUGMENTED MODEL REAL-WORLD VALIDATION
# ============================================================

augmented_realworld_prob = augmented_xgb.predict_proba(
    realworld_semantic_features
)[:, 1]

print("=" * 90)
print("AUGMENTED 21-FEATURE MODEL — REAL-WORLD STYLE TEST")
print("=" * 90)

for url, prob in zip(
    realworld_urls,
    augmented_realworld_prob
):
    print(f"{prob:.4f}  |  {url}")

AUGMENTED 21-FEATURE MODEL — REAL-WORLD STYLE TEST
0.0016  |  https://www.google.com
0.0404  |  https://www.google.com/search
0.0228  |  https://www.google.com/search?q=python
0.0239  |  https://github.com
0.0573  |  https://github.com/user/repository
0.9931  |  https://github.com/login
0.0454  |  https://en.wikipedia.org/wiki/Python
0.9835  |  https://stackoverflow.com/questions/12345
0.0096  |  https://www.nhs.uk/conditions/
0.0138  |  https://www.microsoft.com/en-us/download
0.0069  |  https://www.apple.com/support/
0.0021  |  https://www.amazon.com/s?k=laptop
0.9953  |  https://secure-login-account-verification-12345.com/login
0.9970  |  http://paypal-account-security-check-48291.com/verify
0.9981  |  http://free-prize-winner-92731.com/claim


In [352]:
# ============================================================
# STEP 38 — THRESHOLD ANALYSIS
# AUGMENTED 21-FEATURE MODEL
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

threshold_results = []

thresholds = np.arange(0.10, 0.91, 0.05)

for threshold in thresholds:

    pred = (
        augmented_test_prob >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_new_test,
        pred
    ).ravel()

    accuracy = accuracy_score(
        y_new_test,
        pred
    )

    precision = precision_score(
        y_new_test,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_new_test,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_new_test,
        pred,
        zero_division=0
    )

    fpr = fp / (fp + tn)

    fnr = fn / (fn + tp)

    threshold_results.append({
        "threshold": round(threshold, 2),
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "fpr": fpr,
        "fnr": fnr,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    })

threshold_df = pd.DataFrame(
    threshold_results
)

display(
    threshold_df.round(4)
)

print("\nHighest F1:")
display(
    threshold_df.loc[
        threshold_df["f1"].idxmax()
    ].to_frame().T.round(4)
)

,threshold,accuracy,precision,recall,f1,fpr,fnr,TP,FP,FN,TN
0,0.10,0.9301,0.4265,0.9213,0.5830,0.0694,0.0787,1018,1369,87,18364
1,0.15,0.9479,0.5048,0.8986,0.6465,0.0494,0.1014,993,974,112,18759
2,0.20,0.9569,0.5589,0.8842,0.6849,0.0391,0.1158,977,771,128,18962
3,0.25,0.9613,0.5911,0.8778,0.7065,0.0340,0.1222,970,671,135,19062
4,0.30,0.9642,0.6153,0.8670,0.7198,0.0304,0.1330,958,599,147,19134
5,0.35,0.9675,0.6455,0.8570,0.7364,0.0264,0.1430,947,520,158,19213
6,0.40,0.9707,0.6800,0.8462,0.7540,0.0223,0.1538,935,440,170,19293
7,0.45,0.9725,0.7031,0.8335,0.7627,0.0197,0.1665,921,389,184,19344
8,0.50,0.9748,0.7383,0.8118,0.7733,0.0161,0.1882,897,318,208,19415
9,0.55,0.9758,0.7615,0.7919,0.7764,0.0139,0.2081,875,274,230,19459



Highest F1:


,threshold,accuracy,precision,recall,f1,fpr,fnr,TP,FP,FN,TN
12,0.7,0.9791,0.8579,0.7267,0.7869,0.0067,0.2733,803.0,133.0,302.0,19600.0


In [357]:
# ============================================================
# STEP 39 — CORRECT THRESHOLD TEST ON 6,000 LEGITIMATE URLS
# ============================================================

import re
from urllib.parse import urlparse

def extract_21_features_for_url(url):

    # --------------------------------
    # Original 11 URL features
    # --------------------------------
    base_features = extract_url_features(url)

    parsed = urlparse(url)

    path = parsed.path or ""
    query = parsed.query or ""

    path_clean = path.strip("/")

    path_tokens = [
        token
        for token in re.split(r"[/_\-]+", path_clean)
        if token
    ]

    # --------------------------------
    # Generic path features
    # --------------------------------
    path_length = len(path)

    path_depth = len([
        p for p in path.split("/")
        if p
    ])

    path_token_count = len(path_tokens)

    path_digit_ratio = (
        sum(c.isdigit() for c in path) / len(path)
        if len(path) > 0
        else 0
    )

    path_entropy = calculate_entropy(path)

    last_path_part = (
        path_clean.split("/")[-1]
        if path_clean
        else ""
    )

    if "." in last_path_part:
        extension = last_path_part.rsplit(
            ".", 1
        )[-1].lower()
    else:
        extension = ""

    path_has_extension = int(bool(extension))

    # --------------------------------
    # Semantic path features
    # --------------------------------
    suspicious_words = [
        "login",
        "signin",
        "sign-in",
        "confirmation",
        "update",
        "authentication",
        "verification",
        "account",
        "verify",
        "payment",
        "claim",
        "wallet"
    ]

    benign_words = [
        "about",
        "wiki",
        "article",
        "articles",
        "news",
        "blog",
        "product",
        "repository"
    ]

    path_lower = path.lower()

    path_has_suspicious_word = int(
        any(
            word in path_lower
            for word in suspicious_words
        )
    )

    path_has_benign_word = int(
        any(
            word in path_lower
            for word in benign_words
        )
    )

    path_suspicious_word_count = sum(
        path_lower.count(word)
        for word in suspicious_words
    )

    path_benign_word_count = sum(
        path_lower.count(word)
        for word in benign_words
    )

    # --------------------------------
    # Combine all 21 features
    # --------------------------------
    features = {
        **base_features,

        "path_length": path_length,
        "path_depth": path_depth,
        "path_token_count": path_token_count,
        "path_digit_ratio": path_digit_ratio,
        "path_entropy": path_entropy,
        "path_has_extension": path_has_extension,

        "path_has_suspicious_word":
            path_has_suspicious_word,

        "path_has_benign_word":
            path_has_benign_word,

        "path_suspicious_word_count":
            path_suspicious_word_count,

        "path_benign_word_count":
            path_benign_word_count
    }

    return features

In [359]:
print("balanced_features columns:")
print(balanced_features.columns.tolist())

print("\nShape:")
print(balanced_features.shape)

balanced_features columns:
['entropy', 'url_length', 'num_www', 'ratio_digits_url', 'num_subdomains', 'length_hostname', 'num_dots', 'num_hyphens', 'ratio_digits_host', 'num_question_mark', 'num_eq']

Shape:
(6000, 11)


In [360]:
# Find variables related to our 6,000 URL challenge

[
    name for name in globals()
    if (
        "balanced" in name.lower()
        or "challenge" in name.lower()
        or "url" in name.lower()
    )
]

['duplicate_urls',
 'urlparse',
 'urlunparse',
 'normalize_url',
 'conflicting_normalized_urls',
 'url_features',
 'uncertain_urls',
 'extract_url_features',
 'test_url',
 'sample_url',
 'test_urls',
 'url',
 'url_features_final',
 'predict_url',
 'unseen_test_urls',
 'url_exists',
 'X_train_combined_url',
 'X_test_combined_url',
 'url_errors',
 'test_dns_urls',
 'test_subdomain_urls',
 'RDAP_BOOTSTRAP_URL',
 'analyze_url',
 'analyze_url_complete',
 'urls_to_compare',
 'augmented_urls',
 'new_url',
 'clean_augmented_url',
 'validation_urls',
 'small_augmented_urls',
 'comparison_urls',
 'url_features_no_length',
 'test_urls_complete',
 'challenge_suffixes',
 'challenge_urls',
 'challenge_features',
 'current_challenge_prob',
 'structural_challenge_prob',
 'challenge_results',
 'challenge_comparison',
 'get_url_structure',
 'original_urls',
 'balanced_legit',
 'real_challenge_prob',
 'real_challenge_results',
 'challenge_weight1',
 'challenge_diag_features',
 'diag_challenge_prob',
 'di

In [361]:
# ============================================================
# STEP 39 — THRESHOLD TEST ON 6,000 LEGITIMATE CHALLENGE
# ============================================================

print("Number of challenge URLs:", len(balanced_legit_urls))
print("Number of feature rows:", len(balanced_features))

# Extract the SAME 21 features used by the augmented model
challenge_21_features = pd.DataFrame(
    [
        extract_21_features_for_url(url)
        for url in balanced_legit_urls
    ]
)

# Make sure feature order is EXACTLY the same as training
challenge_21_features = challenge_21_features[
    X_semantic_train.columns
]

print("\nChallenge feature shape:",
      challenge_21_features.shape)

print("Missing values:",
      challenge_21_features.isna().sum().sum())

print("Feature count:",
      len(challenge_21_features.columns))

Number of challenge URLs: 6000
Number of feature rows: 6000

Challenge feature shape: (6000, 21)
Missing values: 0
Feature count: 21


In [362]:
# ============================================================
# STEP 39B — TEST DIFFERENT THRESHOLDS
# ============================================================

challenge_prob_augmented = augmented_xgb.predict_proba(
    challenge_21_features
)[:, 1]

threshold_results = []

for threshold in [
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90
]:

    false_positives = (
        challenge_prob_augmented >= threshold
    ).sum()

    fpr = (
        false_positives /
        len(challenge_prob_augmented)
    )

    threshold_results.append({
        "threshold": threshold,
        "false_positives": false_positives,
        "FPR": fpr
    })

challenge_threshold_augmented_df = pd.DataFrame(
    threshold_results
)

display(
    challenge_threshold_augmented_df.round(4)
)

,threshold,false_positives,FPR
0,0.50,662,0.1103
1,0.55,556,0.0927
2,0.60,458,0.0763
3,0.65,344,0.0573
4,0.70,267,0.0445
5,0.75,191,0.0318
6,0.80,116,0.0193
7,0.85,64,0.0107
8,0.90,29,0.0048


In [363]:
# ============================================================
# STEP 40 — REAL-WORLD SANITY CHECK AT CANDIDATE THRESHOLDS
# ============================================================

realworld_21_features = pd.DataFrame(
    [
        extract_21_features_for_url(url)
        for url in realworld_urls
    ]
)

realworld_21_features = realworld_21_features[
    X_semantic_train.columns
]

realworld_prob = augmented_xgb.predict_proba(
    realworld_21_features
)[:, 1]

realworld_threshold_results = pd.DataFrame({
    "URL": realworld_urls,
    "phishing_probability": realworld_prob
})

for threshold in [0.65, 0.70, 0.75, 0.80, 0.85]:

    realworld_threshold_results[
        f"prediction_{threshold:.2f}"
    ] = np.where(
        realworld_prob >= threshold,
        "Phishing",
        "Legitimate"
    )

display(
    realworld_threshold_results.round(4)
)

,URL,phishing_probability,prediction_0.65,prediction_0.70,prediction_0.75,prediction_0.80,prediction_0.85
0,https://www.google.com,0.0016,Legitimate,Legitimate,Legitimate,Legitimate,Legitimate
1,https://www.google.com/search,0.0404,Legitimate,Legitimate,Legitimate,Legitimate,Legitimate
2,https://www.google.com/search?q=python,0.0228,Legitimate,Legitimate,Legitimate,Legitimate,Legitimate
3,https://github.com,0.0239,Legitimate,Legitimate,Legitimate,Legitimate,Legitimate
4,https://github.com/user/repository,0.0573,Legitimate,Legitimate,Legitimate,Legitimate,Legitimate
5,https://github.com/login,0.9931,Phishing,Phishing,Phishing,Phishing,Phishing
6,https://en.wikipedia.org/wiki/Python,0.0454,Legitimate,Legitimate,Legitimate,Legitimate,Legitimate
7,https://stackoverflow.com/questions/12345,0.9835,Phishing,Phishing,Phishing,Phishing,Phishing
8,https://www.nhs.uk/conditions/,0.0096,Legitimate,Legitimate,Legitimate,Legitimate,Legitimate
9,https://www.microsoft.com/en-us/download,0.0138,Legitimate,Legitimate,Legitimate,Legitimate,Legitimate


In [364]:
# Find the external legitimate augmentation variables

[
    name for name in globals()
    if (
        "external" in name.lower()
        or "safe" in name.lower()
        or "aug" in name.lower()
    )
]

['augmented_urls',
 'aug_df',
 'clean_augmented_url',
 'augmented_features',
 'augmented_labels',
 'X_train_augmented',
 'y_train_augmented',
 'augmented_xgb_model',
 'augmented_pred',
 'augmented_prob',
 'augmented_probability',
 'small_augmented_urls',
 'small_augmented_features',
 'small_augmented_labels',
 'X_train_small_aug',
 'y_train_small_aug',
 'small_augmented_xgb_model',
 'small_aug_pred',
 'small_aug_prob',
 'AUGMENTATION_WEIGHT',
 'external_df',
 'external_legit',
 'external_legit_no_overlap',
 'external_exact_overlap',
 'external_features',
 'external_labels',
 'X_external_real',
 'y_external_real',
 'X_train_real_aug',
 'y_train_real_aug',
 'REAL_AUGMENTATION_WEIGHT',
 'real_aug_xgb_model',
 'real_aug_xgb_1',
 'safe_dataset',
 'safe_legit',
 'safe_legit_path_query',
 'external_path_query',
 'external_safe_candidate',
 'external_augmented',
 'external_aug_features',
 'external_aug_path_features',
 'external_aug_semantic_features',
 'external_aug_X',
 'external_aug_y',
 'X

In [366]:
print("external_augmented shape:")
print(external_augmented.shape)

print("\nexternal_augmented columns:")
print(external_augmented.columns.tolist())

external_augmented shape:
(135, 21)

external_augmented columns:
['URL', 'url_length', 'has_ip_address', 'dot_count', 'https_flag', 'url_entropy', 'token_count', 'subdomain_count', 'query_param_count', 'tld_length', 'path_length', 'has_hyphen_in_domain', 'number_of_digits', 'tld_popularity', 'suspicious_file_extension', 'domain_name_length', 'percentage_numeric_chars', 'ClassLabel', 'registered_domain', 'has_path', 'has_query']


In [367]:
# ============================================================
# STEP 41 — REAL STACK OVERFLOW LEGITIMATE EXAMPLES
# ============================================================

stackoverflow_external = external_augmented[
    external_augmented["URL"].str.contains(
        "stackoverflow.com",
        case=False,
        na=False
    )
].copy()

print("Stack Overflow URLs:", len(stackoverflow_external))
print(
    "Unique URLs:",
    stackoverflow_external["URL"].nunique()
)

print("\nExamples:")
display(
    stackoverflow_external[
        ["URL", "registered_domain", "has_path", "has_query"]
    ]
)

Stack Overflow URLs: 5
Unique URLs: 5

Examples:


,URL,registered_domain,has_path,has_query
92,https://stackoverflow.com/questions/tagged/get...,stackoverflow.com,True,False
93,https://stackoverflow.com/questions/tagged/16-bit,stackoverflow.com,True,False
94,https://stackoverflow.com/questions/tagged/azure,stackoverflow.com,True,False
95,https://stackoverflow.com/questions/79030230/w...,stackoverflow.com,True,False
96,https://stackoverflow.com/questions/tagged/cmake,stackoverflow.com,True,False


In [368]:
# ============================================================
# STEP 42 — CHECK CURRENT MODEL ON THE 5 STACK OVERFLOW EXAMPLES
# ============================================================

stackoverflow_urls = stackoverflow_external["URL"].tolist()

stackoverflow_features = pd.DataFrame(
    [
        extract_21_features_for_url(url)
        for url in stackoverflow_urls
    ]
)[
    augmented_xgb.get_booster().feature_names
]

stackoverflow_prob = augmented_xgb.predict_proba(
    stackoverflow_features
)[:, 1]

stackoverflow_check = pd.DataFrame({
    "URL": stackoverflow_urls,
    "Phishing Probability": stackoverflow_prob
})

stackoverflow_check["Prediction"] = np.where(
    stackoverflow_check["Phishing Probability"] >= 0.70,
    "Phishing",
    "Legitimate"
)

display(stackoverflow_check)

,URL,Phishing Probability,Prediction
0,https://stackoverflow.com/questions/tagged/get...,0.751542,Phishing
1,https://stackoverflow.com/questions/tagged/16-bit,0.845910,Phishing
2,https://stackoverflow.com/questions/tagged/azure,0.862310,Phishing
3,https://stackoverflow.com/questions/79030230/w...,0.133468,Legitimate
4,https://stackoverflow.com/questions/tagged/cmake,0.814709,Phishing


In [370]:
# ============================================================
# STEP 43 — SHAP EXPLANATION FOR STACK OVERFLOW EXAMPLES
# ============================================================

import shap
import pandas as pd
import numpy as np

# Use the exact feature order of the current augmented model
stackoverflow_features = pd.DataFrame(
    [
        extract_21_features_for_url(url)
        for url in stackoverflow_urls
    ]
)[augmented_xgb.get_booster().feature_names]

# Create SHAP explainer
stackoverflow_explainer = shap.TreeExplainer(augmented_xgb)

# Calculate SHAP values
stackoverflow_shap = stackoverflow_explainer(
    stackoverflow_features
)

feature_names = stackoverflow_features.columns.tolist()

for i, url in enumerate(stackoverflow_urls):

    shap_values = stackoverflow_shap.values[i]

    explanation = pd.DataFrame({
        "Feature": feature_names,
        "Value": stackoverflow_features.iloc[i].values,
        "SHAP": shap_values
    })

    explanation["Absolute SHAP"] = explanation["SHAP"].abs()

    explanation = explanation.sort_values(
        "Absolute SHAP",
        ascending=False
    ).head(8)

    print("\n" + "=" * 80)
    print(f"URL: {url}")
    print(
        f"Model probability: "
        f"{stackoverflow_prob[i]:.4f}"
    )
    print("=" * 80)

    display(
        explanation[
            ["Feature", "Value", "SHAP"]
        ]
    )


URL: https://stackoverflow.com/questions/tagged/getopts
Model probability: 0.7515


,Feature,Value,SHAP
4,num_subdomains,0.00000,1.069995
2,num_www,0.00000,0.811862
12,path_depth,3.00000,0.693487
1,url_length,50.00000,0.568142
13,path_token_count,3.00000,-0.436858
0,entropy,4.17915,0.361438
3,ratio_digits_url,0.00000,-0.335075
6,num_dots,1.00000,0.265933



URL: https://stackoverflow.com/questions/tagged/16-bit
Model probability: 0.8459


,Feature,Value,SHAP
2,num_www,0.000000,1.049048
0,entropy,4.474418,1.029394
4,num_subdomains,0.000000,0.953528
13,path_token_count,4.000000,-0.612157
3,ratio_digits_url,0.040816,0.368881
12,path_depth,3.000000,0.367905
1,url_length,49.000000,0.364839
6,num_dots,1.000000,0.301834



URL: https://stackoverflow.com/questions/tagged/azure
Model probability: 0.8623


,Feature,Value,SHAP
4,num_subdomains,0.000000,1.098872
0,entropy,4.335501,0.931512
2,num_www,0.000000,0.919449
12,path_depth,3.000000,0.644004
1,url_length,48.000000,0.524051
13,path_token_count,3.000000,-0.447764
6,num_dots,1.000000,0.342389
3,ratio_digits_url,0.000000,-0.324063



URL: https://stackoverflow.com/questions/79030230/why-arent-file-uploads-banner-images-slider-working-in-laravel-with-react-a
Model probability: 0.1335


,Feature,Value,SHAP
7,num_hyphens,12.000000,-1.420387
13,path_token_count,15.000000,-1.333649
0,entropy,4.597826,1.283624
11,path_length,95.000000,-1.052068
4,num_subdomains,0.000000,0.809849
2,num_www,0.000000,0.787487
1,url_length,120.000000,0.537279
3,ratio_digits_url,0.066667,0.433870



URL: https://stackoverflow.com/questions/tagged/cmake
Model probability: 0.8147


,Feature,Value,SHAP
4,num_subdomains,0.000000,1.100661
2,num_www,0.000000,0.889296
0,entropy,4.278107,0.716707
12,path_depth,3.000000,0.660532
1,url_length,48.000000,0.523160
13,path_token_count,3.000000,-0.461864
3,ratio_digits_url,0.000000,-0.334812
6,num_dots,1.000000,0.310766


In [372]:
# ============================================================
# STEP 44 — CHECK BASE FEATURE CLASS DISTRIBUTIONS
# ============================================================

# Use the exact training data used for augmented_xgb
diagnostic_train = pd.DataFrame(
    X_augmented_train,
    columns=augmented_xgb.get_booster().feature_names
)

diagnostic_train["label"] = np.asarray(y_augmented_train)

features_to_check = [
    "num_www",
    "num_subdomains",
    "path_depth",
    "url_length",
    "entropy"
]

for feature in features_to_check:

    print("\n" + "=" * 70)
    print(f"FEATURE: {feature}")
    print("=" * 70)

    summary = diagnostic_train.groupby("label")[feature].agg(
        ["mean", "median", "min", "max"]
    )

    display(summary)

    print("\nClass-wise zero percentage:")

    zero_percentage = (
        diagnostic_train
        .groupby("label")[feature]
        .apply(lambda x: (x == 0).mean() * 100)
    )

    display(zero_percentage.round(2))


FEATURE: num_www


,mean,median,min,max
label,,,,
0,0.765778,1.0,0,2
1,0.060339,0.0,0,2



Class-wise zero percentage:


,num_www
label,
0,23.47
1,94.04



FEATURE: num_subdomains


...

,url_length
label,
0,0.0
1,0.0



FEATURE: entropy


,mean,median,min,max
label,,,,
0,3.789291,3.782122,2.653544,5.909389
1,4.165509,4.082163,2.904418,6.014637



Class-wise zero percentage:


,entropy
label,
0,0.0
1,0.0


In [376]:
# ============================================================
# STEP 45 — CHECK EXISTING TEST VARIABLES
# ============================================================

candidate_variables = [
    "X_semantic_test",
    "X_augmented_test",
    "X_test_semantic",
    "X_test_21",
    "X_test_combined_url",
    "y_semantic_test",
    "y_augmented_test",
    "y_test_semantic",
    "y_test_21"
]

for name in candidate_variables:

    if name in globals():

        obj = globals()[name]

        try:
            print(f"{name}: {obj.shape}")
        except:
            print(f"{name}: exists ({type(obj).__name__})")

X_semantic_test: (20838, 21)
X_test_combined_url: (12491, 11)


In [377]:
# ============================================================
# STEP 46 — FIND THE 21-FEATURE TEST LABELS
# ============================================================

candidate_labels = [
    "y_semantic_test",
    "y_augmented_test",
    "y_test_semantic",
    "y_test_21",
    "y_test_combined_url",
    "y_test"
]

for name in candidate_labels:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: {obj.shape}")

y_test: (12491,)


In [378]:
# ============================================================
# STEP 46 — FIND LABEL ARRAY MATCHING THE 20,838 TEST ROWS
# ============================================================

print("Variables with 20,838 rows:")

for name, obj in list(globals().items()):
    try:
        if hasattr(obj, "shape") and obj.shape[0] == 20838:
            print(name, "->", obj.shape)
    except:
        pass

Variables with 20,838 rows:
pred -> (20838,)
new_test_df -> (20838, 27)
X_new_test -> (20838, 11)
y_new_test -> (20838,)
rebuilt_test_prob -> (20838,)
rebuilt_test_pred -> (20838,)
weighted_test_prob -> (20838,)
weighted_test_pred -> (20838,)
path_test_features -> (20838, 6)
X_expanded_test -> (20838, 17)
expanded_test_prob -> (20838,)
expanded_test_pred -> (20838,)
semantic_test_features -> (20838, 4)
X_semantic_test -> (20838, 21)
semantic_test_prob -> (20838,)
semantic_test_pred -> (20838,)
X_20_test -> (20838, 20)
xgb_20_prob -> (20838,)
xgb_20_pred -> (20838,)
augmented_test_prob -> (20838,)
augmented_test_pred -> (20838,)


In [379]:
# ============================================================
# STEP 47 — ABLATION: REMOVE num_www + num_subdomains
# ============================================================

from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Identify features
# ------------------------------------------------------------

all_features_21 = augmented_xgb.get_booster().feature_names

ablation_features = [
    feature
    for feature in all_features_21
    if feature not in ["num_www", "num_subdomains"]
]

print("Original features:", len(all_features_21))
print("Ablation features:", len(ablation_features))

print("\nRemoved:")
print(["num_www", "num_subdomains"])

# ------------------------------------------------------------
# 2. Prepare training and untouched test data
# ------------------------------------------------------------

X_ablation_train = pd.DataFrame(
    X_augmented_train,
    columns=all_features_21
)[ablation_features]

X_ablation_test = X_semantic_test[
    ablation_features
].copy()

y_ablation_train = np.asarray(y_augmented_train)
y_ablation_test = np.asarray(y_new_test)

print("\nTraining shape:", X_ablation_train.shape)
print("Test shape:", X_ablation_test.shape)

# ------------------------------------------------------------
# 3. Train same XGBoost architecture
# ------------------------------------------------------------

ablation_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=3
)

ablation_xgb.fit(
    X_ablation_train,
    y_ablation_train
)

# ------------------------------------------------------------
# 4. Predict on untouched unseen-domain test set
# ------------------------------------------------------------

ablation_test_prob = ablation_xgb.predict_proba(
    X_ablation_test
)[:, 1]

ablation_test_pred = (
    ablation_test_prob >= 0.50
).astype(int)

# ------------------------------------------------------------
# 5. Metrics
# ------------------------------------------------------------

ablation_accuracy = accuracy_score(
    y_ablation_test,
    ablation_test_pred
)

ablation_precision = precision_score(
    y_ablation_test,
    ablation_test_pred
)

ablation_recall = recall_score(
    y_ablation_test,
    ablation_test_pred
)

ablation_f1 = f1_score(
    y_ablation_test,
    ablation_test_pred
)

ablation_auc = roc_auc_score(
    y_ablation_test,
    ablation_test_prob
)

ablation_cm = confusion_matrix(
    y_ablation_test,
    ablation_test_pred
)

print("\n" + "=" * 60)
print("ABLATION MODEL RESULTS")
print("=" * 60)

print(f"Accuracy : {ablation_accuracy:.6f}")
print(f"Precision: {ablation_precision:.6f}")
print(f"Recall   : {ablation_recall:.6f}")
print(f"F1       : {ablation_f1:.6f}")
print(f"ROC-AUC  : {ablation_auc:.6f}")

print("\nConfusion Matrix:")
print(ablation_cm)

Original features: 21
Ablation features: 19

Removed:
['num_www', 'num_subdomains']

Training shape: (85025, 19)
Test shape: (20838, 19)

ABLATION MODEL RESULTS
Accuracy : 0.973942
Precision: 0.755455
Recall   : 0.752036
F1       : 0.753741
ROC-AUC  : 0.968518

Confusion Matrix:
[[19464   269]
 [  274   831]]


In [380]:
# ============================================================
# STEP 47B — SHOW ABLATION RESULTS
# ============================================================

print("\n" + "=" * 60)
print("ABLATION MODEL RESULTS")
print("=" * 60)

print(f"Accuracy : {ablation_accuracy:.6f}")
print(f"Precision: {ablation_precision:.6f}")
print(f"Recall   : {ablation_recall:.6f}")
print(f"F1       : {ablation_f1:.6f}")
print(f"ROC-AUC  : {ablation_auc:.6f}")

print("\nConfusion Matrix:")
print(ablation_cm)


ABLATION MODEL RESULTS
Accuracy : 0.973942
Precision: 0.755455
Recall   : 0.752036
F1       : 0.753741
ROC-AUC  : 0.968518

Confusion Matrix:
[[19464   269]
 [  274   831]]


In [381]:
# ============================================================
# STEP 48 — ABLATION: REMOVE ONLY num_www
# ============================================================

features_no_www = [
    feature
    for feature in all_features_21
    if feature != "num_www"
]

X_no_www_train = pd.DataFrame(
    X_augmented_train,
    columns=all_features_21
)[features_no_www]

X_no_www_test = X_semantic_test[
    features_no_www
].copy()

no_www_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=3
)

no_www_xgb.fit(
    X_no_www_train,
    y_ablation_train
)

no_www_test_prob = no_www_xgb.predict_proba(
    X_no_www_test
)[:, 1]

no_www_test_pred = (
    no_www_test_prob >= 0.50
).astype(int)

print("\n" + "=" * 60)
print("REMOVE ONLY num_www")
print("=" * 60)

print(
    f"Accuracy : "
    f"{accuracy_score(y_ablation_test, no_www_test_pred):.6f}"
)

print(
    f"Precision: "
    f"{precision_score(y_ablation_test, no_www_test_pred):.6f}"
)

print(
    f"Recall   : "
    f"{recall_score(y_ablation_test, no_www_test_pred):.6f}"
)

print(
    f"F1       : "
    f"{f1_score(y_ablation_test, no_www_test_pred):.6f}"
)

print(
    f"ROC-AUC  : "
    f"{roc_auc_score(y_ablation_test, no_www_test_prob):.6f}"
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_ablation_test,
        no_www_test_pred
    )
)


REMOVE ONLY num_www
Accuracy : 0.975142
Precision: 0.758590
Recall   : 0.779186
F1       : 0.768750
ROC-AUC  : 0.972302

Confusion Matrix:
[[19459   274]
 [  244   861]]


In [382]:
# ============================================================
# STEP 49 — ABLATION: REMOVE ONLY num_subdomains
# ============================================================

features_no_subdomains = [
    feature
    for feature in all_features_21
    if feature != "num_subdomains"
]

X_no_subdomains_train = pd.DataFrame(
    X_augmented_train,
    columns=all_features_21
)[features_no_subdomains]

X_no_subdomains_test = X_semantic_test[
    features_no_subdomains
].copy()

no_subdomains_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=3
)

no_subdomains_xgb.fit(
    X_no_subdomains_train,
    y_ablation_train
)

no_subdomains_test_prob = no_subdomains_xgb.predict_proba(
    X_no_subdomains_test
)[:, 1]

no_subdomains_test_pred = (
    no_subdomains_test_prob >= 0.50
).astype(int)

print("\n" + "=" * 60)
print("REMOVE ONLY num_subdomains")
print("=" * 60)

print(
    f"Accuracy : "
    f"{accuracy_score(y_ablation_test, no_subdomains_test_pred):.6f}"
)

print(
    f"Precision: "
    f"{precision_score(y_ablation_test, no_subdomains_test_pred):.6f}"
)

print(
    f"Recall   : "
    f"{recall_score(y_ablation_test, no_subdomains_test_pred):.6f}"
)

print(
    f"F1       : "
    f"{f1_score(y_ablation_test, no_subdomains_test_pred):.6f}"
)

print(
    f"ROC-AUC  : "
    f"{roc_auc_score(y_ablation_test, no_subdomains_test_prob):.6f}"
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_ablation_test,
        no_subdomains_test_pred
    )
)


REMOVE ONLY num_subdomains
Accuracy : 0.973750
Precision: 0.726094
Recall   : 0.810860
F1       : 0.766139
ROC-AUC  : 0.978875

Confusion Matrix:
[[19395   338]
 [  209   896]]


In [383]:
# ============================================================
# STEP 50 — FIND LEGITIMATE EXTERNAL URLs WITH SIMILAR
# STRUCTURAL FEATURES TO STACK OVERFLOW
# ============================================================

# Stack Overflow feature vectors
stackoverflow_feature_matrix = pd.DataFrame(
    [
        extract_21_features_for_url(url)
        for url in stackoverflow_urls
    ]
)[all_features_21]

# Calculate mean structure of the 5 Stack Overflow URLs
stackoverflow_mean = stackoverflow_feature_matrix.mean()

# Compare every external legitimate example
external_structural_features = pd.DataFrame(
    [
        extract_21_features_for_url(url)
        for url in external_augmented["URL"]
    ]
)

external_structural_features = (
    external_structural_features[all_features_21]
)

# Simple standardized distance using training feature std
train_feature_std = (
    pd.DataFrame(
        X_augmented_train,
        columns=all_features_21
    )
    .std()
    .replace(0, 1)
)

distances = (
    (
        external_structural_features
        - stackoverflow_mean
    )
    / train_feature_std
).pow(2).sum(axis=1).pow(0.5)

similar_external = external_augmented.copy()

similar_external["structural_distance"] = distances.values

similar_external = similar_external.sort_values(
    "structural_distance"
).head(15)

print("Most structurally similar legitimate external URLs:")
display(
    similar_external[
        [
            "URL",
            "registered_domain",
            "has_path",
            "has_query",
            "structural_distance"
        ]
    ]
)

Most structurally similar legitimate external URLs:


,URL,registered_domain,has_path,has_query,structural_distance
47,https://freshysites.com/web-design-development...,freshysites.com,True,False,2.282633
93,https://stackoverflow.com/questions/tagged/16-bit,stackoverflow.com,True,False,2.689504
28,https://datareportal.com/reports/digital-2022-...,datareportal.com,True,False,2.712087
77,https://provaromarketing.com/12-popular-websit...,provaromarketing.com,True,False,2.831114
51,https://blog.hubspot.com/marketing/best-websit...,hubspot.com,True,False,2.956040
69,https://otechworld.com/most-popular-websites-i...,otechworld.com,True,False,3.033417
53,https://inkbotdesign.com/most-popular-websites/,inkbotdesign.com,True,False,3.319126
27,https://dataforseo.com/free-seo-stats/top-1000...,dataforseo.com,True,False,3.381370
55,https://insightssuccess.com/top-9-most-popular...,insightssuccess.com,True,False,3.522510
21,https://www.cloudflare.com/learning/cdn/what-i...,cloudflare.com,True,False,3.586687


In [384]:
# ============================================================
# STEP 51 — VERIFY EXTERNAL AUGMENTATION HAS NO TEST OVERLAP
# ============================================================

# Registered domains from the untouched unseen test set
test_registered_domains = set(
    new_test_df["registered_domain"].dropna().unique()
)

# Registered domains in external_augmented
external_registered_domains = set(
    external_augmented["registered_domain"].dropna().unique()
)

overlap_domains = (
    external_registered_domains
    & test_registered_domains
)

print("External augmentation domains:",
      len(external_registered_domains))

print("Untouched test domains:",
      len(test_registered_domains))

print("Overlapping domains:",
      len(overlap_domains))

if overlap_domains:
    print("\nWARNING — overlapping domains:")
    print(list(overlap_domains)[:20])
else:
    print("\n✓ No registered-domain overlap.")

External augmentation domains: 110
Untouched test domains: 18001
Overlapping domains: 0

✓ No registered-domain overlap.


In [386]:
# ============================================================
# STEP 52 — FULL EXTERNAL LEGITIMATE PATH/QUERY DISTRIBUTION
# ============================================================

from urllib.parse import urlparse

external_distribution = external_legit.copy()

# Derive path/query information from the URL
external_distribution["has_path"] = (
    external_distribution["URL"]
    .astype(str)
    .apply(lambda x: bool(urlparse(x).path))
)

external_distribution["has_query"] = (
    external_distribution["URL"]
    .astype(str)
    .apply(lambda x: bool(urlparse(x).query))
)

external_distribution["path_length"] = (
    external_distribution["URL"]
    .astype(str)
    .apply(lambda x: len(urlparse(x).path))
)

external_distribution["query_length"] = (
    external_distribution["URL"]
    .astype(str)
    .apply(lambda x: len(urlparse(x).query))
)

print("Full external legitimate rows:",
      len(external_distribution))

print(
    "Unique URLs:",
    external_distribution["URL"].nunique()
)

print("\nUnique registered domains:")
print(
    external_distribution["URL"]
    .apply(get_registered_domain)
    .nunique()
)

print("\nPath distribution:")
print(
    external_distribution["has_path"]
    .value_counts()
)

print("\nPath distribution (%):")
print(
    (
        external_distribution["has_path"]
        .value_counts(normalize=True) * 100
    ).round(2)
)

print("\nQuery distribution:")
print(
    external_distribution["has_query"]
    .value_counts()
)

print("\nQuery distribution (%):")
print(
    (
        external_distribution["has_query"]
        .value_counts(normalize=True) * 100
    ).round(2)
)

print("\nAverage path length:")
print(
    round(
        external_distribution["path_length"].mean(),
        4
    )
)

print("\nAverage query length:")
print(
    round(
        external_distribution["query_length"].mean(),
        4
    )
)

Full external legitimate rows: 37540
Unique URLs: 37380

Unique registered domains:
36556

Path distribution:
has_path
False    36941
True       599
Name: count, dtype: int64

Path distribution (%):
has_path
False    98.4
True      1.6
Name: proportion, dtype: float64

Query distribution:
has_query
False    37489
True        51
Name: count, dtype: int64

Query distribution (%):
has_query
False    99.86
True      0.14
Name: proportion, dtype: float64

Average path length:
0.4868

Average query length:
0.0208


In [387]:
# ============================================================
# STEP 52 — INSPECT EXTERNAL LEGITIMATE PATH/QUERY POOL
# ============================================================

external_path_query = external_distribution[
    external_distribution["has_path"]
    | external_distribution["has_query"]
].copy()

print("Path/query URLs:", len(external_path_query))

print(
    "Unique URLs:",
    external_path_query["URL"].nunique()
)

print(
    "Unique registered domains:",
    external_path_query["URL"]
    .apply(get_registered_domain)
    .nunique()
)

print("\nPath/query distribution:")
print(
    external_path_query[
        ["has_path", "has_query"]
    ].value_counts()
)

print("\nTop registered domains:")

domain_counts = (
    external_path_query["URL"]
    .apply(get_registered_domain)
    .value_counts()
    .head(30)
)

display(domain_counts.to_frame("URL_count"))

print("\nSample URLs:")

display(
    external_path_query[
        ["URL"]
    ].head(30)
)

Path/query URLs: 599
Unique URLs: 440
Unique registered domains: 176

Path/query distribution:
has_path  has_query
True      False        548
          True          51
Name: count, dtype: int64

Top registered domains:


,URL_count
URL,
wikipedia.org,104
stackoverflow.com,100
youtube.com,52
google.com,11
fbi.gov,6
apple.com,6
yahoo.com,5
forbes.com,5
time.com,4



Sample URLs:


,URL
63678,https://stackoverflow.com/questions/tagged/php
63679,https://stackoverflow.com/questions/tagged/mic...
63680,https://stackoverflow.com/questions/tagged/sql
63681,https://stackoverflow.com/questions/79030270/h...
63682,https://stackoverflow.com/questions/tagged/java
63683,https://stackoverflow.com/questions/tagged/iwyu
63684,https://stackoverflow.com/questions/tagged/and...
63685,https://stackoverflow.com/questions/tagged/dep...
63686,https://stackoverflow.com/questions/tagged/mac...
63687,https://stackoverflow.com/questions/tagged/build


In [388]:
# ============================================================
# STEP 53 — BUILD EXPANDED REAL LEGITIMATE AUGMENTATION
# ============================================================

MAX_URLS_PER_DOMAIN_V2 = 10

# ------------------------------------------------------------
# 1. Start with legitimate path/query URLs
# ------------------------------------------------------------

expanded_candidate = external_path_query.copy()

# ------------------------------------------------------------
# 2. Remove duplicate URLs
# ------------------------------------------------------------

expanded_candidate = (
    expanded_candidate
    .drop_duplicates(subset=["URL"])
    .copy()
)

# ------------------------------------------------------------
# 3. Calculate registered domain
# ------------------------------------------------------------

expanded_candidate["registered_domain"] = (
    expanded_candidate["URL"]
    .apply(get_registered_domain)
)

# ------------------------------------------------------------
# 4. Remove domains appearing in untouched test set
# ------------------------------------------------------------

expanded_candidate = expanded_candidate[
    ~expanded_candidate["registered_domain"]
    .isin(test_registered_domains)
].copy()

print(
    "After test-domain filtering:",
    len(expanded_candidate)
)

print(
    "Unique domains:",
    expanded_candidate["registered_domain"].nunique()
)

# ------------------------------------------------------------
# 5. Remove URLs already used in external_augmented
# ------------------------------------------------------------

already_used_urls = set(
    external_augmented["URL"]
    .astype(str)
)

expanded_candidate = expanded_candidate[
    ~expanded_candidate["URL"]
    .astype(str)
    .isin(already_used_urls)
].copy()

print(
    "\nAfter removing already-used URLs:",
    len(expanded_candidate)
)

print(
    "Unique domains:",
    expanded_candidate["registered_domain"].nunique()
)

# ------------------------------------------------------------
# 6. Limit representation to 10 URLs/domain
# ------------------------------------------------------------

expanded_candidate = (
    expanded_candidate
    .groupby(
        "registered_domain",
        group_keys=False
    )
    .head(MAX_URLS_PER_DOMAIN_V2)
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 7. Final inspection
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("EXPANDED AUGMENTATION CANDIDATE")
print("=" * 60)

print(
    "New URLs:",
    len(expanded_candidate)
)

print(
    "Unique domains:",
    expanded_candidate["registered_domain"].nunique()
)

print(
    "Average URLs/domain:",
    round(
        len(expanded_candidate)
        / expanded_candidate["registered_domain"].nunique(),
        2
    )
)

print("\nTop domains:")

display(
    expanded_candidate["registered_domain"]
    .value_counts()
    .head(25)
    .to_frame("URL_count")
)

print("\nStack Overflow new examples:")

display(
    expanded_candidate[
        expanded_candidate["registered_domain"]
        == "stackoverflow.com"
    ][["URL"]]
)

print("\nWikipedia new examples:")

display(
    expanded_candidate[
        expanded_candidate["registered_domain"]
        == "wikipedia.org"
    ][["URL"]].head(10)
)

After test-domain filtering: 433
Unique domains: 169

After removing already-used URLs: 298
Unique domains: 62

EXPANDED AUGMENTATION CANDIDATE
New URLs: 104
Unique domains: 62
Average URLs/domain: 1.68

Top domains:


,URL_count
registered_domain,
stackoverflow.com,10
wikipedia.org,10
youtube.com,10
google.com,8
squarespace.com,3
webflow.com,2
yahoo.com,2
quora.com,2
irs.gov,2



Stack Overflow new examples:


,URL
0,https://stackoverflow.com/questions/tagged/php
1,https://stackoverflow.com/questions/tagged/mic...
2,https://stackoverflow.com/questions/tagged/sql
3,https://stackoverflow.com/questions/79030270/h...
4,https://stackoverflow.com/questions/tagged/java
5,https://stackoverflow.com/questions/tagged/iwyu
6,https://stackoverflow.com/questions/tagged/and...
7,https://stackoverflow.com/questions/tagged/dep...
8,https://stackoverflow.com/questions/tagged/mac...
9,https://stackoverflow.com/questions/tagged/build



Wikipedia new examples:


,URL
10,https://en.wikipedia.org/wiki/Hampton_Dellinger
11,https://en.wikipedia.org/wiki/Nerone_(Mascagni)
12,https://en.wikipedia.org/wiki/Portarlington_(P...
13,https://en.wikipedia.org/wiki/Teresa_Riera
14,https://en.wikipedia.org/wiki/Arch_Rock_(Tasma...
15,https://en.wikipedia.org/wiki/List_of_Falling_...
16,https://en.wikipedia.org/wiki/Hatterrall_Ridge
17,https://en.wikipedia.org/wiki/Red_River_shiner
18,https://en.wikipedia.org/wiki/Blue_Kentucky_Gi...
19,https://en.wikipedia.org/wiki/Queen_Victoria_D...


In [389]:
# ============================================================
# STEP 54 — TRAIN EXPANDED REAL-DATA AUGMENTED MODEL
# ============================================================

# ------------------------------------------------------------
# 1. Extract exact 21 production features
# ------------------------------------------------------------

expanded_aug_features = pd.DataFrame(
    [
        extract_21_features_for_url(url)
        for url in expanded_candidate["URL"]
    ]
)[all_features_21]

expanded_aug_labels = np.zeros(
    len(expanded_aug_features),
    dtype=int
)

print("New augmentation features:",
      expanded_aug_features.shape)

print("New augmentation labels:",
      expanded_aug_labels.shape)

# ------------------------------------------------------------
# 2. Combine with the existing training data
# ------------------------------------------------------------

X_expanded_aug_train = np.vstack([
    X_augmented_train,
    expanded_aug_features.values
])

y_expanded_aug_train = np.concatenate([
    np.asarray(y_augmented_train),
    expanded_aug_labels
])

print("\nCombined training shape:",
      X_expanded_aug_train.shape)

print("Combined label counts:")
print(
    pd.Series(y_expanded_aug_train)
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 3. Train same XGBoost architecture
# ------------------------------------------------------------

expanded_aug_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=3
)

expanded_aug_xgb.fit(
    X_expanded_aug_train,
    y_expanded_aug_train
)

print("\n✓ Expanded augmentation model trained.")

New augmentation features: (104, 21)
New augmentation labels: (104,)

Combined training shape: (85129, 21)
Combined label counts:
0    79345
1     5784
Name: count, dtype: int64

✓ Expanded augmentation model trained.


In [390]:
# ============================================================
# STEP 55 — EVALUATE EXPANDED AUGMENTATION MODEL
# ============================================================

expanded_aug_test_features = X_semantic_test[
    all_features_21
].copy()

expanded_aug_test_prob = (
    expanded_aug_xgb
    .predict_proba(expanded_aug_test_features)[:, 1]
)

expanded_aug_test_pred = (
    expanded_aug_test_prob >= 0.50
).astype(int)

expanded_aug_accuracy = accuracy_score(
    y_new_test,
    expanded_aug_test_pred
)

expanded_aug_precision = precision_score(
    y_new_test,
    expanded_aug_test_pred
)

expanded_aug_recall = recall_score(
    y_new_test,
    expanded_aug_test_pred
)

expanded_aug_f1 = f1_score(
    y_new_test,
    expanded_aug_test_pred
)

expanded_aug_auc = roc_auc_score(
    y_new_test,
    expanded_aug_test_prob
)

expanded_aug_cm = confusion_matrix(
    y_new_test,
    expanded_aug_test_pred
)

print("\n" + "=" * 65)
print("EXPANDED REAL-DATA AUGMENTATION RESULTS")
print("=" * 65)

print(f"Accuracy : {expanded_aug_accuracy:.6f}")
print(f"Precision: {expanded_aug_precision:.6f}")
print(f"Recall   : {expanded_aug_recall:.6f}")
print(f"F1       : {expanded_aug_f1:.6f}")
print(f"ROC-AUC  : {expanded_aug_auc:.6f}")

print("\nConfusion Matrix:")
print(expanded_aug_cm)

# ------------------------------------------------------------
# Direct comparison with current model
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC"
    ],
    "Current 21-feature": [
        aug_accuracy,
        aug_precision,
        aug_recall,
        aug_f1,
        aug_auc
    ],
    "Expanded augmentation": [
        expanded_aug_accuracy,
        expanded_aug_precision,
        expanded_aug_recall,
        expanded_aug_f1,
        expanded_aug_auc
    ]
})

comparison["Change"] = (
    comparison["Expanded augmentation"]
    - comparison["Current 21-feature"]
)

display(comparison.round(6))


EXPANDED REAL-DATA AUGMENTATION RESULTS
Accuracy : 0.974614
Precision: 0.737624
Recall   : 0.809050
F1       : 0.771688
ROC-AUC  : 0.979445

Confusion Matrix:
[[19415   318]
 [  211   894]]


,Metric,Current 21-feature,Expanded augmentation,Change
0,Accuracy,0.974758,0.974614,-0.000144
1,Precision,0.738272,0.737624,-0.000648
2,Recall,0.811765,0.809050,-0.002715
3,F1,0.773276,0.771688,-0.001588
4,ROC-AUC,0.979430,0.979445,0.000015


In [391]:
# ============================================================
# STEP 56 — CURRENT MODEL: THRESHOLD 0.70
# ============================================================

threshold = 0.70

threshold_test_pred = (
    augmented_test_prob >= threshold
).astype(int)

threshold_accuracy = accuracy_score(
    y_new_test,
    threshold_test_pred
)

threshold_precision = precision_score(
    y_new_test,
    threshold_test_pred
)

threshold_recall = recall_score(
    y_new_test,
    threshold_test_pred
)

threshold_f1 = f1_score(
    y_new_test,
    threshold_test_pred
)

threshold_cm = confusion_matrix(
    y_new_test,
    threshold_test_pred
)

tn, fp, fn, tp = threshold_cm.ravel()

threshold_fpr = fp / (fp + tn)
threshold_fnr = fn / (fn + tp)

print("\n" + "=" * 65)
print("CURRENT AUGMENTED MODEL — THRESHOLD 0.70")
print("=" * 65)

print(f"Accuracy : {threshold_accuracy:.6f}")
print(f"Precision: {threshold_precision:.6f}")
print(f"Recall   : {threshold_recall:.6f}")
print(f"F1       : {threshold_f1:.6f}")
print(f"FPR      : {threshold_fpr:.6f}")
print(f"FNR      : {threshold_fnr:.6f}")

print("\nConfusion Matrix:")
print(threshold_cm)

print("\nCounts:")
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)


CURRENT AUGMENTED MODEL — THRESHOLD 0.70
Accuracy : 0.979125
Precision: 0.857906
Recall   : 0.726697
F1       : 0.786869
FPR      : 0.006740
FNR      : 0.273303

Confusion Matrix:
[[19600   133]
 [  302   803]]

Counts:
TN: 19600
FP: 133
FN: 302
TP: 803


In [393]:
print("balanced_features shape:", balanced_features.shape)
print("Model expects:", augmented_xgb.n_features_in_)

print("\nModel feature names:")
print(augmented_xgb.get_booster().feature_names)

balanced_features shape: (6000, 11)
Model expects: 21

Model feature names:
['entropy', 'url_length', 'num_www', 'ratio_digits_url', 'num_subdomains', 'length_hostname', 'num_dots', 'num_hyphens', 'ratio_digits_host', 'num_question_mark', 'num_eq', 'path_length', 'path_depth', 'path_token_count', 'path_digit_ratio', 'path_entropy', 'path_has_extension', 'path_has_suspicious_word', 'path_has_benign_word', 'path_suspicious_word_count', 'path_benign_word_count']


In [394]:
# ============================================================
# STEP 58 — FIND EXISTING 21-FEATURE FUNCTIONS
# ============================================================

import inspect

print("Functions containing 'path' or 'semantic':\n")

for name, obj in globals().items():
    if callable(obj):
        if "path" in name.lower() or "semantic" in name.lower():
            print(name)

Functions containing 'path' or 'semantic':

get_path_length
has_path
has_path_or_query
extract_path_query_features
extract_path_features
get_path_tokens
extract_semantic_path_features
semantic_shap_explainer


In [395]:
print("\nExisting feature-related variables:\n")

for name in [
    "path_test_features",
    "semantic_test_features",
    "X_semantic_test",
    "X_expanded_test",
    "X_20_test",
    "external_aug_semantic_features"
]:
    if name in globals():
        obj = globals()[name]
        print(name, getattr(obj, "shape", type(obj)))


Existing feature-related variables:

path_test_features (20838, 6)
semantic_test_features (20838, 4)
X_semantic_test (20838, 21)
X_expanded_test (20838, 17)
X_20_test (20838, 20)
external_aug_semantic_features (135, 4)


In [396]:
# ============================================================
# STEP 59 — CHECK 21-FEATURE EXTRACTION
# ============================================================

sample_url = balanced_legit_urls[0]

print("Sample URL:")
print(sample_url)

print("\nPath features:")
print(extract_path_features(sample_url))

print("\nSemantic path features:")
print(extract_semantic_path_features(sample_url))

Sample URL:
https://muhc.ca/glen-site

Path features:
{'path_length': 10, 'path_depth': 1, 'path_token_count': 2, 'path_digit_ratio': 0.0, 'path_entropy': 3.121928094887362, 'path_has_extension': 0}

Semantic path features:
{'path_has_suspicious_word': 0, 'path_has_benign_word': 0, 'path_suspicious_word_count': 0, 'path_benign_word_count': 0}


In [397]:
# ============================================================
# STEP 60 — BUILD 21-FEATURE LEGITIMATE CHALLENGE MATRIX
# ============================================================

challenge_21_features = []

for url in balanced_legit_urls:

    # Original 11 URL features
    base_features = extract_url_features(url)

    # 6 path features
    path_features = extract_path_features(url)

    # 4 semantic path features
    semantic_features = extract_semantic_path_features(url)

    # Combine in the exact training order
    combined = {
        **base_features,
        **path_features,
        **semantic_features
    }

    challenge_21_features.append(combined)

balanced_features_21 = pd.DataFrame(
    challenge_21_features
)

# Force exact model feature order
balanced_features_21 = balanced_features_21[
    augmented_xgb.get_booster().feature_names
]

print("=" * 60)
print("21-FEATURE CHALLENGE MATRIX")
print("=" * 60)

print("Shape:", balanced_features_21.shape)

print("\nFeature count:", len(balanced_features_21.columns))

print("\nColumns:")
print(list(balanced_features_21.columns))

print("\nMissing values:")
print(balanced_features_21.isnull().sum().sum())

21-FEATURE CHALLENGE MATRIX
Shape: (6000, 21)

Feature count: 21

Columns:
['entropy', 'url_length', 'num_www', 'ratio_digits_url', 'num_subdomains', 'length_hostname', 'num_dots', 'num_hyphens', 'ratio_digits_host', 'num_question_mark', 'num_eq', 'path_length', 'path_depth', 'path_token_count', 'path_digit_ratio', 'path_entropy', 'path_has_extension', 'path_has_suspicious_word', 'path_has_benign_word', 'path_suspicious_word_count', 'path_benign_word_count']

Missing values:
0


In [398]:
# ============================================================
# STEP 61 — FINAL 6,000 LEGITIMATE CHALLENGE
# THRESHOLD = 0.70
# ============================================================

final_threshold = 0.70

challenge_final_prob = augmented_xgb.predict_proba(
    balanced_features_21
)[:, 1]

challenge_final_pred = (
    challenge_final_prob >= final_threshold
).astype(int)

challenge_fp = int(
    challenge_final_pred.sum()
)

challenge_total = len(challenge_final_pred)

challenge_fpr = challenge_fp / challenge_total

print("=" * 65)
print("FINAL LEGITIMATE CHALLENGE — THRESHOLD 0.70")
print("=" * 65)

print(f"Total legitimate URLs : {challenge_total}")
print(f"False positives       : {challenge_fp}")
print(f"False-positive rate   : {challenge_fpr:.4%}")

print("\nProbability statistics:")
print(
    pd.Series(challenge_final_prob).describe()
)

FINAL LEGITIMATE CHALLENGE — THRESHOLD 0.70
Total legitimate URLs : 6000
False positives       : 271
False-positive rate   : 4.5167%

Probability statistics:
count    6000.000000
mean        0.151702
std         0.226480
min         0.000200
25%         0.004452
50%         0.024865
75%         0.224348
max         0.987291
dtype: float64


In [399]:
# ============================================================
# STEP 62 — FINAL REAL-WORLD VALIDATION
# THRESHOLD = 0.70
# ============================================================

realworld_features = []

for url in realworld_urls:

    base_features = extract_url_features(url)
    path_features = extract_path_features(url)
    semantic_features = extract_semantic_path_features(url)

    combined = {
        **base_features,
        **path_features,
        **semantic_features
    }

    realworld_features.append(combined)

realworld_features = pd.DataFrame(realworld_features)

# Exact model feature order
realworld_features = realworld_features[
    augmented_xgb.get_booster().feature_names
]

# Predictions
realworld_prob = augmented_xgb.predict_proba(
    realworld_features
)[:, 1]

realworld_pred = (
    realworld_prob >= 0.70
).astype(int)

print("=" * 80)
print("FINAL REAL-WORLD VALIDATION — THRESHOLD 0.70")
print("=" * 80)

for url, prob, pred in zip(
    realworld_urls,
    realworld_prob,
    realworld_pred
):
    label = "PHISHING" if pred == 1 else "LEGITIMATE"

    print(
        f"{prob:.4f} | {label:10} | {url}"
    )

FINAL REAL-WORLD VALIDATION — THRESHOLD 0.70
0.0016 | LEGITIMATE | https://www.google.com
0.0404 | LEGITIMATE | https://www.google.com/search
0.0228 | LEGITIMATE | https://www.google.com/search?q=python
0.0239 | LEGITIMATE | https://github.com
0.0573 | LEGITIMATE | https://github.com/user/repository
0.9931 | PHISHING   | https://github.com/login
0.0454 | LEGITIMATE | https://en.wikipedia.org/wiki/Python
0.9835 | PHISHING   | https://stackoverflow.com/questions/12345
0.0096 | LEGITIMATE | https://www.nhs.uk/conditions/
0.0138 | LEGITIMATE | https://www.microsoft.com/en-us/download
0.0069 | LEGITIMATE | https://www.apple.com/support/
0.0021 | LEGITIMATE | https://www.amazon.com/s?k=laptop
0.9953 | PHISHING   | https://secure-login-account-verification-12345.com/login
0.9970 | PHISHING   | http://paypal-account-security-check-48291.com/verify
0.9981 | PHISHING   | http://free-prize-winner-92731.com/claim


In [400]:
# ============================================================
# STEP 63 — SAVE FINAL MODEL
# ============================================================

import os
import joblib

FINAL_MODEL_DIR = "/content/final_model"

os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

# 1. Save XGBoost model
FINAL_MODEL_PATH = os.path.join(
    FINAL_MODEL_DIR,
    "phishing_xgb_model.json"
)

augmented_xgb.save_model(FINAL_MODEL_PATH)

# 2. Save exact feature names
FINAL_FEATURE_NAMES = augmented_xgb.get_booster().feature_names

FEATURE_NAMES_PATH = os.path.join(
    FINAL_MODEL_DIR,
    "feature_names.pkl"
)

joblib.dump(
    FINAL_FEATURE_NAMES,
    FEATURE_NAMES_PATH
)

# 3. Save threshold
THRESHOLD_PATH = os.path.join(
    FINAL_MODEL_DIR,
    "threshold.pkl"
)

joblib.dump(
    0.70,
    THRESHOLD_PATH
)

print("=" * 65)
print("FINAL MODEL SAVED")
print("=" * 65)

print("Model:", FINAL_MODEL_PATH)
print("Features:", FEATURE_NAMES_PATH)
print("Threshold:", THRESHOLD_PATH)

print("\nFeature count:", len(FINAL_FEATURE_NAMES))
print("Threshold:", 0.70)

FINAL MODEL SAVED
Model: /content/final_model/phishing_xgb_model.json
Features: /content/final_model/feature_names.pkl
Threshold: /content/final_model/threshold.pkl

Feature count: 21
Threshold: 0.7


In [401]:
# ============================================================
# STEP 64 — VERIFY SAVED FINAL MODEL
# ============================================================

import os
import joblib
import xgboost as xgb

MODEL_DIR = "/content/final_model"

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "phishing_xgb_model.json"
)

FEATURE_PATH = os.path.join(
    MODEL_DIR,
    "feature_names.pkl"
)

THRESHOLD_PATH = os.path.join(
    MODEL_DIR,
    "threshold.pkl"
)

# Load model
test_model = xgb.XGBClassifier()
test_model.load_model(MODEL_PATH)

# Load metadata
test_features = joblib.load(FEATURE_PATH)
test_threshold = joblib.load(THRESHOLD_PATH)

print("=" * 65)
print("SAVED MODEL VERIFICATION")
print("=" * 65)

print("Model loaded successfully")
print("Feature count:", len(test_features))
print("Threshold:", test_threshold)

print("\nFeatures:")
print(test_features)

print("\nModel expects:",
      test_model.get_booster().feature_names)

SAVED MODEL VERIFICATION
Model loaded successfully
Feature count: 21
Threshold: 0.7

Features:
['entropy', 'url_length', 'num_www', 'ratio_digits_url', 'num_subdomains', 'length_hostname', 'num_dots', 'num_hyphens', 'ratio_digits_host', 'num_question_mark', 'num_eq', 'path_length', 'path_depth', 'path_token_count', 'path_digit_ratio', 'path_entropy', 'path_has_extension', 'path_has_suspicious_word', 'path_has_benign_word', 'path_suspicious_word_count', 'path_benign_word_count']

Model expects: ['entropy', 'url_length', 'num_www', 'ratio_digits_url', 'num_subdomains', 'length_hostname', 'num_dots', 'num_hyphens', 'ratio_digits_host', 'num_question_mark', 'num_eq', 'path_length', 'path_depth', 'path_token_count', 'path_digit_ratio', 'path_entropy', 'path_has_extension', 'path_has_suspicious_word', 'path_has_benign_word', 'path_suspicious_word_count', 'path_benign_word_count']


In [402]:
from google.colab import files

files.download("/content/final_model/phishing_xgb_model.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [403]:
files.download("/content/final_model/feature_names.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [405]:
files.download("/content/final_model/threshold.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>